### BENMAP


Esta celda carga la superficie final del HBM para PM$_{2.5}$, verifica que tenga las columnas esperadas y construye los dos primeros insumos de exposición para BenMAP-CE. El escenario **baseline** se define con la mediana posterior del modelo (\texttt{p50_hbm}), mientras que el escenario **control** se genera de forma configurable, ya sea mediante una reducción porcentual uniforme o imponiendo una concentración objetivo. Además, la celda conserva los percentiles \texttt{p05_hbm} y \texttt{p95_hbm} para análisis de sensibilidad, resume estadísticamente los resultados y exporta los archivos \texttt{benmap_PM25_baseline.csv} y \texttt{benmap_PM25_control.csv}, que servirán como base para la etapa de BenMAP-CE.


In [2]:
# ============================================================
# CELDA 1 CORREGIDA: Preparar superficies baseline y control
# de PM2.5 para BenMAP-CE a partir de la salida final del HBM
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Parámetros del escenario control
#    mode = "relative_reduction"  -> reducción porcentual uniforme
#    mode = "target_cap"          -> límite superior fijo
# ------------------------------------------------------------
mode = "relative_reduction"

# Si usas reducción porcentual:
relative_reduction = 0.10   # 10%

# Si usas concentración objetivo:
target_cap = 10.0           # ejemplo: 10 µg/m3

# ------------------------------------------------------------
# 2) Localizar automáticamente el archivo HBM de PM2.5
# ------------------------------------------------------------
current_dir = Path.cwd()

# Posibles raíces de búsqueda: carpeta actual y niveles superiores
search_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent
]

# Nombre exacto esperado
target_name = "HBM_PM25_M1bclean_surface_final.csv"

matches = []

for root in search_roots:
    if root.exists():
        matches.extend(root.rglob(target_name))

# Si no encuentra exacto, busca algo parecido
if not matches:
    alternative_patterns = [
        "*PM25*surface*final*.csv",
        "*PM25*final*.csv",
        "*PM25*.csv"
    ]
    for root in search_roots:
        if root.exists():
            for pattern in alternative_patterns:
                matches.extend(root.rglob(pattern))

# Eliminar duplicados
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        "No se encontró el archivo final de PM2.5 del HBM dentro del proyecto. "
        "Revisa el nombre exacto del archivo en HBM_PM25_V2_OUT."
    )

print("Archivos candidatos encontrados:")
for i, m in enumerate(matches, 1):
    print(f"{i}. {m}")

# Tomamos el primero; si quieres luego podemos fijar uno específico
input_file = matches[0]
print("\nArchivo seleccionado:", input_file)

# ------------------------------------------------------------
# 3) Carpeta de salida para BenMAP
# ------------------------------------------------------------
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

baseline_file = output_dir / "benmap_PM25_baseline.csv"
control_file  = output_dir / "benmap_PM25_control.csv"

# ------------------------------------------------------------
# 4) Cargar archivo HBM PM2.5
# ------------------------------------------------------------
df = pd.read_csv(input_file)

print("\nArchivo cargado correctamente.")
print("Dimensión:", df.shape)
print("Columnas:", df.columns.tolist())

# ------------------------------------------------------------
# 5) Verificar estructura mínima esperada
# ------------------------------------------------------------
required_cols = [
    "cell_id", "fecha", "year", "month",
    "p05_hbm", "p50_hbm", "p95_hbm"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias en el archivo HBM: {missing}")

# ------------------------------------------------------------
# 6) Estandarizar tipos
# ------------------------------------------------------------
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["cell_id"] = pd.to_numeric(df["cell_id"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["month"] = pd.to_numeric(df["month"], errors="coerce")

if df["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en la columna 'fecha'.")
if df["cell_id"].isna().any():
    raise ValueError("Hay valores inválidos en 'cell_id'.")
if df["year"].isna().any() or df["month"].isna().any():
    raise ValueError("Hay valores inválidos en 'year' o 'month'.")

df["cell_id"] = df["cell_id"].astype(int)
df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)

# ------------------------------------------------------------
# 7) Crear baseline BenMAP
#    Usamos p50_hbm como superficie central del HBM
# ------------------------------------------------------------
baseline = df[["cell_id", "fecha", "year", "month", "p05_hbm", "p50_hbm", "p95_hbm"]].copy()

baseline = baseline.rename(columns={
    "p50_hbm": "pm25_baseline",
    "p05_hbm": "pm25_p05",
    "p95_hbm": "pm25_p95"
})

baseline["pm25_baseline"] = baseline["pm25_baseline"].clip(lower=0)
baseline["pm25_p05"] = baseline["pm25_p05"].clip(lower=0)
baseline["pm25_p95"] = baseline["pm25_p95"].clip(lower=0)

# ------------------------------------------------------------
# 8) Crear control BenMAP
# ------------------------------------------------------------
control = baseline.copy()

if mode == "relative_reduction":
    control["pm25_control"] = control["pm25_baseline"] * (1 - relative_reduction)
    scenario_desc = f"Reducción uniforme del {relative_reduction:.0%}"

elif mode == "target_cap":
    control["pm25_control"] = np.minimum(control["pm25_baseline"], target_cap)
    scenario_desc = f"Concentración máxima objetivo de {target_cap} µg/m3"

else:
    raise ValueError("El parámetro 'mode' debe ser 'relative_reduction' o 'target_cap'.")

control["pm25_control"] = control["pm25_control"].clip(lower=0)

# ------------------------------------------------------------
# 9) Seleccionar columnas finales para exportación
# ------------------------------------------------------------
baseline_export = baseline[[
    "cell_id", "fecha", "year", "month",
    "pm25_baseline", "pm25_p05", "pm25_p95"
]].copy()

control_export = control[[
    "cell_id", "fecha", "year", "month",
    "pm25_control"
]].copy()

baseline_export["fecha"] = baseline_export["fecha"].dt.strftime("%Y-%m-%d")
control_export["fecha"] = control_export["fecha"].dt.strftime("%Y-%m-%d")

# ------------------------------------------------------------
# 10) Exportar archivos
# ------------------------------------------------------------
baseline_export.to_csv(baseline_file, index=False)
control_export.to_csv(control_file, index=False)

# ------------------------------------------------------------
# 11) Resumen de control de calidad
# ------------------------------------------------------------
summary = pd.DataFrame({
    "archivo": ["baseline", "control"],
    "n_filas": [len(baseline_export), len(control_export)],
    "n_celdas": [baseline_export["cell_id"].nunique(), control_export["cell_id"].nunique()],
    "periodo_inicio": [baseline_export["fecha"].min(), control_export["fecha"].min()],
    "periodo_fin": [baseline_export["fecha"].max(), control_export["fecha"].max()],
    "media": [baseline_export["pm25_baseline"].mean(), control_export["pm25_control"].mean()],
    "min": [baseline_export["pm25_baseline"].min(), control_export["pm25_control"].min()],
    "max": [baseline_export["pm25_baseline"].max(), control_export["pm25_control"].max()]
})

print("\nEscenario control aplicado:", scenario_desc)
print("\nResumen de superficies exportadas:")
print(summary)

print("\nArchivos generados:")
print(" -", baseline_file)
print(" -", control_file)

print("\nVista previa baseline:")
print(baseline_export.head())

print("\nVista previa control:")
print(control_export.head())

Archivos candidatos encontrados:
1. d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_surface_final.csv

Archivo seleccionado: d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_PM25_V2_OUT\HBM_PM25_M1bclean_surface_final.csv

Archivo cargado correctamente.
Dimensión: (15240, 10)
Columnas: ['cell_id', 'fecha', 'year', 'month', 'cell_idx', 'time_idx', 'p05_hbm', 'p50_hbm', 'p95_hbm', 'width_90']

Escenario control aplicado: Reducción uniforme del 10%

Resumen de superficies exportadas:
    archivo  n_filas  n_celdas periodo_inicio periodo_fin      media  \
0  baseline    15240       254     2020-01-01  2024-12-01  14.780428   
1   control    15240       254     2020-01-01  2024-12-01  13.302385   

        min        max  
0  4.751628  44.161183  
1  4.276465  39.745065  

Archivos generados:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_PM25_baseline.csv
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_PM25_control.csv

Vista previa baseline:
   cell_id       fecha  y

## siguiente celda  

Esta celda identifica automáticamente el archivo espacial correspondiente a la malla de 3 km utilizada en el proyecto, valida que contenga un identificador de celda consistente con las superficies de exposición previamente generadas y construye la definición espacial base para BenMAP-CE. Para ello, se cargan los identificadores presentes en el archivo \texttt{benmap_PM25_baseline.csv}, se busca una capa espacial de la grilla en formatos comunes como \texttt{.shp}, \texttt{.gpkg} o \texttt{.geojson}, y se filtran únicamente las celdas que realmente participan en el análisis. Además, se calcula un resumen geométrico con área y centroides, y se exportan dos productos: un archivo espacial limpio \texttt{benmap_grid_definition.shp} y una tabla auxiliar \texttt{benmap_grid_definition_summary.csv}. De esta manera, se garantiza que la definición espacial empleada en BenMAP sea coherente con la malla usada en el modelo jerárquico bayesiano y en la construcción de escenarios de exposición.

In [3]:
# ============================================================
# CELDA 2: Localizar y exportar la grilla espacial para BenMAP
# ============================================================

import pandas as pd
from pathlib import Path

try:
    import geopandas as gpd
except ImportError:
    raise ImportError(
        "No se encontró geopandas en tu entorno. "
        "Instálalo antes de continuar, por ejemplo con: pip install geopandas"
    )

# ------------------------------------------------------------
# 1) Definir rutas base
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent

output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

baseline_file = output_dir / "benmap_PM25_baseline.csv"

if not baseline_file.exists():
    raise FileNotFoundError(
        f"No existe el archivo baseline esperado: {baseline_file}\n"
        "Primero debes ejecutar la celda anterior."
    )

# ------------------------------------------------------------
# 2) Cargar celdas que sí entran al análisis
# ------------------------------------------------------------
df_base = pd.read_csv(baseline_file)
required_base_cols = ["cell_id"]
missing_base = [c for c in required_base_cols if c not in df_base.columns]
if missing_base:
    raise ValueError(f"Faltan columnas en baseline: {missing_base}")

cell_ids_baseline = set(pd.to_numeric(df_base["cell_id"], errors="coerce").dropna().astype(int).unique())

print("Celdas encontradas en baseline:", len(cell_ids_baseline))

# ------------------------------------------------------------
# 3) Buscar automáticamente la grilla espacial
# ------------------------------------------------------------
search_roots = [
    project_root,
    project_root / "CODIGO",
    project_root / "GRILLA-GEOVISOR MAPAS BEN-MAP",
    project_root / "Qgis"
]

patterns = [
    "*grid*3km*.shp",
    "*grilla*3km*.shp",
    "*malla*3km*.shp",
    "*3km*.shp",
    "*grid*.gpkg",
    "*grilla*.gpkg",
    "*malla*.gpkg",
    "*3km*.gpkg",
    "*grid*.geojson",
    "*grilla*.geojson",
    "*malla*.geojson",
    "*3km*.geojson"
]

candidates = []

for root in search_roots:
    if root.exists():
        for pattern in patterns:
            candidates.extend(root.rglob(pattern))

# Quitar duplicados
candidates = list(dict.fromkeys(candidates))

if not candidates:
    raise FileNotFoundError(
        "No se encontró automáticamente un archivo espacial de la grilla.\n"
        "Revisa si la malla 3 km está en formato .shp, .gpkg o .geojson dentro del proyecto."
    )

print("\nArchivos espaciales candidatos encontrados:")
for i, c in enumerate(candidates, 1):
    print(f"{i}. {c}")

# ------------------------------------------------------------
# 4) Evaluar candidatos y escoger el mejor
# ------------------------------------------------------------
possible_id_cols = [
    "cell_id", "cellid", "cellid_", "cell_idx", "id", "ID",
    "grid_id", "gridid", "id_grid", "cell", "CellID"
]

best_match = None
best_score = -1
best_gdf = None
best_id_col = None

for candidate in candidates:
    try:
        gdf_tmp = gpd.read_file(candidate)
    except Exception:
        continue

    if gdf_tmp.empty:
        continue

    # buscar columna identificadora
    found_id_col = None
    for col in possible_id_cols:
        if col in gdf_tmp.columns:
            found_id_col = col
            break

    if found_id_col is None:
        continue

    ids_tmp = pd.to_numeric(gdf_tmp[found_id_col], errors="coerce").dropna().astype(int)
    overlap = len(set(ids_tmp.unique()).intersection(cell_ids_baseline))

    if overlap > best_score:
        best_score = overlap
        best_match = candidate
        best_gdf = gdf_tmp.copy()
        best_id_col = found_id_col

if best_match is None:
    raise ValueError(
        "Se encontraron archivos espaciales, pero ninguno tiene una columna identificadora "
        "compatible con las celdas del baseline."
    )

print("\nArchivo espacial seleccionado:", best_match)
print("Columna identificadora usada:", best_id_col)
print("Traslape con baseline:", best_score)

# ------------------------------------------------------------
# 5) Limpiar y filtrar grilla
# ------------------------------------------------------------
gdf = best_gdf.copy()

gdf["cell_id"] = pd.to_numeric(gdf[best_id_col], errors="coerce")
gdf = gdf.dropna(subset=["cell_id"]).copy()
gdf["cell_id"] = gdf["cell_id"].astype(int)

# Filtrar solo celdas presentes en baseline
gdf = gdf[gdf["cell_id"].isin(cell_ids_baseline)].copy()

if gdf.empty:
    raise ValueError("Después del filtrado, la grilla quedó vacía.")

# Eliminar duplicados por cell_id si existieran
gdf = gdf.drop_duplicates(subset=["cell_id"]).copy()

print("\nNúmero de celdas en la grilla filtrada:", len(gdf))

# ------------------------------------------------------------
# 6) Validar CRS y calcular resumen geométrico
# ------------------------------------------------------------
if gdf.crs is None:
    print("\nAdvertencia: la grilla no tiene CRS definido. "
          "Se conservará así, pero conviene revisar la capa original.")

# Si está en geográficas, se proyecta a EPSG:3116 para áreas y centroides
gdf_metric = gdf.copy()
try:
    if gdf_metric.crs is not None and gdf_metric.crs.is_geographic:
        gdf_metric = gdf_metric.to_crs(epsg=3116)
except Exception:
    pass

gdf_metric["area_m2"] = gdf_metric.geometry.area
gdf_metric["centroid_x"] = gdf_metric.geometry.centroid.x
gdf_metric["centroid_y"] = gdf_metric.geometry.centroid.y

summary = pd.DataFrame({
    "cell_id": gdf_metric["cell_id"],
    "area_m2": gdf_metric["area_m2"],
    "centroid_x": gdf_metric["centroid_x"],
    "centroid_y": gdf_metric["centroid_y"]
}).sort_values("cell_id").reset_index(drop=True)

# ------------------------------------------------------------
# 7) Exportar productos para BenMAP
# ------------------------------------------------------------
grid_shp = output_dir / "benmap_grid_definition.shp"
grid_summary_csv = output_dir / "benmap_grid_definition_summary.csv"

# Conservar solo columnas necesarias + geometría
gdf_export = gdf[["cell_id", "geometry"]].copy()
gdf_export.to_file(grid_shp)

summary.to_csv(grid_summary_csv, index=False)

# ------------------------------------------------------------
# 8) Resumen final
# ------------------------------------------------------------
print("\nProductos generados:")
print(" -", grid_shp)
print(" -", grid_summary_csv)

print("\nResumen geométrico:")
print(summary.head())

print("\nControl final:")
print("Número de celdas únicas exportadas:", gdf_export["cell_id"].nunique())
print("Número de celdas esperadas según baseline:", len(cell_ids_baseline))

missing_in_grid = sorted(cell_ids_baseline - set(gdf_export["cell_id"].unique()))
print("Celdas faltantes en grilla:", len(missing_in_grid))

if len(missing_in_grid) > 0:
    print("Primeras celdas faltantes:", missing_in_grid[:10])

Celdas encontradas en baseline: 254

Archivos espaciales candidatos encontrados:
1. d:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time_idw_2020_2024.gpkg

Archivo espacial seleccionado: d:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time_idw_2020_2024.gpkg
Columna identificadora usada: cell_id
Traslape con baseline: 254

Número de celdas en la grilla filtrada: 254

Productos generados:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition.shp
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_summary.csv

Resumen geométrico:
   cell_id       area_m2     centroid_x     centroid_y
0        0  6.721942e+06  960189.386999  906093.408749
1        1  3.650667e+06  961004.729405  908663.874418
2        2  6.935858e+04  961371.538155  910412.361905
3       41  5.431189e+06  963118.749994  906387.194845
4       42  9.000000e+06  963146.695573  908837.529063

Control final:
Número de celdas únicas exportadas: 254
Número de celdas esperadas según baseline: 254
Celdas falta

d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'grid_3km_time_idw_2020_2024.gpkg': 'grid_3km_static' (default), 'localidades_bogota', 'grid_3km_time_idw'. Specify layer parameter to avoid this warning.
  result = read_func(


## siguiente celda


Esta celda localiza automáticamente archivos de población dentro del proyecto, revisa su estructura y genera un inventario preliminar de las fuentes candidatas para BenMAP-CE. El objetivo es identificar qué archivo contiene la información más adecuada para construir la población expuesta en la misma unidad espacial del análisis. Para ello, el código busca archivos con nombres asociados a población en formatos \texttt{.csv}, \texttt{.xlsx} y \texttt{.parquet}, intenta cargarlos, resume el número de filas, columnas y nombres de variables, y exporta un inventario consolidado en \texttt{population_candidates_inventory.csv}. Con esta revisión previa se evita construir el insumo poblacional sobre una fuente incorrecta o incompleta y se facilita la selección de la base más consistente para la etapa siguiente.

In [4]:
# ============================================================
# CELDA 3: Buscar e inventariar archivos de población
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas base
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent

output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

inventory_file = output_dir / "population_candidates_inventory.csv"

# ------------------------------------------------------------
# 2) Buscar archivos candidatos de población
# ------------------------------------------------------------
search_roots = [
    project_root,
    project_root / "BASES DE DATOS",
    project_root / "CODIGO"
]

patterns = [
    "*poblacion*.csv",
    "*población*.csv",
    "*population*.csv",
    "*demo*.csv",
    "*pob*.csv",
    "*poblacion*.xlsx",
    "*población*.xlsx",
    "*population*.xlsx",
    "*demo*.xlsx",
    "*pob*.xlsx",
    "*poblacion*.parquet",
    "*población*.parquet",
    "*population*.parquet",
    "*demo*.parquet",
    "*pob*.parquet"
]

candidates = []
for root in search_roots:
    if root.exists():
        for pattern in patterns:
            candidates.extend(root.rglob(pattern))

# Quitar duplicados
candidates = list(dict.fromkeys(candidates))

if not candidates:
    raise FileNotFoundError(
        "No se encontraron archivos candidatos de población en el proyecto."
    )

print("Archivos candidatos encontrados:")
for i, c in enumerate(candidates, 1):
    print(f"{i}. {c}")

# ------------------------------------------------------------
# 3) Función auxiliar para cargar solo muestra y columnas
# ------------------------------------------------------------
def inspect_file(path_obj):
    suffix = path_obj.suffix.lower()
    info = {
        "archivo": str(path_obj),
        "formato": suffix,
        "ok": False,
        "n_filas": None,
        "n_columnas": None,
        "columnas": None,
        "error": None
    }
    
    try:
        if suffix == ".csv":
            df = pd.read_csv(path_obj, nrows=1000)
            # contar filas reales de forma liviana
            try:
                n_total = sum(1 for _ in open(path_obj, "r", encoding="utf-8", errors="ignore")) - 1
            except Exception:
                n_total = None
        
        elif suffix == ".xlsx":
            df = pd.read_excel(path_obj, nrows=1000)
            n_total = None
        
        elif suffix == ".parquet":
            df = pd.read_parquet(path_obj)
            n_total = len(df)
            df = df.head(1000).copy()
        
        else:
            info["error"] = "Formato no soportado en esta inspección"
            return info
        
        info["ok"] = True
        info["n_filas"] = n_total if n_total is not None else len(df)
        info["n_columnas"] = len(df.columns)
        info["columnas"] = " | ".join(map(str, df.columns.tolist()))
    
    except Exception as e:
        info["error"] = str(e)
    
    return info

# ------------------------------------------------------------
# 4) Inspeccionar candidatos
# ------------------------------------------------------------
inventory = pd.DataFrame([inspect_file(p) for p in candidates])

# Ordenar: primero los que cargaron bien
inventory = inventory.sort_values(
    by=["ok", "n_columnas", "n_filas"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Exportar inventario
inventory.to_csv(inventory_file, index=False, encoding="utf-8-sig")

print("\nResumen del inventario:")
print(inventory[["archivo", "formato", "ok", "n_filas", "n_columnas"]])

print("\nArchivo de inventario generado:")
print(inventory_file)

print("\nVista previa de columnas detectadas:")
for idx, row in inventory.head(10).iterrows():
    print(f"\n--- Candidato {idx+1} ---")
    print("Archivo:", row["archivo"])
    print("OK:", row["ok"])
    print("Columnas:", row["columnas"] if pd.notna(row["columnas"]) else "No disponible")
    if pd.notna(row["error"]):
        print("Error:", row["error"])

Archivos candidatos encontrados:
1. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS POBLACIONAL DE BOGOTA  (2020-2024)\poblacion_Bogota_2020_2024.csv
2. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS POBLACIONAL DE BOGOTA  (2020-2024)\BASE DE DATOS LIMPIA POR RANGOS ETARIOS\poblacion_bogota_rangos.csv
3. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS POBLACIONAL DE BOGOTA  (2020-2024)\poblacion_Bogota_2020_2024 (base de datos crudo).xlsx

Resumen del inventario:
                                             archivo formato    ok  n_filas  \
0  d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BAS...   .xlsx  True       17   
1  d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BAS...    .csv  True       15   
2  d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BAS...    .csv  True       75   

   n_columnas  
0         306  
1         306  
2           7  

Archivo de inventario generado:
d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\population_candidates_inventory.csv

Vista prev

# poblacion etaria

Esta celda carga la base poblacional por rangos etarios de Bogotá, la limpia y estandariza, verifica su estructura mínima y proyecta la población al año 2026 para cada grupo de edad mediante una extrapolación lineal simple con base en la serie 2020–2024. El resultado no asigna todavía la población a cada celda de la grilla, sino que construye primero una versión consolidada y coherente de la población expuesta por grupo etario, sexo y total, que servirá como insumo base para la etapa posterior de espacialización y para la preparación del archivo final de población en BenMAP-CE. Además, la celda exporta dos productos: una base limpia histórica y una proyección poblacional para 2026.

In [5]:
# ============================================================
# CELDA 4: Limpiar población por rangos etarios y proyectar 2026
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent

output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

input_file = project_root / "BASES DE DATOS" / "BASE DE DATOS POBLACIONAL DE BOGOTA  (2020-2024)" / "BASE DE DATOS LIMPIA POR RANGOS ETARIOS" / "poblacion_bogota_rangos.csv"

if not input_file.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo esperado:\n{input_file}"
    )

population_clean_file = output_dir / "population_bogota_rangos_clean.csv"
population_2026_file = output_dir / "population_bogota_rangos_2026.csv"

# ------------------------------------------------------------
# 2) Cargar archivo
# ------------------------------------------------------------
df = pd.read_csv(input_file)

print("Archivo cargado:", input_file)
print("Dimensión:", df.shape)
print("Columnas originales:", df.columns.tolist())

# ------------------------------------------------------------
# 3) Verificar columnas mínimas
# ------------------------------------------------------------
required_cols = ["MPIO", "DPMP", "AÑO", "GrupoEdad", "Hombres", "Mujeres", "Total"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Faltan columnas obligatorias en la base poblacional: {missing}")

# ------------------------------------------------------------
# 4) Limpiar y estandarizar
# ------------------------------------------------------------
df = df[required_cols].copy()

df = df.rename(columns={
    "AÑO": "year",
    "GrupoEdad": "age_group",
    "Hombres": "male",
    "Mujeres": "female",
    "Total": "total",
    "MPIO": "mpio",
    "DPMP": "dpmp"
})

# Estandarizar tipos
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["male"] = pd.to_numeric(df["male"], errors="coerce")
df["female"] = pd.to_numeric(df["female"], errors="coerce")
df["total"] = pd.to_numeric(df["total"], errors="coerce")

df["age_group"] = df["age_group"].astype(str).str.strip()

# Eliminar filas inválidas
df = df.dropna(subset=["year", "age_group", "male", "female", "total"]).copy()

df["year"] = df["year"].astype(int)
df["male"] = df["male"].astype(float)
df["female"] = df["female"].astype(float)
df["total"] = df["total"].astype(float)

# Forzar no negatividad
for col in ["male", "female", "total"]:
    df[col] = df[col].clip(lower=0)

# ------------------------------------------------------------
# 5) Verificación de consistencia
# ------------------------------------------------------------
df["total_diff"] = df["total"] - (df["male"] + df["female"])

max_diff = df["total_diff"].abs().max()
print("\nMáxima diferencia entre total y hombres+mujeres:", max_diff)

# Si hay pequeñas diferencias por redondeo, corregimos total
df["total"] = df["male"] + df["female"]

# ------------------------------------------------------------
# 6) Guardar base limpia histórica
# ------------------------------------------------------------
df_clean = df[["mpio", "dpmp", "year", "age_group", "male", "female", "total"]].copy()
df_clean = df_clean.sort_values(["year", "age_group"]).reset_index(drop=True)

df_clean.to_csv(population_clean_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 7) Proyectar a 2026 por grupo etario
#    Se usa una tendencia lineal simple con datos disponibles
# ------------------------------------------------------------
target_year = 2026
projection_rows = []

for age_group, g in df_clean.groupby("age_group"):
    g = g.sort_values("year").copy()

    # Asegurar años únicos por grupo
    g = g.groupby(["age_group", "year"], as_index=False)[["male", "female", "total"]].sum()

    x = g["year"].values.astype(float)

    # Solo si hay al menos 2 años, se ajusta tendencia lineal
    if len(g) >= 2:
        male_coef = np.polyfit(x, g["male"].values, 1)
        female_coef = np.polyfit(x, g["female"].values, 1)
        total_coef = np.polyfit(x, g["total"].values, 1)

        male_pred = np.polyval(male_coef, target_year)
        female_pred = np.polyval(female_coef, target_year)
        total_pred = np.polyval(total_coef, target_year)
    else:
        # Si solo hay un año, se repite el último valor
        male_pred = g["male"].iloc[-1]
        female_pred = g["female"].iloc[-1]
        total_pred = g["total"].iloc[-1]

    male_pred = max(male_pred, 0)
    female_pred = max(female_pred, 0)

    # Recalcular total para asegurar consistencia
    total_pred = male_pred + female_pred

    projection_rows.append({
        "mpio": g["mpio"].iloc[0] if "mpio" in g.columns else df_clean["mpio"].iloc[0],
        "dpmp": g["dpmp"].iloc[0] if "dpmp" in g.columns else df_clean["dpmp"].iloc[0],
        "year": target_year,
        "age_group": age_group,
        "male": male_pred,
        "female": female_pred,
        "total": total_pred
    })

df_2026 = pd.DataFrame(projection_rows)

# ------------------------------------------------------------
# 8) Redondear y ordenar
# ------------------------------------------------------------
for col in ["male", "female", "total"]:
    df_2026[col] = df_2026[col].round(0).astype(int)

df_2026 = df_2026.sort_values("age_group").reset_index(drop=True)

df_2026.to_csv(population_2026_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Resumen
# ------------------------------------------------------------
summary_hist = df_clean.groupby("year", as_index=False)["total"].sum()
summary_2026 = df_2026["total"].sum()

print("\nResumen histórico de población total:")
print(summary_hist)

print("\nPoblación total proyectada para 2026:", summary_2026)

print("\nPrimeras filas de la proyección 2026:")
print(df_2026.head())

print("\nArchivos generados:")
print(" -", population_clean_file)
print(" -", population_2026_file)

Archivo cargado: d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\BASE DE DATOS POBLACIONAL DE BOGOTA  (2020-2024)\BASE DE DATOS LIMPIA POR RANGOS ETARIOS\poblacion_bogota_rangos.csv
Dimensión: (75, 7)
Columnas originales: ['MPIO', 'DPMP', 'AÑO', 'GrupoEdad', 'Hombres', 'Mujeres', 'Total']

Máxima diferencia entre total y hombres+mujeres: 0.0

Resumen histórico de población total:
   year       total
0  2020  15437000.0
1  2021  15611484.0
2  2022  15699774.0
3  2023  15769026.0
4  2024  15838570.0

Población total proyectada para 2026: 16055444

Primeras filas de la proyección 2026:
    mpio          dpmp  year age_group     male   female    total
0  11001  Bogotá, D.C.  2026       0-4   437385   415064   852449
1  11001  Bogotá, D.C.  2026     15-44  3995564  3965513  7961078
2  11001  Bogotá, D.C.  2026     45-64  1651487  1951952  3603439
3  11001  Bogotá, D.C.  2026      5-14   992173   942245  1934418
4  11001  Bogotá, D.C.  2026       65+   699016  1005044  1704060

Archivos generados

# poblacion compatible con benmap ce

Esta celda transforma la proyección poblacional de 2026 en un archivo espacial compatible con la malla de BenMAP-CE. Como todavía no se ha incorporado una fuente de distribución espacial más detallada por localidad o celda, la asignación se realiza de manera provisional usando la participación de área de cada celda dentro de la grilla de 3 km. De esta forma, la población total proyectada por grupo etario se reparte entre las 254 celdas según su peso relativo en el área total del dominio. El procedimiento preserva exactamente los totales por grupo etario mediante un ajuste de redondeo y exporta el archivo \texttt{benmap_population_2026.csv}, que servirá como insumo técnico inicial para BenMAP-CE. Esta versión permite probar el flujo completo del modelo, aunque más adelante conviene reemplazarla por una distribución espacial basada en población por localidad o por celda si esa información está disponible.

In [8]:
# ============================================================
# CELDA 5: Construir benmap_population_2026.csv
#          usando distribución espacial provisional por área
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent

output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

population_2026_file = output_dir / "population_bogota_rangos_2026.csv"
grid_summary_file = output_dir / "benmap_grid_definition_summary.csv"

if not population_2026_file.exists():
    raise FileNotFoundError(
        f"No existe el archivo esperado de población 2026:\n{population_2026_file}"
    )

if not grid_summary_file.exists():
    raise FileNotFoundError(
        f"No existe el archivo esperado de resumen de grilla:\n{grid_summary_file}"
    )

benmap_population_file = output_dir / "benmap_population_2026.csv"

# ------------------------------------------------------------
# 2) Cargar insumos
# ------------------------------------------------------------
pop = pd.read_csv(population_2026_file)
grid = pd.read_csv(grid_summary_file)

print("Archivo de población cargado:", population_2026_file)
print("Dimensión población:", pop.shape)

print("\nArchivo de grilla cargado:", grid_summary_file)
print("Dimensión grilla:", grid.shape)

# ------------------------------------------------------------
# 3) Verificar columnas mínimas
# ------------------------------------------------------------
required_pop_cols = ["year", "age_group", "male", "female", "total"]
required_grid_cols = ["cell_id", "area_m2"]

missing_pop = [c for c in required_pop_cols if c not in pop.columns]
missing_grid = [c for c in required_grid_cols if c not in grid.columns]

if missing_pop:
    raise ValueError(f"Faltan columnas en población: {missing_pop}")
if missing_grid:
    raise ValueError(f"Faltan columnas en grilla: {missing_grid}")

# ------------------------------------------------------------
# 4) Limpiar y estandarizar
# ------------------------------------------------------------
pop = pop.copy()
grid = grid.copy()

pop["year"] = pd.to_numeric(pop["year"], errors="coerce").astype("Int64")
pop["male"] = pd.to_numeric(pop["male"], errors="coerce")
pop["female"] = pd.to_numeric(pop["female"], errors="coerce")
pop["total"] = pd.to_numeric(pop["total"], errors="coerce")
pop["age_group"] = pop["age_group"].astype(str).str.strip()

grid["cell_id"] = pd.to_numeric(grid["cell_id"], errors="coerce").astype("Int64")
grid["area_m2"] = pd.to_numeric(grid["area_m2"], errors="coerce")

pop = pop.dropna(subset=["year", "age_group", "male", "female", "total"]).copy()
grid = grid.dropna(subset=["cell_id", "area_m2"]).copy()

pop["year"] = pop["year"].astype(int)
grid["cell_id"] = grid["cell_id"].astype(int)

grid = grid[grid["area_m2"] > 0].copy()

if grid.empty:
    raise ValueError("La grilla quedó vacía después de filtrar áreas positivas.")

# ------------------------------------------------------------
# 5) Calcular pesos espaciales por área
# ------------------------------------------------------------
grid = grid.sort_values("cell_id").reset_index(drop=True)
grid["area_weight"] = grid["area_m2"] / grid["area_m2"].sum()

print("\nNúmero de celdas válidas:", len(grid))
print("Suma de pesos espaciales:", grid["area_weight"].sum())

# ------------------------------------------------------------
# 6) Función para asignar enteros preservando total exacto
#    Método: floor + largest remainder
# ------------------------------------------------------------
def allocate_integer_total(total_value, weights):
    total_value = int(round(total_value))
    raw = total_value * weights
    base = np.floor(raw).astype(int)
    remainder = raw - base
    diff = total_value - base.sum()

    if diff > 0:
        idx = np.argsort(-remainder)[:diff]
        base[idx] += 1
    elif diff < 0:
        idx = np.argsort(remainder)[:abs(diff)]
        base[idx] -= 1

    return base

# ------------------------------------------------------------
# 7) Distribuir población 2026 a la grilla
# ------------------------------------------------------------
rows = []

for _, row in pop.iterrows():
    age_group = row["age_group"]
    year = int(row["year"])

    male_alloc = allocate_integer_total(row["male"], grid["area_weight"].values)
    female_alloc = allocate_integer_total(row["female"], grid["area_weight"].values)
    total_alloc = male_alloc + female_alloc

    tmp = pd.DataFrame({
        "cell_id": grid["cell_id"].values,
        "year": year,
        "age_group": age_group,
        "male": male_alloc,
        "female": female_alloc,
        "total": total_alloc,
        "area_weight": grid["area_weight"].values,
        "allocation_method": "area_weighted_provisional"
    })

    rows.append(tmp)

benmap_pop = pd.concat(rows, ignore_index=True)

# ------------------------------------------------------------
# 8) Ordenar y exportar
# ------------------------------------------------------------
benmap_pop = benmap_pop.sort_values(["year", "age_group", "cell_id"]).reset_index(drop=True)
benmap_pop.to_csv(benmap_population_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Validaciones
# ------------------------------------------------------------
check_original = pop.groupby(["year", "age_group"], as_index=False)[["male", "female", "total"]].sum()
check_alloc = benmap_pop.groupby(["year", "age_group"], as_index=False)[["male", "female", "total"]].sum()

check = check_original.merge(
    check_alloc,
    on=["year", "age_group"],
    suffixes=("_original", "_allocated")
)

check["diff_male"] = check["male_allocated"] - check["male_original"]
check["diff_female"] = check["female_allocated"] - check["female_original"]
check["diff_total"] = check["total_allocated"] - check["total_original"]

print("\nValidación de preservación de totales por grupo etario:")
print(check[["year", "age_group", "diff_male", "diff_female", "diff_total"]])

print("\nPrimeras filas del archivo final de población BenMAP:")
print(benmap_pop.head())

print("\nResumen general:")
print("Filas exportadas:", len(benmap_pop))
print("Celdas únicas:", benmap_pop["cell_id"].nunique())
print("Grupos etarios:", benmap_pop["age_group"].nunique())
print("Año(s):", benmap_pop["year"].unique().tolist())
print("Población total final:", int(benmap_pop["total"].sum()))

print("\nArchivo generado:")
print(" -", benmap_population_file)

Archivo de población cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\population_bogota_rangos_2026.csv
Dimensión población: (5, 7)

Archivo de grilla cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_summary.csv
Dimensión grilla: (254, 4)

Número de celdas válidas: 254
Suma de pesos espaciales: 0.9999999999999999

Validación de preservación de totales por grupo etario:
   year age_group  diff_male  diff_female  diff_total
0  2026       0-4          0            0           0
1  2026     15-44          0            0           0
2  2026     45-64          0            0           0
3  2026      5-14          0            0           0
4  2026       65+          0            0           0

Primeras filas del archivo final de población BenMAP:
   cell_id  year age_group  male  female  total  area_weight  \
0        0  2026       0-4  1798    1706   3504     0.004111   
1        1  2026       0-4   977     927   1904     0.002233   
2        2  2026       0-4  

## correccion distribucion espacial (si se corrigio volver a correr celda 5)

Esta celda corrige la proyección poblacional 2026 para asegurar consistencia interna entre las columnas de hombres, mujeres y total. El ajuste consiste en recalcular la columna \texttt{total} como la suma exacta de \texttt{male + female}, evitando discrepancias de una unidad causadas por redondeos independientes en la celda anterior. Después de guardar el archivo corregido, se puede volver a ejecutar la celda de distribución espacial para que la validación final de población en BenMAP-CE quede exacta en todos los grupos etarios.

In [7]:
# ============================================================
# CELDA 4B: Corregir total = male + female en población 2026
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent

output_dir = project_root / "SALIDAS_BENMAP"
population_2026_file = output_dir / "population_bogota_rangos_2026.csv"

if not population_2026_file.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo esperado:\n{population_2026_file}"
    )

# ------------------------------------------------------------
# 2) Cargar archivo
# ------------------------------------------------------------
df = pd.read_csv(population_2026_file)

print("Archivo cargado:", population_2026_file)
print("Dimensión:", df.shape)

required_cols = ["male", "female", "total"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}")

# ------------------------------------------------------------
# 3) Corregir total
# ------------------------------------------------------------
df["male"] = pd.to_numeric(df["male"], errors="coerce").fillna(0)
df["female"] = pd.to_numeric(df["female"], errors="coerce").fillna(0)
df["total_original"] = pd.to_numeric(df["total"], errors="coerce").fillna(0)

df["total_corregido"] = df["male"] + df["female"]
df["diferencia"] = df["total_corregido"] - df["total_original"]

print("\nDiferencias detectadas antes de corregir:")
print(df[["age_group", "male", "female", "total_original", "total_corregido", "diferencia"]])

# Reemplazar total por el consistente
df["total"] = df["total_corregido"]

# Limpiar columnas auxiliares
df = df.drop(columns=["total_original", "total_corregido", "diferencia"])

# ------------------------------------------------------------
# 4) Guardar archivo corregido
# ------------------------------------------------------------
df.to_csv(population_2026_file, index=False, encoding="utf-8-sig")

print("\nArchivo corregido guardado en:")
print(population_2026_file)

print("\nVista previa corregida:")
print(df.head())

Archivo cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\population_bogota_rangos_2026.csv
Dimensión: (5, 7)

Diferencias detectadas antes de corregir:
  age_group     male   female  total_original  total_corregido  diferencia
0       0-4   437385   415064          852449           852449           0
1     15-44  3995564  3965513         7961078          7961077          -1
2     45-64  1651487  1951952         3603439          3603439           0
3      5-14   992173   942245         1934418          1934418           0
4       65+   699016  1005044         1704060          1704060           0

Archivo corregido guardado en:
d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\population_bogota_rangos_2026.csv

Vista previa corregida:
    mpio          dpmp  year age_group     male   female    total
0  11001  Bogotá, D.C.  2026       0-4   437385   415064   852449
1  11001  Bogotá, D.C.  2026     15-44  3995564  3965513  7961077
2  11001  Bogotá, D.C.  2026     45-64  1651487  1951952  36034

# incidencia

In [9]:
# ============================================================
# CELDA 6: Buscar e inventariar archivos candidatos de incidencia
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas base
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent

output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

inventory_file = output_dir / "incidence_candidates_inventory.csv"

# ------------------------------------------------------------
# 2) Buscar archivos candidatos de salud / incidencia
# ------------------------------------------------------------
search_roots = [
    project_root,
    project_root / "BASES DE DATOS",
    project_root / "CODIGO"
]

patterns = [
    "*incidencia*.csv",
    "*incidence*.csv",
    "*mortalidad*.csv",
    "*morbilidad*.csv",
    "*defunc*.csv",
    "*muert*.csv",
    "*salud*.csv",
    "*rips*.csv",
    "*hospital*.csv",
    "*urgencias*.csv",
    "*respir*.csv",
    "*cardio*.csv",
    "*incidencia*.xlsx",
    "*incidence*.xlsx",
    "*mortalidad*.xlsx",
    "*morbilidad*.xlsx",
    "*defunc*.xlsx",
    "*muert*.xlsx",
    "*salud*.xlsx",
    "*rips*.xlsx",
    "*hospital*.xlsx",
    "*urgencias*.xlsx",
    "*respir*.xlsx",
    "*cardio*.xlsx",
    "*incidencia*.parquet",
    "*incidence*.parquet",
    "*mortalidad*.parquet",
    "*morbilidad*.parquet",
    "*defunc*.parquet",
    "*muert*.parquet",
    "*salud*.parquet",
    "*rips*.parquet",
    "*hospital*.parquet",
    "*urgencias*.parquet",
    "*respir*.parquet",
    "*cardio*.parquet"
]

candidates = []
for root in search_roots:
    if root.exists():
        for pattern in patterns:
            candidates.extend(root.rglob(pattern))

# Quitar duplicados
candidates = list(dict.fromkeys(candidates))

if not candidates:
    raise FileNotFoundError(
        "No se encontraron archivos candidatos de incidencia/salud en el proyecto."
    )

print("Archivos candidatos encontrados:")
for i, c in enumerate(candidates, 1):
    print(f"{i}. {c}")

# ------------------------------------------------------------
# 3) Función auxiliar para inspeccionar archivos
# ------------------------------------------------------------
def inspect_file(path_obj):
    suffix = path_obj.suffix.lower()
    info = {
        "archivo": str(path_obj),
        "formato": suffix,
        "ok": False,
        "n_filas": None,
        "n_columnas": None,
        "columnas": None,
        "error": None
    }

    try:
        if suffix == ".csv":
            df = pd.read_csv(path_obj, nrows=1000)
            try:
                n_total = sum(1 for _ in open(path_obj, "r", encoding="utf-8", errors="ignore")) - 1
            except Exception:
                n_total = None

        elif suffix == ".xlsx":
            df = pd.read_excel(path_obj, nrows=1000)
            n_total = None

        elif suffix == ".parquet":
            df = pd.read_parquet(path_obj)
            n_total = len(df)
            df = df.head(1000).copy()

        else:
            info["error"] = "Formato no soportado en esta inspección"
            return info

        info["ok"] = True
        info["n_filas"] = n_total if n_total is not None else len(df)
        info["n_columnas"] = len(df.columns)
        info["columnas"] = " | ".join(map(str, df.columns.tolist()))

    except Exception as e:
        info["error"] = str(e)

    return info

# ------------------------------------------------------------
# 4) Inspeccionar candidatos
# ------------------------------------------------------------
inventory = pd.DataFrame([inspect_file(p) for p in candidates])

inventory = inventory.sort_values(
    by=["ok", "n_columnas", "n_filas"],
    ascending=[False, False, False]
).reset_index(drop=True)

inventory.to_csv(inventory_file, index=False, encoding="utf-8-sig")

print("\nResumen del inventario:")
print(inventory[["archivo", "formato", "ok", "n_filas", "n_columnas"]])

print("\nArchivo de inventario generado:")
print(inventory_file)

print("\nVista previa de columnas detectadas:")
for idx, row in inventory.head(12).iterrows():
    print(f"\n--- Candidato {idx+1} ---")
    print("Archivo:", row["archivo"])
    print("OK:", row["ok"])
    print("Columnas:", row["columnas"] if pd.notna(row["columnas"]) else "No disponible")
    if pd.notna(row["error"]):
        print("Error:", row["error"])

Archivos candidatos encontrados:
1. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MORBILIDAD Y MORTALIDAD\DATOS TRATADOS\MORTALIDAD\Bogota_panel_mortalidad_2020_2024.csv
2. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MORBILIDAD Y MORTALIDAD\DATOS CRUDO\MORBILIDAD\metadato-morbilidad-bogota-identificada-en-rips.csv
3. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MORBILIDAD Y MORTALIDAD\DATOS CRUDO\MORBILIDAD\descargable_rips_general_2020_1.csv
4. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MORBILIDAD Y MORTALIDAD\DATOS CRUDO\MORBILIDAD\descargable_rips_general_2020_2.csv
5. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MORBILIDAD Y MORTALIDAD\DATOS CRUDO\MORBILIDAD\descargable_rips_general_2021_1.csv
6. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MORBILIDAD Y MORTALIDAD\DATOS CRUDO\MORBILIDAD\descargable_rips_general_2021_2.csv
7. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DATOS\MORBILIDAD Y MORTALIDAD\DATOS CRUDO\MORBILIDAD\descargable_rips_general_2022_1.csv
8. d:\TRABAJO DE GRADO BEN-MAP\BASES DE DAT

# tratamiento panel mortalidad

Esta celda reconstruye nuevamente el panel tratado de mortalidad, pero además de limpiar los encabezados elimina comillas residuales y espacios en los valores de todas las columnas. Esto es necesario porque, aunque la tabla ya se separó correctamente por comas, algunas celdas conservaron caracteres como comillas dobles al inicio o al final, lo que impide convertir los datos a formato numérico. Después de esa limpieza, el código identifica las columnas \texttt{year}, \texttt{def_I} y \texttt{def_J}, convierte sus valores a numéricos, conserva únicamente las filas válidas y exporta un panel limpio listo para proyectar los endpoints al año 2026 y construir el archivo de incidencia para BenMAP-CE.

In [13]:
# ============================================================
# CELDA 7A FINAL: Limpiar encabezados y valores del panel
# tratado de mortalidad
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

mortality_file = (
    project_root
    / "BASES DE DATOS"
    / "MORBILIDAD Y MORTALIDAD"
    / "DATOS TRATADOS"
    / "MORTALIDAD"
    / "Bogota_panel_mortalidad_2020_2024.csv"
)

mortality_clean_file = output_dir / "mortality_panel_endpoints_clean.csv"

if not mortality_file.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{mortality_file}")

# ------------------------------------------------------------
# 2) Leer archivo bruto como texto
# ------------------------------------------------------------
encodings_to_try = ["utf-8", "utf-8-sig", "latin1", "cp1252"]

raw_lines = None
used_encoding = None

for enc in encodings_to_try:
    try:
        with open(mortality_file, "r", encoding=enc, errors="strict") as f:
            raw_lines = f.read().splitlines()
        used_encoding = enc
        break
    except Exception:
        continue

if raw_lines is None:
    raise ValueError("No fue posible leer el archivo de mortalidad con las codificaciones probadas.")

print("Archivo leído correctamente con codificación:", used_encoding)
print("Número de líneas detectadas:", len(raw_lines))

if len(raw_lines) < 2:
    raise ValueError("El archivo no tiene suficientes filas para reconstruir una tabla.")

# ------------------------------------------------------------
# 3) Reconstruir tabla separando por coma
# ------------------------------------------------------------
split_rows = [line.split(",") for line in raw_lines]

max_len = max(len(r) for r in split_rows)
split_rows = [r + [""] * (max_len - len(r)) for r in split_rows]

df = pd.DataFrame(split_rows[1:], columns=split_rows[0])

print("\nDimensión reconstruida:", df.shape)
print("Primeras columnas antes de limpiar:", df.columns.tolist()[:15])

# ------------------------------------------------------------
# 4) Limpiar encabezados
# ------------------------------------------------------------
df.columns = [
    str(c)
    .replace("\ufeff", "")
    .replace('"', "")
    .replace("'", "")
    .strip()
    for c in df.columns
]

# Renombrar año
rename_candidates = {
    "AÃ±o": "year",
    "Año": "year",
    "ANO": "year",
    "ano": "year",
    "año": "year"
}

for old, new in rename_candidates.items():
    if old in df.columns:
        df = df.rename(columns={old: new})

if "year" not in df.columns:
    for c in df.columns:
        c_low = str(c).lower()
        if "a" in c_low and ("ño" in c_low or "ã" in c_low or c_low == "ano"):
            df = df.rename(columns={c: "year"})
            break

print("\nPrimeras columnas después de limpiar:", df.columns.tolist()[:15])

# ------------------------------------------------------------
# 5) Limpiar valores de texto en TODAS las columnas
# ------------------------------------------------------------
for col in df.columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('"', "", regex=False)
        .str.replace("'", "", regex=False)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

# ------------------------------------------------------------
# 6) Verificar columnas necesarias
# ------------------------------------------------------------
required_cols = ["year", "def_I", "def_J"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        f"No se identificaron las columnas esperadas: {missing}\n"
        f"Columnas disponibles: {df.columns.tolist()}"
    )

print("\nVista previa cruda de columnas objetivo:")
print(df[["year", "def_I", "def_J"]].head())

# ------------------------------------------------------------
# 7) Convertir a numérico
# ------------------------------------------------------------
for col in ["year", "def_I", "def_J"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nCantidad de NA después de convertir a numérico:")
print(df[["year", "def_I", "def_J"]].isna().sum())

# ------------------------------------------------------------
# 8) Filtrar filas válidas
# ------------------------------------------------------------
df = df.dropna(subset=["year", "def_I", "def_J"]).copy()

df["year"] = df["year"].astype(int)
df["def_I"] = df["def_I"].astype(float)
df["def_J"] = df["def_J"].astype(float)

df = df.sort_values("year").reset_index(drop=True)

# ------------------------------------------------------------
# 9) Exportar panel limpio
# ------------------------------------------------------------
df = df[["year", "def_I", "def_J"]].copy()
df.to_csv(mortality_clean_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 10) Mostrar resultado
# ------------------------------------------------------------
print("\nPanel limpio de mortalidad:")
print(df)

print("\nArchivo generado:")
print(" -", mortality_clean_file)

Archivo leído correctamente con codificación: utf-8
Número de líneas detectadas: 6

Dimensión reconstruida: (5, 108)
Primeras columnas antes de limpiar: ['\ufeff"AÃ±o', 'c_001', 'c_002', 'c_003', 'c_004', 'c_005', 'c_006', 'c_007', 'c_008', 'c_009', 'c_010', 'c_011', 'c_012', 'c_013', 'c_014']

Primeras columnas después de limpiar: ['year', 'c_001', 'c_002', 'c_003', 'c_004', 'c_005', 'c_006', 'c_007', 'c_008', 'c_009', 'c_010', 'c_011', 'c_012', 'c_013', 'c_014']

Vista previa cruda de columnas objetivo:
   year  def_I def_J
0  2020   7630  1809
1  2021  11809  1935
2  2022   1934  1859
3  2023    614  1879
4  2024    553  2093

Cantidad de NA después de convertir a numérico:
year     0
def_I    0
def_J    0
dtype: int64

Panel limpio de mortalidad:
   year    def_I   def_J
0  2020   7630.0  1809.0
1  2021  11809.0  1935.0
2  2022   1934.0  1859.0
3  2023    614.0  1879.0
4  2024    553.0  2093.0

Archivo generado:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\mortality_panel_endpoint

# proyecta al año 2026 los endpoints \texttt{def_I} y \texttt{def_J} mediante una tendencia lineal simple y construye un archivo provisional de incidencia para BenMAP-CE.

Esta celda toma el panel limpio de mortalidad, proyecta al año 2026 los endpoints \texttt{def_I} y \texttt{def_J} mediante una tendencia lineal simple y construye un archivo provisional de incidencia para BenMAP-CE. Para ello, convierte los conteos proyectados en tasas anuales usando como denominador la población total de 2026 previamente consolidada y replica esa tasa de forma uniforme en todas las celdas de la grilla. Este archivo no representa todavía una distribución espacial diferencial del riesgo, sino una aproximación inicial coherente con la información disponible, útil para avanzar en el flujo completo de BenMAP-CE y posteriormente refinar la especificación epidemiológica.

In [14]:
# ============================================================
# CELDA 7B: Proyectar endpoints a 2026 y construir incidencia
# provisional para BenMAP-CE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

mortality_clean_file = output_dir / "mortality_panel_endpoints_clean.csv"
population_file = output_dir / "benmap_population_2026.csv"
grid_file = output_dir / "benmap_grid_definition_summary.csv"

mortality_2026_file = output_dir / "mortality_panel_endpoints_2026.csv"
incidence_file = output_dir / "benmap_incidence_2026.csv"

for f in [mortality_clean_file, population_file, grid_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

# ------------------------------------------------------------
# 2) Cargar insumos
# ------------------------------------------------------------
mort = pd.read_csv(mortality_clean_file)
pop = pd.read_csv(population_file)
grid = pd.read_csv(grid_file)

print("Panel de mortalidad cargado:", mortality_clean_file)
print("Dimensión:", mort.shape)

print("\nArchivo de población cargado:", population_file)
print("Dimensión:", pop.shape)

print("\nArchivo de grilla cargado:", grid_file)
print("Dimensión:", grid.shape)

# ------------------------------------------------------------
# 3) Verificar columnas mínimas
# ------------------------------------------------------------
required_mort_cols = ["year", "def_I", "def_J"]
required_pop_cols = ["cell_id", "year", "total"]
required_grid_cols = ["cell_id"]

missing_mort = [c for c in required_mort_cols if c not in mort.columns]
missing_pop = [c for c in required_pop_cols if c not in pop.columns]
missing_grid = [c for c in required_grid_cols if c not in grid.columns]

if missing_mort:
    raise ValueError(f"Faltan columnas en mortalidad: {missing_mort}")
if missing_pop:
    raise ValueError(f"Faltan columnas en población: {missing_pop}")
if missing_grid:
    raise ValueError(f"Faltan columnas en grilla: {missing_grid}")

# ------------------------------------------------------------
# 4) Limpiar tipos
# ------------------------------------------------------------
for col in ["year", "def_I", "def_J"]:
    mort[col] = pd.to_numeric(mort[col], errors="coerce")

for col in ["cell_id", "year", "total"]:
    pop[col] = pd.to_numeric(pop[col], errors="coerce")

grid["cell_id"] = pd.to_numeric(grid["cell_id"], errors="coerce")

mort = mort.dropna(subset=["year", "def_I", "def_J"]).copy()
pop = pop.dropna(subset=["cell_id", "year", "total"]).copy()
grid = grid.dropna(subset=["cell_id"]).copy()

mort["year"] = mort["year"].astype(int)
pop["cell_id"] = pop["cell_id"].astype(int)
pop["year"] = pop["year"].astype(int)
grid["cell_id"] = grid["cell_id"].astype(int)

mort = mort.sort_values("year").reset_index(drop=True)

# ------------------------------------------------------------
# 5) Proyección lineal a 2026
# ------------------------------------------------------------
target_year = 2026
x = mort["year"].values.astype(float)

def project_linear(y_values, x_values, target):
    if len(y_values) >= 2:
        coef = np.polyfit(x_values, y_values, 1)
        pred = np.polyval(coef, target)
    else:
        pred = y_values[-1]
    return max(float(pred), 0.0)

pred_def_I = project_linear(mort["def_I"].values.astype(float), x, target_year)
pred_def_J = project_linear(mort["def_J"].values.astype(float), x, target_year)

mort_2026 = pd.DataFrame({
    "year": [target_year, target_year],
    "endpoint_code": ["def_I", "def_J"],
    "count_2026": [round(pred_def_I), round(pred_def_J)],
    "projection_method": ["linear_trend_2020_2024", "linear_trend_2020_2024"]
})

mort_2026.to_csv(mortality_2026_file, index=False, encoding="utf-8-sig")

print("\nProyección 2026 de endpoints:")
print(mort_2026)

# ------------------------------------------------------------
# 6) Calcular población total 2026
# ------------------------------------------------------------
pop_2026 = pop[pop["year"] == target_year].copy()

if pop_2026.empty:
    raise ValueError("No se encontraron registros de población para 2026.")

total_population_2026 = int(pop_2026["total"].sum())

if total_population_2026 <= 0:
    raise ValueError("La población total 2026 es inválida.")

print("\nPoblación total 2026 usada como denominador:", total_population_2026)

# ------------------------------------------------------------
# 7) Construir incidencia provisional uniforme por celda
# ------------------------------------------------------------
cell_ids = sorted(grid["cell_id"].unique())

rows = []
for _, r in mort_2026.iterrows():
    endpoint_code = r["endpoint_code"]
    count_2026 = float(r["count_2026"])
    incidence_rate = count_2026 / total_population_2026  # tasa anual por persona

    for cell_id in cell_ids:
        rows.append({
            "cell_id": cell_id,
            "year": target_year,
            "endpoint_code": endpoint_code,
            "age_group": "ALL",
            "incidence_rate": incidence_rate,
            "count_basis_citywide_2026": int(round(count_2026)),
            "population_basis_citywide_2026": total_population_2026,
            "incidence_source": "Bogota_panel_mortalidad_2020_2024",
            "incidence_method": "citywide_uniform_rate_provisional"
        })

incidence = pd.DataFrame(rows)
incidence = incidence.sort_values(["endpoint_code", "cell_id"]).reset_index(drop=True)

# ------------------------------------------------------------
# 8) Exportar archivo final
# ------------------------------------------------------------
incidence.to_csv(incidence_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Resumen
# ------------------------------------------------------------
summary = incidence.groupby("endpoint_code", as_index=False).agg(
    n_rows=("cell_id", "count"),
    n_cells=("cell_id", "nunique"),
    incidence_rate=("incidence_rate", "first"),
    count_basis_citywide_2026=("count_basis_citywide_2026", "first"),
    population_basis_citywide_2026=("population_basis_citywide_2026", "first")
)

print("\nResumen de incidencia provisional BenMAP:")
print(summary)

print("\nPrimeras filas del archivo final:")
print(incidence.head())

print("\nArchivos generados:")
print(" -", mortality_2026_file)
print(" -", incidence_file)

Panel de mortalidad cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\mortality_panel_endpoints_clean.csv
Dimensión: (5, 3)

Archivo de población cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_population_2026.csv
Dimensión: (1270, 8)

Archivo de grilla cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_summary.csv
Dimensión: (254, 4)

Proyección 2026 de endpoints:
   year endpoint_code  count_2026       projection_method
0  2026         def_I           0  linear_trend_2020_2024
1  2026         def_J        2120  linear_trend_2020_2024

Población total 2026 usada como denominador: 16055443

Resumen de incidencia provisional BenMAP:
  endpoint_code  n_rows  n_cells  incidence_rate  count_basis_citywide_2026  \
0         def_I     254      254        0.000000                          0   
1         def_J     254      254        0.000132                       2120   

   population_basis_citywide_2026  
0                        16055443  
1            

# proyección provisional de incidencia para BenMAP-CE

Esta celda corrige la proyección provisional de incidencia para BenMAP-CE cuando una extrapolación lineal produce conteos nulos o negativos. En lugar de mantener esos valores extremos, el código aplica una regla más robusta: si la tendencia lineal proyectada para 2026 es positiva, se conserva; si resulta nula o negativa, se sustituye por el promedio de los últimos tres años observados. Este ajuste es especialmente útil cuando la serie presenta cambios abruptos o caídas muy marcadas que hacen inestable la extrapolación lineal. Con ello se obtiene una proyección más razonable de los endpoints \texttt{def_I} y \texttt{def_J}, y se reconstruye el archivo \texttt{benmap_incidence_2026.csv} con tasas provisionales más estables para su uso en BenMAP-CE.

In [15]:
# ============================================================
# CELDA 7C: Reproyectar endpoints 2026 con regla robusta
#          y reconstruir benmap_incidence_2026.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

mortality_clean_file = output_dir / "mortality_panel_endpoints_clean.csv"
population_file = output_dir / "benmap_population_2026.csv"
grid_file = output_dir / "benmap_grid_definition_summary.csv"

mortality_2026_file = output_dir / "mortality_panel_endpoints_2026.csv"
incidence_file = output_dir / "benmap_incidence_2026.csv"

for f in [mortality_clean_file, population_file, grid_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

# ------------------------------------------------------------
# 2) Cargar insumos
# ------------------------------------------------------------
mort = pd.read_csv(mortality_clean_file)
pop = pd.read_csv(population_file)
grid = pd.read_csv(grid_file)

# ------------------------------------------------------------
# 3) Limpiar tipos
# ------------------------------------------------------------
for col in ["year", "def_I", "def_J"]:
    mort[col] = pd.to_numeric(mort[col], errors="coerce")

for col in ["cell_id", "year", "total"]:
    pop[col] = pd.to_numeric(pop[col], errors="coerce")

grid["cell_id"] = pd.to_numeric(grid["cell_id"], errors="coerce")

mort = mort.dropna(subset=["year", "def_I", "def_J"]).copy()
pop = pop.dropna(subset=["cell_id", "year", "total"]).copy()
grid = grid.dropna(subset=["cell_id"]).copy()

mort["year"] = mort["year"].astype(int)
pop["cell_id"] = pop["cell_id"].astype(int)
pop["year"] = pop["year"].astype(int)
grid["cell_id"] = grid["cell_id"].astype(int)

mort = mort.sort_values("year").reset_index(drop=True)

# ------------------------------------------------------------
# 4) Función de proyección robusta
#    - usa tendencia lineal si es positiva
#    - si no, usa promedio de últimos 3 años
# ------------------------------------------------------------
def robust_projection(series_years, series_values, target_year):
    series_years = np.asarray(series_years, dtype=float)
    series_values = np.asarray(series_values, dtype=float)

    # proyección lineal
    if len(series_values) >= 2:
        coef = np.polyfit(series_years, series_values, 1)
        linear_pred = float(np.polyval(coef, target_year))
    else:
        linear_pred = float(series_values[-1])

    # promedio últimos 3 años
    tail_n = min(3, len(series_values))
    recent_mean = float(np.mean(series_values[-tail_n:]))

    # regla robusta
    if linear_pred > 0:
        final_pred = linear_pred
        method = "linear_trend_2020_2024"
    else:
        final_pred = recent_mean
        method = "fallback_mean_last_3_years"

    return max(final_pred, 0.0), linear_pred, recent_mean, method

# ------------------------------------------------------------
# 5) Proyectar def_I y def_J
# ------------------------------------------------------------
target_year = 2026

pred_I, linear_I, mean3_I, method_I = robust_projection(
    mort["year"].values, mort["def_I"].values, target_year
)

pred_J, linear_J, mean3_J, method_J = robust_projection(
    mort["year"].values, mort["def_J"].values, target_year
)

mort_2026 = pd.DataFrame({
    "year": [target_year, target_year],
    "endpoint_code": ["def_I", "def_J"],
    "count_2026": [round(pred_I), round(pred_J)],
    "linear_projection_raw": [linear_I, linear_J],
    "mean_last_3_years": [mean3_I, mean3_J],
    "projection_method": [method_I, method_J]
})

mort_2026.to_csv(mortality_2026_file, index=False, encoding="utf-8-sig")

print("Proyección robusta 2026 de endpoints:")
print(mort_2026)

# ------------------------------------------------------------
# 6) Calcular población total 2026
# ------------------------------------------------------------
pop_2026 = pop[pop["year"] == target_year].copy()

if pop_2026.empty:
    raise ValueError("No se encontraron registros de población para 2026.")

total_population_2026 = int(pop_2026["total"].sum())

if total_population_2026 <= 0:
    raise ValueError("La población total 2026 es inválida.")

print("\nPoblación total 2026 usada como denominador:", total_population_2026)

# ------------------------------------------------------------
# 7) Construir incidencia provisional uniforme por celda
# ------------------------------------------------------------
cell_ids = sorted(grid["cell_id"].unique())

rows = []
for _, r in mort_2026.iterrows():
    endpoint_code = r["endpoint_code"]
    count_2026 = float(r["count_2026"])
    incidence_rate = count_2026 / total_population_2026

    for cell_id in cell_ids:
        rows.append({
            "cell_id": cell_id,
            "year": target_year,
            "endpoint_code": endpoint_code,
            "age_group": "ALL",
            "incidence_rate": incidence_rate,
            "count_basis_citywide_2026": int(round(count_2026)),
            "population_basis_citywide_2026": total_population_2026,
            "incidence_source": "Bogota_panel_mortalidad_2020_2024",
            "incidence_method": "citywide_uniform_rate_provisional_robust"
        })

incidence = pd.DataFrame(rows)
incidence = incidence.sort_values(["endpoint_code", "cell_id"]).reset_index(drop=True)
incidence.to_csv(incidence_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 8) Resumen final
# ------------------------------------------------------------
summary = incidence.groupby("endpoint_code", as_index=False).agg(
    n_rows=("cell_id", "count"),
    n_cells=("cell_id", "nunique"),
    incidence_rate=("incidence_rate", "first"),
    count_basis_citywide_2026=("count_basis_citywide_2026", "first"),
    population_basis_citywide_2026=("population_basis_citywide_2026", "first")
)

print("\nResumen de incidencia provisional BenMAP:")
print(summary)

print("\nPrimeras filas del archivo final:")
print(incidence.head())

print("\nArchivos actualizados:")
print(" -", mortality_2026_file)
print(" -", incidence_file)

Proyección robusta 2026 de endpoints:
   year endpoint_code  count_2026  linear_projection_raw  mean_last_3_years  \
0  2026         def_I        1034                -5631.6        1033.666667   
1  2026         def_J        2120                 2119.8        1943.666667   

            projection_method  
0  fallback_mean_last_3_years  
1      linear_trend_2020_2024  

Población total 2026 usada como denominador: 16055443

Resumen de incidencia provisional BenMAP:
  endpoint_code  n_rows  n_cells  incidence_rate  count_basis_citywide_2026  \
0         def_I     254      254        0.000064                       1034   
1         def_J     254      254        0.000132                       2120   

   population_basis_citywide_2026  
0                        16055443  
1                        16055443  

Primeras filas del archivo final:
   cell_id  year endpoint_code age_group  incidence_rate  \
0        0  2026         def_I       ALL        0.000064   
1        1  2026         def_

# plantilla funciones C-R

Esta celda construye una plantilla inicial de funciones concentración–respuesta para BenMAP-CE a partir de los campos base que exige el manual del programa. Dado que en esta etapa todavía no se ha cerrado de forma definitiva la selección de coeficientes epidemiológicos para cada endpoint, el objetivo no es producir una base final cerrada, sino dejar organizada una tabla de trabajo que vincule los códigos locales de salud del proyecto con combinaciones plausibles de contaminante, endpoint y grupo etario. El código exporta dos archivos: \texttt{benmap_health_impact_functions_working.csv}, que incluye columnas adicionales para documentar beta, forma funcional y referencias, y \texttt{benmap_health_impact_functions.csv}, que conserva únicamente la estructura mínima tipo BenMAP para su ajuste posterior y validación dentro del software.

In [16]:
# ============================================================
# CELDA 8: Crear plantilla de funciones concentración-respuesta
# para BenMAP-CE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

working_file = output_dir / "benmap_health_impact_functions_working.csv"
benmap_file = output_dir / "benmap_health_impact_functions.csv"

# ------------------------------------------------------------
# 2) Construir plantilla de trabajo
#    NOTA:
#    - Esto NO cierra todavía el beta final.
#    - Sirve para documentar y organizar la selección final.
# ------------------------------------------------------------
rows = [
    {
        "Endpoint Group": "Mortality",
        "Endpoint": "Circulatory mortality",
        "Pollutant": "PM25",
        "Metric": "EDIT_METRIC_PM25",
        "Annual Statistic": "",
        "Seasonal Metric": "",
        "Race": "",
        "Ethnicity": "",
        "Gender": "",
        "Start Age": 0,
        "End Age": 99,
        "Author": "Pope & Dockery",
        "Apply Function To": "Entire Area",
        "Year of Publication": 2006,
        "Qualifier": "EDIT_IF_NEEDED",
        "endpoint_code_local": "def_I",
        "reference_short": "Pope & Dockery (2006)",
        "reference_full": "Pope, C. A., & Dockery, D. W. (2006). Health effects of fine particulate air pollution: Lines that connect.",
        "effect_type": "mortality",
        "beta_per_unit": np.nan,
        "beta_se": np.nan,
        "unit_change": 1.0,
        "concentration_unit": "ug/m3",
        "function_form": "log-linear",
        "status": "PENDING_FINAL_BETA_SELECTION",
        "notes": "Revisar estudio epidemiológico final y rango etario aplicable."
    },
    {
        "Endpoint Group": "Mortality",
        "Endpoint": "Respiratory mortality",
        "Pollutant": "PM25",
        "Metric": "EDIT_METRIC_PM25",
        "Annual Statistic": "",
        "Seasonal Metric": "",
        "Race": "",
        "Ethnicity": "",
        "Gender": "",
        "Start Age": 0,
        "End Age": 99,
        "Author": "Pope & Dockery",
        "Apply Function To": "Entire Area",
        "Year of Publication": 2006,
        "Qualifier": "EDIT_IF_NEEDED",
        "endpoint_code_local": "def_J",
        "reference_short": "Pope & Dockery (2006)",
        "reference_full": "Pope, C. A., & Dockery, D. W. (2006). Health effects of fine particulate air pollution: Lines that connect.",
        "effect_type": "mortality",
        "beta_per_unit": np.nan,
        "beta_se": np.nan,
        "unit_change": 1.0,
        "concentration_unit": "ug/m3",
        "function_form": "log-linear",
        "status": "PENDING_FINAL_BETA_SELECTION",
        "notes": "Revisar si el endpoint final se mantiene como respiratorio o se redefine."
    },
    {
        "Endpoint Group": "Mortality",
        "Endpoint": "Respiratory mortality",
        "Pollutant": "O3",
        "Metric": "EDIT_METRIC_O3",
        "Annual Statistic": "",
        "Seasonal Metric": "",
        "Race": "",
        "Ethnicity": "",
        "Gender": "",
        "Start Age": 0,
        "End Age": 99,
        "Author": "Tagaris et al.",
        "Apply Function To": "Entire Area",
        "Year of Publication": 2010,
        "Qualifier": "EDIT_IF_NEEDED",
        "endpoint_code_local": "def_J",
        "reference_short": "Tagaris et al. (2010)",
        "reference_full": "Tagaris, E., et al. (2010). Sensitivity of air pollution-induced premature mortality to precursor emissions under the influence of climate change.",
        "effect_type": "mortality",
        "beta_per_unit": np.nan,
        "beta_se": np.nan,
        "unit_change": 1.0,
        "concentration_unit": "ppb_or_ugm3_EDIT",
        "function_form": "log-linear",
        "status": "PENDING_FINAL_BETA_SELECTION",
        "notes": "Confirmar métrica de O3 y unidad usada en BenMAP."
    },
    {
        "Endpoint Group": "Mortality",
        "Endpoint": "Circulatory mortality",
        "Pollutant": "NO2",
        "Metric": "EDIT_METRIC_NO2",
        "Annual Statistic": "",
        "Seasonal Metric": "",
        "Race": "",
        "Ethnicity": "",
        "Gender": "",
        "Start Age": 0,
        "End Age": 99,
        "Author": "EDIT_AUTHOR",
        "Apply Function To": "Entire Area",
        "Year of Publication": 2021,
        "Qualifier": "OPTIONAL_CUSTOM_NO2",
        "endpoint_code_local": "def_I",
        "reference_short": "EDIT_REFERENCE",
        "reference_full": "Seleccionar estudio epidemiológico final si se decide incluir NO2 con función custom.",
        "effect_type": "mortality",
        "beta_per_unit": np.nan,
        "beta_se": np.nan,
        "unit_change": 1.0,
        "concentration_unit": "ug/m3",
        "function_form": "log-linear",
        "status": "OPTIONAL_CUSTOM_NO2",
        "notes": "Fila opcional. Mantener solo si se cargará función personalizada para NO2."
    }
]

df_work = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3) Exportar archivo de trabajo completo
# ------------------------------------------------------------
df_work.to_csv(working_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 4) Exportar versión mínima tipo BenMAP
#    (solo campos base que luego ajustarás/validarás)
# ------------------------------------------------------------
benmap_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Pollutant",
    "Metric",
    "Annual Statistic",
    "Seasonal Metric",
    "Race",
    "Ethnicity",
    "Gender",
    "Start Age",
    "End Age",
    "Author",
    "Apply Function To",
    "Year of Publication",
    "Qualifier"
]

df_benmap = df_work[benmap_base_cols].copy()
df_benmap.to_csv(benmap_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 5) Resumen
# ------------------------------------------------------------
print("Plantilla de trabajo generada:")
print(df_work[[
    "endpoint_code_local",
    "Pollutant",
    "Endpoint",
    "Start Age",
    "End Age",
    "Author",
    "Year of Publication",
    "status"
]])

print("\nArchivos generados:")
print(" -", working_file)
print(" -", benmap_file)

print("\nVista previa del archivo tipo BenMAP:")
print(df_benmap.head())

Plantilla de trabajo generada:
  endpoint_code_local Pollutant               Endpoint  Start Age  End Age  \
0               def_I      PM25  Circulatory mortality          0       99   
1               def_J      PM25  Respiratory mortality          0       99   
2               def_J        O3  Respiratory mortality          0       99   
3               def_I       NO2  Circulatory mortality          0       99   

           Author  Year of Publication                        status  
0  Pope & Dockery                 2006  PENDING_FINAL_BETA_SELECTION  
1  Pope & Dockery                 2006  PENDING_FINAL_BETA_SELECTION  
2  Tagaris et al.                 2010  PENDING_FINAL_BETA_SELECTION  
3     EDIT_AUTHOR                 2021           OPTIONAL_CUSTOM_NO2  

Archivos generados:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_working.csv
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions.csv

Vista previa del archivo tipo B

# plantilla inicial de funciones de valoración económica

Esta celda construye una plantilla inicial de funciones de valoración económica para BenMAP-CE, basada en la estructura mínima que exige el manual del programa. Dado que en esta etapa todavía no se ha cerrado de manera definitiva la selección de valores monetarios para cada endpoint, el objetivo no es producir una base final cerrada, sino dejar organizada una tabla de trabajo que vincule los endpoints locales del proyecto con funciones de valoración compatibles con BenMAP. El código exporta dos archivos: \texttt{benmap_valuation_functions_working.csv}, que incluye columnas adicionales para documentar el tipo de valoración, la moneda, el año base y notas metodológicas, y \texttt{benmap_valuation_functions.csv}, que conserva la estructura mínima tipo BenMAP para su ajuste posterior y validación dentro del software.

In [17]:
# ============================================================
# CELDA 9: Crear plantilla de funciones de valoración
# económica para BenMAP-CE
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

working_file = output_dir / "benmap_valuation_functions_working.csv"
benmap_file = output_dir / "benmap_valuation_functions.csv"

# ------------------------------------------------------------
# 2) Construir plantilla de trabajo
#    NOTA:
#    - Point Estimate se deja editable.
#    - Function se deja simple y compatible para ajuste posterior.
# ------------------------------------------------------------
rows = [
    {
        "Endpoint Group": "Mortality",
        "Endpoint": "Circulatory mortality",
        "Qualifier": "VSL_PROVISIONAL",
        "Reference": "World Bank & IHME (2016); ajustar al enfoque final del estudio",
        "Start Age": 0,
        "End Age": 99,
        "Point Estimate": "",
        "Function": "PointEstimate*Incidence",
        "A Description": "",
        "A": "",
        "A Distribution": "",
        "A Parameter 1": "",
        "A Parameter 2": "",
        "Constant Description": "Valor monetario unitario provisional",
        "Constant Value": "",
        "endpoint_code_local": "def_I",
        "valuation_type": "VSL_or_unit_value_EDIT",
        "currency": "COP",
        "price_year": "EDIT_YEAR",
        "status": "PENDING_FINAL_VALUATION_SELECTION",
        "notes": "Definir si se usa VSL, transferencia de beneficios o valoración local."
    },
    {
        "Endpoint Group": "Mortality",
        "Endpoint": "Respiratory mortality",
        "Qualifier": "VSL_PROVISIONAL",
        "Reference": "World Bank & IHME (2016); ajustar al enfoque final del estudio",
        "Start Age": 0,
        "End Age": 99,
        "Point Estimate": "",
        "Function": "PointEstimate*Incidence",
        "A Description": "",
        "A": "",
        "A Distribution": "",
        "A Parameter 1": "",
        "A Parameter 2": "",
        "Constant Description": "Valor monetario unitario provisional",
        "Constant Value": "",
        "endpoint_code_local": "def_J",
        "valuation_type": "VSL_or_unit_value_EDIT",
        "currency": "COP",
        "price_year": "EDIT_YEAR",
        "status": "PENDING_FINAL_VALUATION_SELECTION",
        "notes": "Mantener coherencia con el mismo esquema de valoración del endpoint de mortalidad."
    }
]

df_work = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3) Exportar archivo de trabajo completo
# ------------------------------------------------------------
df_work.to_csv(working_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 4) Exportar versión mínima tipo BenMAP
# ------------------------------------------------------------
benmap_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Qualifier",
    "Reference",
    "Start Age",
    "End Age",
    "Point Estimate",
    "Function",
    "A Description",
    "A",
    "A Distribution",
    "A Parameter 1",
    "A Parameter 2",
    "Constant Description",
    "Constant Value"
]

df_benmap = df_work[benmap_base_cols].copy()
df_benmap.to_csv(benmap_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 5) Resumen
# ------------------------------------------------------------
print("Plantilla de trabajo de valoración generada:")
print(df_work[[
    "endpoint_code_local",
    "Endpoint Group",
    "Endpoint",
    "valuation_type",
    "currency",
    "price_year",
    "status"
]])

print("\nArchivos generados:")
print(" -", working_file)
print(" -", benmap_file)

print("\nVista previa del archivo tipo BenMAP:")
print(df_benmap.head())

Plantilla de trabajo de valoración generada:
  endpoint_code_local Endpoint Group               Endpoint  \
0               def_I      Mortality  Circulatory mortality   
1               def_J      Mortality  Respiratory mortality   

           valuation_type currency price_year  \
0  VSL_or_unit_value_EDIT      COP  EDIT_YEAR   
1  VSL_or_unit_value_EDIT      COP  EDIT_YEAR   

                              status  
0  PENDING_FINAL_VALUATION_SELECTION  
1  PENDING_FINAL_VALUATION_SELECTION  

Archivos generados:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_working.csv
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions.csv

Vista previa del archivo tipo BenMAP:
  Endpoint Group               Endpoint        Qualifier  \
0      Mortality  Circulatory mortality  VSL_PROVISIONAL   
1      Mortality  Respiratory mortality  VSL_PROVISIONAL   

                                           Reference  Start Age  End Age  \
0  World Bank & IHME 

# celda de verificación final

Esta celda realiza una verificación integral de los archivos de entrada que se han ido construyendo para BenMAP-CE. El objetivo es confirmar que los insumos principales del flujo ya existen en la carpeta de salidas, revisar sus dimensiones y columnas básicas, y detectar de forma automática campos pendientes de completar, como marcadores tipo \texttt{EDIT_}, \texttt{PENDING_} o valores vacíos en las plantillas de funciones concentración–respuesta y valoración económica. De esta manera, la celda sirve como control final de consistencia antes de pasar a la fase de ajuste definitivo de métricas, coeficientes epidemiológicos y valores monetarios dentro del proyecto.

In [18]:
# ============================================================
# CELDA 10: Verificación final de insumos BenMAP-CE
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

files_to_check = {
    "pm25_baseline": output_dir / "benmap_PM25_baseline.csv",
    "pm25_control": output_dir / "benmap_PM25_control.csv",
    "grid_summary": output_dir / "benmap_grid_definition_summary.csv",
    "population_2026": output_dir / "benmap_population_2026.csv",
    "incidence_2026": output_dir / "benmap_incidence_2026.csv",
    "health_working": output_dir / "benmap_health_impact_functions_working.csv",
    "health_benmap": output_dir / "benmap_health_impact_functions.csv",
    "valuation_working": output_dir / "benmap_valuation_functions_working.csv",
    "valuation_benmap": output_dir / "benmap_valuation_functions.csv",
    "mortality_clean": output_dir / "mortality_panel_endpoints_clean.csv",
    "mortality_2026": output_dir / "mortality_panel_endpoints_2026.csv",
}

# ------------------------------------------------------------
# 2) Verificar existencia y estructura básica
# ------------------------------------------------------------
summary_rows = []

for name, path_file in files_to_check.items():
    exists = path_file.exists()

    info = {
        "artifact_name": name,
        "file_name": path_file.name,
        "exists": exists,
        "n_rows": None,
        "n_cols": None,
        "columns_preview": None,
        "notes": ""
    }

    if exists:
        try:
            df_tmp = pd.read_csv(path_file)
            info["n_rows"] = len(df_tmp)
            info["n_cols"] = len(df_tmp.columns)
            info["columns_preview"] = " | ".join(df_tmp.columns[:8].tolist())
        except Exception as e:
            info["notes"] = f"No se pudo leer como CSV: {e}"

    summary_rows.append(info)

summary = pd.DataFrame(summary_rows)

print("Resumen general de archivos generados:")
print(summary)

# ------------------------------------------------------------
# 3) Revisar placeholders en health impact functions
# ------------------------------------------------------------
placeholder_tokens = ["EDIT_", "PENDING_", "OPTIONAL_", ""]

def detect_placeholders(df):
    placeholder_report = []

    for col in df.columns:
        series = df[col].astype(str).fillna("").str.strip()

        has_edit = series.str.contains("EDIT_", na=False).sum()
        has_pending = series.str.contains("PENDING_", na=False).sum()
        has_optional = series.str.contains("OPTIONAL_", na=False).sum()
        has_empty = (series == "").sum()

        if has_edit > 0 or has_pending > 0 or has_optional > 0 or has_empty > 0:
            placeholder_report.append({
                "column": col,
                "n_EDIT": int(has_edit),
                "n_PENDING": int(has_pending),
                "n_OPTIONAL": int(has_optional),
                "n_empty": int(has_empty)
            })

    return pd.DataFrame(placeholder_report)

# ------------------------------------------------------------
# 4) Revisar archivo de funciones C-R
# ------------------------------------------------------------
health_working_file = files_to_check["health_working"]
if health_working_file.exists():
    df_health = pd.read_csv(health_working_file)
    health_placeholders = detect_placeholders(df_health)

    print("\nRevisión de campos pendientes en benmap_health_impact_functions_working.csv:")
    if health_placeholders.empty:
        print("No se detectaron placeholders ni vacíos relevantes.")
    else:
        print(health_placeholders)

# ------------------------------------------------------------
# 5) Revisar archivo de valoración económica
# ------------------------------------------------------------
valuation_working_file = files_to_check["valuation_working"]
if valuation_working_file.exists():
    df_val = pd.read_csv(valuation_working_file)
    valuation_placeholders = detect_placeholders(df_val)

    print("\nRevisión de campos pendientes en benmap_valuation_functions_working.csv:")
    if valuation_placeholders.empty:
        print("No se detectaron placeholders ni vacíos relevantes.")
    else:
        print(valuation_placeholders)

# ------------------------------------------------------------
# 6) Diagnóstico específico útil
# ------------------------------------------------------------
print("\nDiagnóstico específico:")

if files_to_check["pm25_baseline"].exists():
    df_pm25_base = pd.read_csv(files_to_check["pm25_baseline"])
    print(f"- PM2.5 baseline: {df_pm25_base['cell_id'].nunique()} celdas, {len(df_pm25_base)} filas")

if files_to_check["population_2026"].exists():
    df_pop = pd.read_csv(files_to_check["population_2026"])
    print(f"- Población 2026: {df_pop['cell_id'].nunique()} celdas, {df_pop['age_group'].nunique()} grupos etarios, total={int(df_pop['total'].sum())}")

if files_to_check["incidence_2026"].exists():
    df_inc = pd.read_csv(files_to_check["incidence_2026"])
    print(f"- Incidencia 2026: endpoints={df_inc['endpoint_code'].nunique()}, filas={len(df_inc)}")

if files_to_check["mortality_2026"].exists():
    df_m2026 = pd.read_csv(files_to_check["mortality_2026"])
    print("- Endpoints proyectados 2026:")
    print(df_m2026)

# ------------------------------------------------------------
# 7) Exportar resumen final del chequeo
# ------------------------------------------------------------
final_check_file = output_dir / "benmap_artifacts_checklist.csv"
summary.to_csv(final_check_file, index=False, encoding="utf-8-sig")

print("\nArchivo de verificación generado:")
print(" -", final_check_file)

Resumen general de archivos generados:
        artifact_name                                   file_name  exists  \
0       pm25_baseline                    benmap_PM25_baseline.csv    True   
1        pm25_control                     benmap_PM25_control.csv    True   
2        grid_summary          benmap_grid_definition_summary.csv    True   
3     population_2026                  benmap_population_2026.csv    True   
4      incidence_2026                   benmap_incidence_2026.csv    True   
5      health_working  benmap_health_impact_functions_working.csv    True   
6       health_benmap          benmap_health_impact_functions.csv    True   
7   valuation_working      benmap_valuation_functions_working.csv    True   
8    valuation_benmap              benmap_valuation_functions.csv    True   
9     mortality_clean         mortality_panel_endpoints_clean.csv    True   
10     mortality_2026          mortality_panel_endpoints_2026.csv    True   

    n_rows  n_cols                  



# actualizacion de las plantillas de funciones concentración–respuesta y de valoración económica

Esta celda actualiza automáticamente las plantillas de funciones concentración–respuesta y de valoración económica para dejar cerrados los campos de texto que todavía tenían marcadores provisionales. El código asigna métricas por contaminante, completa autores y referencias cortas cuando están vacíos, reemplaza etiquetas de estado pendientes por estados operativos y fija un año base de valoración editable. Después, vuelve a exportar tanto los archivos de trabajo como las versiones simplificadas tipo BenMAP. El objetivo no es cerrar todavía los coeficientes epidemiológicos ni los valores monetarios finales, sino dejar las plantillas sin marcadores \texttt{EDIT_}, \texttt{PENDING_} u otros textos transitorios en los campos descriptivos, de modo que el paquete quede más limpio y listo para la fase final de ajuste.

In [19]:
# ============================================================
# CELDA 11: Limpiar placeholders y cerrar campos de texto
# en funciones C-R y valoración económica
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

health_working_file = output_dir / "benmap_health_impact_functions_working.csv"
health_benmap_file = output_dir / "benmap_health_impact_functions.csv"

valuation_working_file = output_dir / "benmap_valuation_functions_working.csv"
valuation_benmap_file = output_dir / "benmap_valuation_functions.csv"

for f in [health_working_file, valuation_working_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

# ------------------------------------------------------------
# 2) Parámetros editables
#    OJO: estas métricas son provisionales y deben coincidir
#    con los nombres que finalmente cargues/configures en BenMAP.
# ------------------------------------------------------------
metric_map = {
    "PM25": "PM25 Annual Mean",
    "O3": "O3 Seasonal Mean",
    "NO2": "NO2 Annual Mean"
}

author_map = {
    "def_I_PM25": "Pope & Dockery",
    "def_J_PM25": "Pope & Dockery",
    "def_J_O3": "Tagaris et al.",
    "def_I_NO2": "Custom NO2 Function"
}

reference_short_map = {
    "def_I_PM25": "Pope & Dockery (2006)",
    "def_J_PM25": "Pope & Dockery (2006)",
    "def_J_O3": "Tagaris et al. (2010)",
    "def_I_NO2": "Custom NO2 reference"
}

qualifier_health_default = "Provisional epidemiologic function"
qualifier_valuation_default = "Provisional valuation function"
valuation_price_year = 2024

# ------------------------------------------------------------
# 3) Cargar funciones C-R de trabajo
# ------------------------------------------------------------
df_health = pd.read_csv(health_working_file)

# Asegurar columnas esperadas
expected_health_cols = [
    "Endpoint Group", "Endpoint", "Pollutant", "Metric", "Author",
    "Qualifier", "endpoint_code_local", "reference_short", "status"
]
missing_health = [c for c in expected_health_cols if c not in df_health.columns]
if missing_health:
    raise ValueError(f"Faltan columnas en health working: {missing_health}")

# ------------------------------------------------------------
# 4) Completar campos de texto de funciones C-R
# ------------------------------------------------------------
def build_health_key(row):
    return f"{row['endpoint_code_local']}_{row['Pollutant']}"

health_keys = df_health.apply(build_health_key, axis=1)

# Metric
df_health["Metric"] = df_health["Pollutant"].map(metric_map).fillna(df_health["Metric"])

# Author
df_health["Author"] = [
    author_map.get(k, a) if (pd.isna(a) or str(a).strip() == "" or "EDIT" in str(a)) else a
    for k, a in zip(health_keys, df_health["Author"])
]

# reference_short
df_health["reference_short"] = [
    reference_short_map.get(k, r) if (pd.isna(r) or str(r).strip() == "" or "EDIT" in str(r)) else r
    for k, r in zip(health_keys, df_health["reference_short"])
]

# Qualifier
df_health["Qualifier"] = [
    qualifier_health_default if (
        pd.isna(q) or str(q).strip() == "" or "EDIT" in str(q) or "OPTIONAL" in str(q)
    ) else q
    for q in df_health["Qualifier"]
]

# status
def clean_health_status(x):
    x = "" if pd.isna(x) else str(x).strip()
    if x == "" or "PENDING" in x:
        return "PROVISIONAL_READY_FOR_BETA"
    if "OPTIONAL" in x:
        return "NO2_CUSTOM_REVIEW"
    return x

df_health["status"] = df_health["status"].apply(clean_health_status)

# ------------------------------------------------------------
# 5) Exportar funciones C-R actualizadas
# ------------------------------------------------------------
df_health.to_csv(health_working_file, index=False, encoding="utf-8-sig")

health_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Pollutant",
    "Metric",
    "Annual Statistic",
    "Seasonal Metric",
    "Race",
    "Ethnicity",
    "Gender",
    "Start Age",
    "End Age",
    "Author",
    "Apply Function To",
    "Year of Publication",
    "Qualifier"
]

missing_base_health = [c for c in health_base_cols if c not in df_health.columns]
if missing_base_health:
    raise ValueError(f"Faltan columnas para exportar versión BenMAP de health: {missing_base_health}")

df_health_benmap = df_health[health_base_cols].copy()
df_health_benmap.to_csv(health_benmap_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 6) Cargar funciones de valoración de trabajo
# ------------------------------------------------------------
df_val = pd.read_csv(valuation_working_file)

expected_val_cols = [
    "Endpoint Group", "Endpoint", "Qualifier", "Reference",
    "price_year", "status"
]
missing_val = [c for c in expected_val_cols if c not in df_val.columns]
if missing_val:
    raise ValueError(f"Faltan columnas en valuation working: {missing_val}")

# ------------------------------------------------------------
# 7) Completar campos de texto de valoración
# ------------------------------------------------------------
df_val["Qualifier"] = [
    qualifier_valuation_default if (
        pd.isna(q) or str(q).strip() == "" or "EDIT" in str(q) or "PENDING" in str(q)
    ) else q
    for q in df_val["Qualifier"]
]

df_val["price_year"] = [
    valuation_price_year if (pd.isna(v) or str(v).strip() == "" or "EDIT" in str(v)) else v
    for v in df_val["price_year"]
]

def clean_valuation_status(x):
    x = "" if pd.isna(x) else str(x).strip()
    if x == "" or "PENDING" in x:
        return "PROVISIONAL_READY_FOR_VALUE"
    return x

df_val["status"] = df_val["status"].apply(clean_valuation_status)

# ------------------------------------------------------------
# 8) Exportar funciones de valoración actualizadas
# ------------------------------------------------------------
df_val.to_csv(valuation_working_file, index=False, encoding="utf-8-sig")

valuation_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Qualifier",
    "Reference",
    "Start Age",
    "End Age",
    "Point Estimate",
    "Function",
    "A Description",
    "A",
    "A Distribution",
    "A Parameter 1",
    "A Parameter 2",
    "Constant Description",
    "Constant Value"
]

missing_base_val = [c for c in valuation_base_cols if c not in df_val.columns]
if missing_base_val:
    raise ValueError(f"Faltan columnas para exportar versión BenMAP de valuation: {missing_base_val}")

df_val_benmap = df_val[valuation_base_cols].copy()
df_val_benmap.to_csv(valuation_benmap_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Resumen final
# ------------------------------------------------------------
print("Funciones C-R actualizadas:")
print(df_health[[
    "endpoint_code_local", "Pollutant", "Metric", "Author",
    "Qualifier", "reference_short", "status"
]])

print("\nFunciones de valoración actualizadas:")
print(df_val[[
    "endpoint_code_local", "Endpoint", "Qualifier",
    "price_year", "status"
]])

print("\nArchivos actualizados:")
print(" -", health_working_file)
print(" -", health_benmap_file)
print(" -", valuation_working_file)
print(" -", valuation_benmap_file)

print("\nVista previa health tipo BenMAP:")
print(df_health_benmap.head())

print("\nVista previa valuation tipo BenMAP:")
print(df_val_benmap.head())

Funciones C-R actualizadas:
  endpoint_code_local Pollutant            Metric               Author  \
0               def_I      PM25  PM25 Annual Mean       Pope & Dockery   
1               def_J      PM25  PM25 Annual Mean       Pope & Dockery   
2               def_J        O3  O3 Seasonal Mean       Tagaris et al.   
3               def_I       NO2   NO2 Annual Mean  Custom NO2 Function   

                            Qualifier        reference_short  \
0  Provisional epidemiologic function  Pope & Dockery (2006)   
1  Provisional epidemiologic function  Pope & Dockery (2006)   
2  Provisional epidemiologic function  Tagaris et al. (2010)   
3  Provisional epidemiologic function   Custom NO2 reference   

                       status  
0  PROVISIONAL_READY_FOR_BETA  
1  PROVISIONAL_READY_FOR_BETA  
2  PROVISIONAL_READY_FOR_BETA  
3           NO2_CUSTOM_REVIEW  

Funciones de valoración actualizadas:
  endpoint_code_local               Endpoint        Qualifier  price_year  \
0   

# reporte exacto de pendientes
Esta celda revisa las plantillas de funciones concentración–respuesta y de valoración económica para identificar los campos que todavía faltan por completar antes de una importación final en BenMAP-CE. Además, permite excluir temporalmente la función personalizada de NO$2$ si se desea trabajar primero solo con PM${2.5}$ y O$_3$. El código genera un reporte consolidado de pendientes, separando los faltantes de texto y de valores numéricos, y exporta una versión \texttt{candidate_final} de los archivos de trabajo y de sus equivalentes tipo BenMAP. De esta manera, el flujo queda organizado en dos niveles: un conjunto de archivos de trabajo que siguen siendo editables y un conjunto de archivos candidatos para importación final, útiles para continuar el cierre metodológico del paquete.

In [21]:
# ============================================================
# CELDA 12: Generar reporte de pendientes y archivos
# candidate final para BenMAP-CE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Parámetros de control
# ------------------------------------------------------------
include_no2_custom = True   # False = excluir fila custom NO2 por ahora
export_candidate_final = True

# ------------------------------------------------------------
# 2) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

health_working_file = output_dir / "benmap_health_impact_functions_working.csv"
valuation_working_file = output_dir / "benmap_valuation_functions_working.csv"

for f in [health_working_file, valuation_working_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

health_candidate_working_file = output_dir / "benmap_health_impact_functions_candidate_final_working.csv"
health_candidate_benmap_file = output_dir / "benmap_health_impact_functions_candidate_final.csv"

valuation_candidate_working_file = output_dir / "benmap_valuation_functions_candidate_final_working.csv"
valuation_candidate_benmap_file = output_dir / "benmap_valuation_functions_candidate_final.csv"

pending_report_file = output_dir / "benmap_pending_inputs_report.csv"

# ------------------------------------------------------------
# 3) Cargar archivos de trabajo
# ------------------------------------------------------------
df_health = pd.read_csv(health_working_file)
df_val = pd.read_csv(valuation_working_file)

# ------------------------------------------------------------
# 4) Filtrar NO2 opcional si así se define
# ------------------------------------------------------------
df_health_candidate = df_health.copy()

if not include_no2_custom:
    if "Pollutant" in df_health_candidate.columns:
        df_health_candidate = df_health_candidate[df_health_candidate["Pollutant"] != "NO2"].copy()
    if "status" in df_health_candidate.columns:
        df_health_candidate = df_health_candidate[df_health_candidate["status"] != "NO2_CUSTOM_REVIEW"].copy()

df_health_candidate = df_health_candidate.reset_index(drop=True)

df_val_candidate = df_val.copy().reset_index(drop=True)

# ------------------------------------------------------------
# 5) Funciones auxiliares para detectar pendientes
# ------------------------------------------------------------
def is_empty_like(x):
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() == "nan"

def has_placeholder(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    tokens = ["EDIT_", "PENDING_", "OPTIONAL_", "REVIEW", "PROVISIONAL"]
    return any(tok in s for tok in tokens)

pending_rows = []

def append_pending(df, artifact_name, row_id_cols, required_text_cols=None, required_numeric_cols=None):
    required_text_cols = required_text_cols or []
    required_numeric_cols = required_numeric_cols or []

    for idx, row in df.iterrows():
        row_id = " | ".join([f"{c}={row[c]}" for c in row_id_cols if c in df.columns])

        # texto requerido
        for col in required_text_cols:
            if col in df.columns:
                value = row[col]
                if is_empty_like(value) or has_placeholder(value):
                    pending_rows.append({
                        "artifact": artifact_name,
                        "row_identifier": row_id,
                        "field_name": col,
                        "field_type": "text",
                        "current_value": value,
                        "issue": "missing_or_placeholder"
                    })

        # numérico requerido
        for col in required_numeric_cols:
            if col in df.columns:
                value = row[col]
                if pd.isna(pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]):
                    pending_rows.append({
                        "artifact": artifact_name,
                        "row_identifier": row_id,
                        "field_name": col,
                        "field_type": "numeric",
                        "current_value": value,
                        "issue": "missing_numeric_value"
                    })

# ------------------------------------------------------------
# 6) Detectar pendientes en funciones C-R
# ------------------------------------------------------------
health_row_id = ["endpoint_code_local", "Pollutant", "Endpoint"]

health_required_text = [
    "Metric", "Author", "Qualifier", "reference_short", "status"
]

health_required_numeric = [
    "beta_per_unit", "unit_change"
]

# beta_se puede quedar vacío si luego decides no usarlo directamente,
# pero lo reportamos aparte como informativo
append_pending(
    df_health_candidate,
    artifact_name="health_impact_functions",
    row_id_cols=health_row_id,
    required_text_cols=health_required_text,
    required_numeric_cols=health_required_numeric
)

# Reporte adicional opcional de beta_se
if "beta_se" in df_health_candidate.columns:
    for _, row in df_health_candidate.iterrows():
        row_id = " | ".join([f"{c}={row[c]}" for c in health_row_id if c in df_health_candidate.columns])
        value = row["beta_se"]
        if pd.isna(pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]):
            pending_rows.append({
                "artifact": "health_impact_functions",
                "row_identifier": row_id,
                "field_name": "beta_se",
                "field_type": "numeric_optional_review",
                "current_value": value,
                "issue": "missing_optional_review"
            })

# ------------------------------------------------------------
# 7) Detectar pendientes en funciones de valoración
# ------------------------------------------------------------
valuation_row_id = ["endpoint_code_local", "Endpoint"]

valuation_required_text = [
    "Qualifier", "Reference", "price_year", "status"
]

# Como la función está definida como PointEstimate*Incidence,
# Point Estimate sí debe cerrarse.
valuation_required_numeric = [
    "Point Estimate"
]

append_pending(
    df_val_candidate,
    artifact_name="valuation_functions",
    row_id_cols=valuation_row_id,
    required_text_cols=valuation_required_text,
    required_numeric_cols=valuation_required_numeric
)

# ------------------------------------------------------------
# 8) Construir reporte de pendientes
# ------------------------------------------------------------
pending_report = pd.DataFrame(pending_rows)

if pending_report.empty:
    print("No se detectaron pendientes en los archivos candidate final.")
else:
    pending_report = pending_report.sort_values(
        by=["artifact", "row_identifier", "field_type", "field_name"]
    ).reset_index(drop=True)

pending_report.to_csv(pending_report_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Exportar candidate final working + tipo BenMAP
# ------------------------------------------------------------
if export_candidate_final:
    # Guardar working filtrados
    df_health_candidate.to_csv(health_candidate_working_file, index=False, encoding="utf-8-sig")
    df_val_candidate.to_csv(valuation_candidate_working_file, index=False, encoding="utf-8-sig")

    # Exportar tipo BenMAP
    health_base_cols = [
        "Endpoint Group",
        "Endpoint",
        "Pollutant",
        "Metric",
        "Annual Statistic",
        "Seasonal Metric",
        "Race",
        "Ethnicity",
        "Gender",
        "Start Age",
        "End Age",
        "Author",
        "Apply Function To",
        "Year of Publication",
        "Qualifier"
    ]

    valuation_base_cols = [
        "Endpoint Group",
        "Endpoint",
        "Qualifier",
        "Reference",
        "Start Age",
        "End Age",
        "Point Estimate",
        "Function",
        "A Description",
        "A",
        "A Distribution",
        "A Parameter 1",
        "A Parameter 2",
        "Constant Description",
        "Constant Value"
    ]

    missing_health_base = [c for c in health_base_cols if c not in df_health_candidate.columns]
    missing_valuation_base = [c for c in valuation_base_cols if c not in df_val_candidate.columns]

    if missing_health_base:
        raise ValueError(f"Faltan columnas para exportar health candidate final: {missing_health_base}")
    if missing_valuation_base:
        raise ValueError(f"Faltan columnas para exportar valuation candidate final: {missing_valuation_base}")

    df_health_candidate_benmap = df_health_candidate[health_base_cols].copy()
    df_val_candidate_benmap = df_val_candidate[valuation_base_cols].copy()

    df_health_candidate_benmap.to_csv(health_candidate_benmap_file, index=False, encoding="utf-8-sig")
    df_val_candidate_benmap.to_csv(valuation_candidate_benmap_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 10) Mostrar resultados
# ------------------------------------------------------------
print("Configuración usada:")
print(f"- include_no2_custom = {include_no2_custom}")
print(f"- export_candidate_final = {export_candidate_final}")

print("\nResumen candidate final - Health:")
print(df_health_candidate[[
    c for c in ["endpoint_code_local", "Pollutant", "Endpoint", "Metric", "status"]
    if c in df_health_candidate.columns
]])

print("\nResumen candidate final - Valuation:")
print(df_val_candidate[[
    c for c in ["endpoint_code_local", "Endpoint", "Qualifier", "price_year", "status"]
    if c in df_val_candidate.columns
]])

if pending_report.empty:
    print("\nNo se detectaron pendientes.")
else:
    print("\nReporte de pendientes detectados:")
    print(pending_report)

print("\nArchivos generados/actualizados:")
if export_candidate_final:
    print(" -", health_candidate_working_file)
    print(" -", health_candidate_benmap_file)
    print(" -", valuation_candidate_working_file)
    print(" -", valuation_candidate_benmap_file)

if not pending_report.empty:
    print(" -", pending_report_file)

Configuración usada:
- include_no2_custom = True
- export_candidate_final = True

Resumen candidate final - Health:
  endpoint_code_local Pollutant               Endpoint            Metric  \
0               def_I      PM25  Circulatory mortality  PM25 Annual Mean   
1               def_J      PM25  Respiratory mortality  PM25 Annual Mean   
2               def_J        O3  Respiratory mortality  O3 Seasonal Mean   
3               def_I       NO2  Circulatory mortality   NO2 Annual Mean   

                       status  
0  PROVISIONAL_READY_FOR_BETA  
1  PROVISIONAL_READY_FOR_BETA  
2  PROVISIONAL_READY_FOR_BETA  
3           NO2_CUSTOM_REVIEW  

Resumen candidate final - Valuation:
  endpoint_code_local               Endpoint        Qualifier  price_year  \
0               def_I  Circulatory mortality  VSL_PROVISIONAL        2024   
1               def_J  Respiratory mortality  VSL_PROVISIONAL        2024   

                        status  
0  PROVISIONAL_READY_FOR_VALUE  
1  PROV

# completa campos numéricos pendientes en las funciones concentración–respuesta y en las funciones de valoración económica

Esta celda completa los campos numéricos pendientes en las funciones concentración–respuesta y en las funciones de valoración económica. En lugar de fijar automáticamente coeficientes epidemiológicos o valores monetarios sin validación previa, el código define una tabla editable donde se pueden ingresar manualmente los riesgos relativos (\texttt{RR}), la magnitud del cambio en concentración (\texttt{unit_change}), el error estándar asociado y el valor monetario unitario para cada endpoint. Cuando se proporciona un \texttt{RR}, la celda calcula automáticamente el coeficiente \texttt{beta_per_unit} usando la relación log-lineal $\beta = \ln(RR)/\Delta C$. Si ya se dispone directamente del beta, también puede ingresarse sin usar RR. Finalmente, la celda actualiza los archivos de trabajo y vuelve a exportar las versiones \texttt{candidate_final} de BenMAP-CE.

In [22]:
# ============================================================
# CELDA 13: Completar betas y valoración monetaria
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

health_working_file = output_dir / "benmap_health_impact_functions_working.csv"
valuation_working_file = output_dir / "benmap_valuation_functions_working.csv"

health_candidate_working_file = output_dir / "benmap_health_impact_functions_candidate_final_working.csv"
health_candidate_benmap_file = output_dir / "benmap_health_impact_functions_candidate_final.csv"

valuation_candidate_working_file = output_dir / "benmap_valuation_functions_candidate_final_working.csv"
valuation_candidate_benmap_file = output_dir / "benmap_valuation_functions_candidate_final.csv"

for f in [health_working_file, valuation_working_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

# ------------------------------------------------------------
# 2) Cargar archivos de trabajo
# ------------------------------------------------------------
df_health = pd.read_csv(health_working_file)
df_val = pd.read_csv(valuation_working_file)

# ------------------------------------------------------------
# 3) TABLA EDITABLE DE PARÁMETROS
#    Rellena estos valores con tus decisiones finales.
#
#    Opción A: poner rr_value y unit_change -> calcula beta
#    Opción B: dejar rr_value vacío y poner beta_direct directamente
# ------------------------------------------------------------
health_params = pd.DataFrame([
    {
        "endpoint_code_local": "def_I",
        "Pollutant": "PM25",
        "rr_value": np.nan,         # <- EDITAR
        "unit_change": 10.0,        # <- EDITAR si tu RR está por otra unidad
        "beta_se": np.nan,          # <- EDITAR si lo tienes
        "beta_direct": np.nan       # <- usar solo si ya tienes beta final
    },
    {
        "endpoint_code_local": "def_J",
        "Pollutant": "PM25",
        "rr_value": np.nan,         # <- EDITAR
        "unit_change": 10.0,        # <- EDITAR
        "beta_se": np.nan,          # <- EDITAR
        "beta_direct": np.nan
    },
    {
        "endpoint_code_local": "def_J",
        "Pollutant": "O3",
        "rr_value": np.nan,         # <- EDITAR
        "unit_change": 10.0,        # <- EDITAR
        "beta_se": np.nan,          # <- EDITAR
        "beta_direct": np.nan
    },
    {
        "endpoint_code_local": "def_I",
        "Pollutant": "NO2",
        "rr_value": np.nan,         # <- EDITAR
        "unit_change": 10.0,        # <- EDITAR
        "beta_se": np.nan,          # <- EDITAR
        "beta_direct": np.nan
    }
])

valuation_params = pd.DataFrame([
    {
        "endpoint_code_local": "def_I",
        "Point Estimate": np.nan,   # <- EDITAR valor monetario final
        "Constant Value": np.nan    # <- opcional, si tu función lo requiere
    },
    {
        "endpoint_code_local": "def_J",
        "Point Estimate": np.nan,   # <- EDITAR valor monetario final
        "Constant Value": np.nan
    }
])

# ------------------------------------------------------------
# 4) Función para calcular beta desde RR
#    beta = ln(RR) / unit_change
# ------------------------------------------------------------
def compute_beta(rr_value, unit_change, beta_direct):
    if pd.notna(beta_direct):
        return float(beta_direct)
    if pd.notna(rr_value) and pd.notna(unit_change):
        if rr_value > 0 and unit_change != 0:
            return float(np.log(rr_value) / unit_change)
    return np.nan

# ------------------------------------------------------------
# 5) Actualizar funciones C-R
# ------------------------------------------------------------
for idx, row in health_params.iterrows():
    mask = (
        (df_health["endpoint_code_local"] == row["endpoint_code_local"]) &
        (df_health["Pollutant"] == row["Pollutant"])
    )

    if mask.sum() == 0:
        print(f"No se encontró fila en health para {row['endpoint_code_local']} - {row['Pollutant']}")
        continue

    beta_value = compute_beta(
        rr_value=row["rr_value"],
        unit_change=row["unit_change"],
        beta_direct=row["beta_direct"]
    )

    df_health.loc[mask, "unit_change"] = row["unit_change"]
    df_health.loc[mask, "beta_per_unit"] = beta_value
    df_health.loc[mask, "beta_se"] = row["beta_se"]

# Recalcular status
def health_status(row):
    if pd.isna(row.get("beta_per_unit", np.nan)):
        if row.get("Pollutant") == "NO2":
            return "NO2_CUSTOM_REVIEW"
        return "PROVISIONAL_READY_FOR_BETA"
    return "READY_NUMERIC"

df_health["status"] = df_health.apply(health_status, axis=1)

# ------------------------------------------------------------
# 6) Actualizar funciones de valoración
# ------------------------------------------------------------
for idx, row in valuation_params.iterrows():
    mask = df_val["endpoint_code_local"] == row["endpoint_code_local"]

    if mask.sum() == 0:
        print(f"No se encontró fila en valuation para {row['endpoint_code_local']}")
        continue

    df_val.loc[mask, "Point Estimate"] = row["Point Estimate"]
    df_val.loc[mask, "Constant Value"] = row["Constant Value"]

def valuation_status(row):
    if pd.isna(pd.to_numeric(pd.Series([row.get("Point Estimate", np.nan)]), errors="coerce").iloc[0]):
        return "PROVISIONAL_READY_FOR_VALUE"
    return "READY_NUMERIC"

df_val["status"] = df_val.apply(valuation_status, axis=1)

# ------------------------------------------------------------
# 7) Guardar archivos de trabajo actualizados
# ------------------------------------------------------------
df_health.to_csv(health_working_file, index=False, encoding="utf-8-sig")
df_val.to_csv(valuation_working_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 8) Exportar candidate final actualizados
# ------------------------------------------------------------
df_health_candidate = df_health.copy().reset_index(drop=True)
df_val_candidate = df_val.copy().reset_index(drop=True)

df_health_candidate.to_csv(health_candidate_working_file, index=False, encoding="utf-8-sig")
df_val_candidate.to_csv(valuation_candidate_working_file, index=False, encoding="utf-8-sig")

health_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Pollutant",
    "Metric",
    "Annual Statistic",
    "Seasonal Metric",
    "Race",
    "Ethnicity",
    "Gender",
    "Start Age",
    "End Age",
    "Author",
    "Apply Function To",
    "Year of Publication",
    "Qualifier"
]

valuation_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Qualifier",
    "Reference",
    "Start Age",
    "End Age",
    "Point Estimate",
    "Function",
    "A Description",
    "A",
    "A Distribution",
    "A Parameter 1",
    "A Parameter 2",
    "Constant Description",
    "Constant Value"
]

df_health_candidate[health_base_cols].to_csv(health_candidate_benmap_file, index=False, encoding="utf-8-sig")
df_val_candidate[valuation_base_cols].to_csv(valuation_candidate_benmap_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Resumen
# ------------------------------------------------------------
print("Resumen actualizado de funciones C-R:")
print(df_health[[
    "endpoint_code_local", "Pollutant", "beta_per_unit",
    "beta_se", "unit_change", "status"
]])

print("\nResumen actualizado de valoración:")
print(df_val[[
    "endpoint_code_local", "Point Estimate", "Constant Value", "status"
]])

print("\nArchivos actualizados:")
print(" -", health_working_file)
print(" -", valuation_working_file)
print(" -", health_candidate_working_file)
print(" -", health_candidate_benmap_file)
print(" -", valuation_candidate_working_file)
print(" -", valuation_candidate_benmap_file)

Resumen actualizado de funciones C-R:
  endpoint_code_local Pollutant  beta_per_unit  beta_se  unit_change  \
0               def_I      PM25            NaN      NaN         10.0   
1               def_J      PM25            NaN      NaN         10.0   
2               def_J        O3            NaN      NaN         10.0   
3               def_I       NO2            NaN      NaN         10.0   

                       status  
0  PROVISIONAL_READY_FOR_BETA  
1  PROVISIONAL_READY_FOR_BETA  
2  PROVISIONAL_READY_FOR_BETA  
3           NO2_CUSTOM_REVIEW  

Resumen actualizado de valoración:
  endpoint_code_local  Point Estimate  Constant Value  \
0               def_I             NaN             NaN   
1               def_J             NaN             NaN   

                        status  
0  PROVISIONAL_READY_FOR_VALUE  
1  PROVISIONAL_READY_FOR_VALUE  

Archivos actualizados:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_working.csv
 - d:\TRABAJO DE GRAD

Esta celda crea una plantilla única y compacta con los campos numéricos que aún faltan para cerrar el paquete de entrada de BenMAP-CE. En lugar de editar varias tablas por separado, el código extrae desde los archivos de trabajo las combinaciones activas de contaminante y endpoint, construye una tabla para funciones concentración–respuesta y otra para valoración económica, y las exporta en un solo archivo editable. Así, los valores de riesgo relativo, cambio de concentración, beta, error estándar y valoración monetaria podrán diligenciarse de forma centralizada antes de incorporarlos nuevamente al flujo final.

In [23]:
# ============================================================
# CELDA 14: Crear plantilla única editable de pendientes numéricos
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

health_working_file = output_dir / "benmap_health_impact_functions_working.csv"
valuation_working_file = output_dir / "benmap_valuation_functions_working.csv"

for f in [health_working_file, valuation_working_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

numeric_inputs_template_file = output_dir / "benmap_numeric_inputs_template.csv"

# ------------------------------------------------------------
# 2) Cargar archivos de trabajo
# ------------------------------------------------------------
df_health = pd.read_csv(health_working_file)
df_val = pd.read_csv(valuation_working_file)

# ------------------------------------------------------------
# 3) Construir plantilla para funciones C-R
# ------------------------------------------------------------
health_cols_needed = [
    "endpoint_code_local", "Pollutant", "Endpoint", "Metric",
    "reference_short", "beta_per_unit", "beta_se", "unit_change"
]
missing_health_cols = [c for c in health_cols_needed if c not in df_health.columns]
if missing_health_cols:
    raise ValueError(f"Faltan columnas en health working: {missing_health_cols}")

health_template = df_health[health_cols_needed].copy()
health_template["section"] = "health_impact_functions"
health_template["rr_value"] = np.nan
health_template["beta_direct"] = health_template["beta_per_unit"]
health_template["fill_priority"] = "Llenar rr_value o beta_direct; unit_change debe quedar definido"
health_template["notes_numeric"] = "beta_se opcional pero recomendado"

health_template = health_template[[
    "section",
    "endpoint_code_local",
    "Pollutant",
    "Endpoint",
    "Metric",
    "reference_short",
    "rr_value",
    "unit_change",
    "beta_direct",
    "beta_se",
    "fill_priority",
    "notes_numeric"
]].copy()

# ------------------------------------------------------------
# 4) Construir plantilla para valoración económica
# ------------------------------------------------------------
valuation_cols_needed = [
    "endpoint_code_local", "Endpoint", "Reference",
    "Point Estimate", "Constant Value"
]
missing_val_cols = [c for c in valuation_cols_needed if c not in df_val.columns]
if missing_val_cols:
    raise ValueError(f"Faltan columnas en valuation working: {missing_val_cols}")

valuation_template = df_val[valuation_cols_needed].copy()
valuation_template["section"] = "valuation_functions"
valuation_template["rr_value"] = np.nan
valuation_template["unit_change"] = np.nan
valuation_template["beta_direct"] = np.nan
valuation_template["beta_se"] = np.nan
valuation_template["fill_priority"] = "Llenar Point Estimate; Constant Value solo si aplica"
valuation_template["notes_numeric"] = "Usar misma moneda y año base definidos en la tesis"

valuation_template = valuation_template.rename(columns={
    "Point Estimate": "point_estimate",
    "Constant Value": "constant_value",
    "Reference": "reference_short"
})

valuation_template["Pollutant"] = ""
valuation_template["Metric"] = ""

valuation_template = valuation_template[[
    "section",
    "endpoint_code_local",
    "Pollutant",
    "Endpoint",
    "Metric",
    "reference_short",
    "rr_value",
    "unit_change",
    "beta_direct",
    "beta_se",
    "fill_priority",
    "notes_numeric",
    "point_estimate",
    "constant_value"
]].copy()

# ------------------------------------------------------------
# 5) Unir y exportar plantilla única
# ------------------------------------------------------------
combined_template = pd.concat(
    [health_template, valuation_template],
    ignore_index=True
)

combined_template.to_csv(numeric_inputs_template_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 6) Mostrar resultado
# ------------------------------------------------------------
print("Plantilla única de pendientes numéricos generada:")
print(combined_template)

print("\nArchivo generado:")
print(" -", numeric_inputs_template_file)

Plantilla única de pendientes numéricos generada:
                   section endpoint_code_local Pollutant  \
0  health_impact_functions               def_I      PM25   
1  health_impact_functions               def_J      PM25   
2  health_impact_functions               def_J        O3   
3  health_impact_functions               def_I       NO2   
4      valuation_functions               def_I             
5      valuation_functions               def_J             

                Endpoint            Metric  \
0  Circulatory mortality  PM25 Annual Mean   
1  Respiratory mortality  PM25 Annual Mean   
2  Respiratory mortality  O3 Seasonal Mean   
3  Circulatory mortality   NO2 Annual Mean   
4  Circulatory mortality                     
5  Respiratory mortality                     

                                     reference_short  rr_value  unit_change  \
0                              Pope & Dockery (2006)       NaN         10.0   
1                              Pope & Dockery (2

Esta celda completa de forma provisional las funciones concentración–respuesta y de valoración económica con una configuración más defendible para el estado actual del proyecto. En PM$_{2.5}$ se propone usar una función estándar de mortalidad de la librería BenMAP basada en \texttt{Di et al. (2017)}, mientras que para O$_3$ se propone una función de mortalidad respiratoria basada en \texttt{Katsouyanni et al. (2009)}. La fila de NO$_2$ se mantiene dentro del flujo, pero en estado de revisión personalizada, sin asignarle todavía un coeficiente numérico. Para valoración económica se propone usar el valor estadístico de la vida (VSL) central reportado por EPA/BenMAP en dólares de 2015, evitando por ahora conversiones no justificadas a COP de 2024. Esta celda no cierra definitivamente todo el paquete, pero sí deja una versión mucho más razonable para seguir trabajando y para revisar después en el reporte de pendientes.

In [24]:
# ============================================================
# CELDA 15: Diligenciar configuración provisional recomendada
# para C-R y valoración económica
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

health_working_file = output_dir / "benmap_health_impact_functions_working.csv"
valuation_working_file = output_dir / "benmap_valuation_functions_working.csv"

if not health_working_file.exists():
    raise FileNotFoundError(f"No se encontró:\n{health_working_file}")
if not valuation_working_file.exists():
    raise FileNotFoundError(f"No se encontró:\n{valuation_working_file}")

# Archivos de salida
health_candidate_working_file = output_dir / "benmap_health_impact_functions_candidate_final_working.csv"
health_candidate_benmap_file = output_dir / "benmap_health_impact_functions_candidate_final.csv"

valuation_candidate_working_file = output_dir / "benmap_valuation_functions_candidate_final_working.csv"
valuation_candidate_benmap_file = output_dir / "benmap_valuation_functions_candidate_final.csv"

# ------------------------------------------------------------
# 2) Cargar archivos
# ------------------------------------------------------------
df_health = pd.read_csv(health_working_file)
df_val = pd.read_csv(valuation_working_file)

# ------------------------------------------------------------
# 3) PM2.5 -> usar una función estándar más defendible:
#    def_I = All-cause mortality, Di et al. (2017)
#    beta = 0.001094 ; se = 0.000050 ; unit_change = 1
# ------------------------------------------------------------
mask_pm25_defI = (
    (df_health["endpoint_code_local"] == "def_I") &
    (df_health["Pollutant"] == "PM25")
)

if mask_pm25_defI.sum() > 0:
    df_health.loc[mask_pm25_defI, "Endpoint"] = "All-cause mortality"
    df_health.loc[mask_pm25_defI, "Metric"] = "Annual (D8HourMax)"
    df_health.loc[mask_pm25_defI, "Author"] = "Di et al."
    df_health.loc[mask_pm25_defI, "Year of Publication"] = 2017
    df_health.loc[mask_pm25_defI, "Start Age"] = 65
    df_health.loc[mask_pm25_defI, "End Age"] = 99
    df_health.loc[mask_pm25_defI, "reference_short"] = "Di et al. (2017)"
    df_health.loc[mask_pm25_defI, "beta_per_unit"] = 0.001094
    df_health.loc[mask_pm25_defI, "beta_se"] = 0.000050
    df_health.loc[mask_pm25_defI, "unit_change"] = 1.0
    df_health.loc[mask_pm25_defI, "Qualifier"] = "BenMAP core PM2.5 mortality function"
    df_health.loc[mask_pm25_defI, "status"] = "READY_NUMERIC"

# ------------------------------------------------------------
# 4) PM2.5 -> fila def_J:
#    no la lleno todavía porque no quiero inventar una función
#    separada de mortalidad respiratoria para PM2.5
# ------------------------------------------------------------
mask_pm25_defJ = (
    (df_health["endpoint_code_local"] == "def_J") &
    (df_health["Pollutant"] == "PM25")
)

if mask_pm25_defJ.sum() > 0:
    df_health.loc[mask_pm25_defJ, "Qualifier"] = "REVIEW_PM25_DUPLICATE_OR_REDEFINE"
    df_health.loc[mask_pm25_defJ, "status"] = "REVIEW_PM25_DUPLICATE"

# ------------------------------------------------------------
# 5) O3 -> usar una función estándar de mortalidad respiratoria:
#    Katsouyanni et al. (2009), single-pollutant D8HourMax
#    beta = 0.000867 ; se = 0.000304 ; unit_change = 1
# ------------------------------------------------------------
mask_o3_defJ = (
    (df_health["endpoint_code_local"] == "def_J") &
    (df_health["Pollutant"] == "O3")
)

if mask_o3_defJ.sum() > 0:
    df_health.loc[mask_o3_defJ, "Endpoint"] = "Respiratory mortality"
    df_health.loc[mask_o3_defJ, "Metric"] = "D8HourMax"
    df_health.loc[mask_o3_defJ, "Author"] = "Katsouyanni et al."
    df_health.loc[mask_o3_defJ, "Year of Publication"] = 2009
    df_health.loc[mask_o3_defJ, "Start Age"] = 0
    df_health.loc[mask_o3_defJ, "End Age"] = 99
    df_health.loc[mask_o3_defJ, "reference_short"] = "Katsouyanni et al. (2009)"
    df_health.loc[mask_o3_defJ, "beta_per_unit"] = 0.000867
    df_health.loc[mask_o3_defJ, "beta_se"] = 0.000304
    df_health.loc[mask_o3_defJ, "unit_change"] = 1.0
    df_health.loc[mask_o3_defJ, "Qualifier"] = "BenMAP core ozone respiratory mortality"
    df_health.loc[mask_o3_defJ, "status"] = "READY_NUMERIC"

# ------------------------------------------------------------
# 6) NO2 -> mantener dentro del flujo, pero sin llenar beta aún
# ------------------------------------------------------------
mask_no2 = df_health["Pollutant"] == "NO2"
if mask_no2.sum() > 0:
    df_health.loc[mask_no2, "Qualifier"] = "NO2 custom function under review"
    df_health.loc[mask_no2, "status"] = "NO2_CUSTOM_REVIEW"

# ------------------------------------------------------------
# 7) Valoración económica:
#    usar VSL central EPA/BenMAP = 8,705,114 USD (2015$)
#    y dejar currency/price_year coherentes con eso
# ------------------------------------------------------------
vsl_2015_usd = 8705114

for endpoint_code in ["def_I", "def_J"]:
    mask_val = df_val["endpoint_code_local"] == endpoint_code
    if mask_val.sum() > 0:
        df_val.loc[mask_val, "Point Estimate"] = vsl_2015_usd
        df_val.loc[mask_val, "Constant Value"] = np.nan
        df_val.loc[mask_val, "Qualifier"] = "EPA VSL central value (2015 USD)"
        df_val.loc[mask_val, "Reference"] = "EPA TSD 2024, Table 22; VSL based on 26 studies"
        if "price_year" in df_val.columns:
            df_val.loc[mask_val, "price_year"] = 2015
        if "currency" in df_val.columns:
            df_val.loc[mask_val, "currency"] = "USD"
        df_val.loc[mask_val, "status"] = "READY_NUMERIC"

# ------------------------------------------------------------
# 8) Guardar working actualizados
# ------------------------------------------------------------
df_health.to_csv(health_working_file, index=False, encoding="utf-8-sig")
df_val.to_csv(valuation_working_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Exportar candidate final working
# ------------------------------------------------------------
df_health_candidate = df_health.copy().reset_index(drop=True)
df_val_candidate = df_val.copy().reset_index(drop=True)

df_health_candidate.to_csv(health_candidate_working_file, index=False, encoding="utf-8-sig")
df_val_candidate.to_csv(valuation_candidate_working_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 10) Exportar candidate final tipo BenMAP
# ------------------------------------------------------------
health_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Pollutant",
    "Metric",
    "Annual Statistic",
    "Seasonal Metric",
    "Race",
    "Ethnicity",
    "Gender",
    "Start Age",
    "End Age",
    "Author",
    "Apply Function To",
    "Year of Publication",
    "Qualifier"
]

valuation_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Qualifier",
    "Reference",
    "Start Age",
    "End Age",
    "Point Estimate",
    "Function",
    "A Description",
    "A",
    "A Distribution",
    "A Parameter 1",
    "A Parameter 2",
    "Constant Description",
    "Constant Value"
]

df_health_candidate[health_base_cols].to_csv(
    health_candidate_benmap_file, index=False, encoding="utf-8-sig"
)

df_val_candidate[valuation_base_cols].to_csv(
    valuation_candidate_benmap_file, index=False, encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 11) Resumen
# ------------------------------------------------------------
print("Resumen actualizado de funciones C-R:")
print(df_health[[
    "endpoint_code_local", "Pollutant", "Endpoint", "Metric",
    "beta_per_unit", "beta_se", "unit_change", "status"
]])

print("\nResumen actualizado de valoración:")
print(df_val[[
    "endpoint_code_local", "Endpoint", "Point Estimate",
    "Qualifier", "status"
]])

print("\nArchivos actualizados:")
print(" -", health_working_file)
print(" -", valuation_working_file)
print(" -", health_candidate_working_file)
print(" -", health_candidate_benmap_file)
print(" -", valuation_candidate_working_file)
print(" -", valuation_candidate_benmap_file)

Resumen actualizado de funciones C-R:
  endpoint_code_local Pollutant               Endpoint              Metric  \
0               def_I      PM25    All-cause mortality  Annual (D8HourMax)   
1               def_J      PM25  Respiratory mortality    PM25 Annual Mean   
2               def_J        O3  Respiratory mortality           D8HourMax   
3               def_I       NO2  Circulatory mortality     NO2 Annual Mean   

   beta_per_unit   beta_se  unit_change                 status  
0       0.001094  0.000050          1.0          READY_NUMERIC  
1            NaN       NaN         10.0  REVIEW_PM25_DUPLICATE  
2       0.000867  0.000304          1.0          READY_NUMERIC  
3            NaN       NaN         10.0      NO2_CUSTOM_REVIEW  

Resumen actualizado de valoración:
  endpoint_code_local               Endpoint  Point Estimate  \
0               def_I  Circulatory mortality       8705114.0   
1               def_J  Respiratory mortality       8705114.0   

                 



### Qué puse y por qué

Puse esto porque es lo más defendible:

* **PM2.5**: el manual de BenMAP incluye funciones núcleo de mortalidad como **Di et al. (2017)** y **Turner et al. (2016)**, y en la tabla visible aparece Di con beta **0.001094** y SE **0.000050**. 
* **O₃**: el manual muestra para mortalidad respiratoria de **Katsouyanni et al. (2009)** una beta efectiva de **0.000867** y SE **0.000304** para `D8HourMax` single-pollutant. 
* **VSL**: el TSD 2024 reporta como valor central del VSL **8,705,114** en **USD de 2015**; por eso prefiero dejarlo así y no inventarte una conversión a COP de 2024. 

Lo único que dejé sin llenar a propósito es **NO₂**, porque mantenerlo dentro del flujo está bien, pero ponerle un beta numérico sin una función primaria bien justificada sería más débil metodológicamente que dejarlo como revisión custom.


In [25]:
# ============================================================
# CELDA 12: Generar reporte de pendientes y archivos
# candidate final para BenMAP-CE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Parámetros de control
# ------------------------------------------------------------
include_no2_custom = True   # False = excluir fila custom NO2 por ahora
export_candidate_final = True

# ------------------------------------------------------------
# 2) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

health_working_file = output_dir / "benmap_health_impact_functions_working.csv"
valuation_working_file = output_dir / "benmap_valuation_functions_working.csv"

for f in [health_working_file, valuation_working_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

health_candidate_working_file = output_dir / "benmap_health_impact_functions_candidate_final_working.csv"
health_candidate_benmap_file = output_dir / "benmap_health_impact_functions_candidate_final.csv"

valuation_candidate_working_file = output_dir / "benmap_valuation_functions_candidate_final_working.csv"
valuation_candidate_benmap_file = output_dir / "benmap_valuation_functions_candidate_final.csv"

pending_report_file = output_dir / "benmap_pending_inputs_report.csv"

# ------------------------------------------------------------
# 3) Cargar archivos de trabajo
# ------------------------------------------------------------
df_health = pd.read_csv(health_working_file)
df_val = pd.read_csv(valuation_working_file)

# ------------------------------------------------------------
# 4) Filtrar NO2 opcional si así se define
# ------------------------------------------------------------
df_health_candidate = df_health.copy()

if not include_no2_custom:
    if "Pollutant" in df_health_candidate.columns:
        df_health_candidate = df_health_candidate[df_health_candidate["Pollutant"] != "NO2"].copy()
    if "status" in df_health_candidate.columns:
        df_health_candidate = df_health_candidate[df_health_candidate["status"] != "NO2_CUSTOM_REVIEW"].copy()

df_health_candidate = df_health_candidate.reset_index(drop=True)

df_val_candidate = df_val.copy().reset_index(drop=True)

# ------------------------------------------------------------
# 5) Funciones auxiliares para detectar pendientes
# ------------------------------------------------------------
def is_empty_like(x):
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() == "nan"

def has_placeholder(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    tokens = ["EDIT_", "PENDING_", "OPTIONAL_", "REVIEW", "PROVISIONAL"]
    return any(tok in s for tok in tokens)

pending_rows = []

def append_pending(df, artifact_name, row_id_cols, required_text_cols=None, required_numeric_cols=None):
    required_text_cols = required_text_cols or []
    required_numeric_cols = required_numeric_cols or []

    for idx, row in df.iterrows():
        row_id = " | ".join([f"{c}={row[c]}" for c in row_id_cols if c in df.columns])

        # texto requerido
        for col in required_text_cols:
            if col in df.columns:
                value = row[col]
                if is_empty_like(value) or has_placeholder(value):
                    pending_rows.append({
                        "artifact": artifact_name,
                        "row_identifier": row_id,
                        "field_name": col,
                        "field_type": "text",
                        "current_value": value,
                        "issue": "missing_or_placeholder"
                    })

        # numérico requerido
        for col in required_numeric_cols:
            if col in df.columns:
                value = row[col]
                if pd.isna(pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]):
                    pending_rows.append({
                        "artifact": artifact_name,
                        "row_identifier": row_id,
                        "field_name": col,
                        "field_type": "numeric",
                        "current_value": value,
                        "issue": "missing_numeric_value"
                    })

# ------------------------------------------------------------
# 6) Detectar pendientes en funciones C-R
# ------------------------------------------------------------
health_row_id = ["endpoint_code_local", "Pollutant", "Endpoint"]

health_required_text = [
    "Metric", "Author", "Qualifier", "reference_short", "status"
]

health_required_numeric = [
    "beta_per_unit", "unit_change"
]

# beta_se puede quedar vacío si luego decides no usarlo directamente,
# pero lo reportamos aparte como informativo
append_pending(
    df_health_candidate,
    artifact_name="health_impact_functions",
    row_id_cols=health_row_id,
    required_text_cols=health_required_text,
    required_numeric_cols=health_required_numeric
)

# Reporte adicional opcional de beta_se
if "beta_se" in df_health_candidate.columns:
    for _, row in df_health_candidate.iterrows():
        row_id = " | ".join([f"{c}={row[c]}" for c in health_row_id if c in df_health_candidate.columns])
        value = row["beta_se"]
        if pd.isna(pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]):
            pending_rows.append({
                "artifact": "health_impact_functions",
                "row_identifier": row_id,
                "field_name": "beta_se",
                "field_type": "numeric_optional_review",
                "current_value": value,
                "issue": "missing_optional_review"
            })

# ------------------------------------------------------------
# 7) Detectar pendientes en funciones de valoración
# ------------------------------------------------------------
valuation_row_id = ["endpoint_code_local", "Endpoint"]

valuation_required_text = [
    "Qualifier", "Reference", "price_year", "status"
]

# Como la función está definida como PointEstimate*Incidence,
# Point Estimate sí debe cerrarse.
valuation_required_numeric = [
    "Point Estimate"
]

append_pending(
    df_val_candidate,
    artifact_name="valuation_functions",
    row_id_cols=valuation_row_id,
    required_text_cols=valuation_required_text,
    required_numeric_cols=valuation_required_numeric
)

# ------------------------------------------------------------
# 8) Construir reporte de pendientes
# ------------------------------------------------------------
pending_report = pd.DataFrame(pending_rows)

if pending_report.empty:
    print("No se detectaron pendientes en los archivos candidate final.")
else:
    pending_report = pending_report.sort_values(
        by=["artifact", "row_identifier", "field_type", "field_name"]
    ).reset_index(drop=True)

pending_report.to_csv(pending_report_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Exportar candidate final working + tipo BenMAP
# ------------------------------------------------------------
if export_candidate_final:
    # Guardar working filtrados
    df_health_candidate.to_csv(health_candidate_working_file, index=False, encoding="utf-8-sig")
    df_val_candidate.to_csv(valuation_candidate_working_file, index=False, encoding="utf-8-sig")

    # Exportar tipo BenMAP
    health_base_cols = [
        "Endpoint Group",
        "Endpoint",
        "Pollutant",
        "Metric",
        "Annual Statistic",
        "Seasonal Metric",
        "Race",
        "Ethnicity",
        "Gender",
        "Start Age",
        "End Age",
        "Author",
        "Apply Function To",
        "Year of Publication",
        "Qualifier"
    ]

    valuation_base_cols = [
        "Endpoint Group",
        "Endpoint",
        "Qualifier",
        "Reference",
        "Start Age",
        "End Age",
        "Point Estimate",
        "Function",
        "A Description",
        "A",
        "A Distribution",
        "A Parameter 1",
        "A Parameter 2",
        "Constant Description",
        "Constant Value"
    ]

    missing_health_base = [c for c in health_base_cols if c not in df_health_candidate.columns]
    missing_valuation_base = [c for c in valuation_base_cols if c not in df_val_candidate.columns]

    if missing_health_base:
        raise ValueError(f"Faltan columnas para exportar health candidate final: {missing_health_base}")
    if missing_valuation_base:
        raise ValueError(f"Faltan columnas para exportar valuation candidate final: {missing_valuation_base}")

    df_health_candidate_benmap = df_health_candidate[health_base_cols].copy()
    df_val_candidate_benmap = df_val_candidate[valuation_base_cols].copy()

    df_health_candidate_benmap.to_csv(health_candidate_benmap_file, index=False, encoding="utf-8-sig")
    df_val_candidate_benmap.to_csv(valuation_candidate_benmap_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 10) Mostrar resultados
# ------------------------------------------------------------
print("Configuración usada:")
print(f"- include_no2_custom = {include_no2_custom}")
print(f"- export_candidate_final = {export_candidate_final}")

print("\nResumen candidate final - Health:")
print(df_health_candidate[[
    c for c in ["endpoint_code_local", "Pollutant", "Endpoint", "Metric", "status"]
    if c in df_health_candidate.columns
]])

print("\nResumen candidate final - Valuation:")
print(df_val_candidate[[
    c for c in ["endpoint_code_local", "Endpoint", "Qualifier", "price_year", "status"]
    if c in df_val_candidate.columns
]])

if pending_report.empty:
    print("\nNo se detectaron pendientes.")
else:
    print("\nReporte de pendientes detectados:")
    print(pending_report)

print("\nArchivos generados/actualizados:")
if export_candidate_final:
    print(" -", health_candidate_working_file)
    print(" -", health_candidate_benmap_file)
    print(" -", valuation_candidate_working_file)
    print(" -", valuation_candidate_benmap_file)

if not pending_report.empty:
    print(" -", pending_report_file)

Configuración usada:
- include_no2_custom = True
- export_candidate_final = True

Resumen candidate final - Health:
  endpoint_code_local Pollutant               Endpoint              Metric  \
0               def_I      PM25    All-cause mortality  Annual (D8HourMax)   
1               def_J      PM25  Respiratory mortality    PM25 Annual Mean   
2               def_J        O3  Respiratory mortality           D8HourMax   
3               def_I       NO2  Circulatory mortality     NO2 Annual Mean   

                  status  
0          READY_NUMERIC  
1  REVIEW_PM25_DUPLICATE  
2          READY_NUMERIC  
3      NO2_CUSTOM_REVIEW  

Resumen candidate final - Valuation:
  endpoint_code_local               Endpoint  \
0               def_I  Circulatory mortality   
1               def_J  Respiratory mortality   

                          Qualifier  price_year         status  
0  EPA VSL central value (2015 USD)        2015  READY_NUMERIC  
1  EPA VSL central value (2015 USD)        20

# archivos de exposición para BenMAP-CE
Esta celda localiza automáticamente las salidas finales del HBM para O$_3$ y NO$_2$, verifica que tengan la estructura esperada y construye sus archivos de exposición para BenMAP-CE. Para cada contaminante se genera un escenario baseline usando la mediana posterior del modelo (\texttt{p50_hbm}) y un escenario control configurable mediante una reducción porcentual uniforme o un valor objetivo máximo. Además, se conservan los percentiles \texttt{p05_hbm} y \texttt{p95_hbm} para análisis posteriores de sensibilidad. Finalmente, la celda exporta los archivos \texttt{benmap_O3_baseline.csv}, \texttt{benmap_O3_control.csv}, \texttt{benmap_NO2_baseline.csv} y \texttt{benmap_NO2_control.csv}, dejando más completo el bloque de exposición que luego se mostrará y evaluará en BenMAP-CE.

In [1]:
# ============================================================
# CELDA 16: Generar baseline/control para O3 y NO2
# a partir de las salidas finales del HBM
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Configuración de escenarios por contaminante
#    mode:
#      - "relative_reduction"
#      - "target_cap"
# ------------------------------------------------------------
pollutant_configs = {
    "O3": {
        "target_name": "HBM_O3_M1bclean_surface_final.csv",
        "prefix": "o3",
        "mode": "relative_reduction",
        "relative_reduction": 0.10,
        "target_cap": 60.0  # editable si luego decides usar meta fija
    },
    "NO2": {
        "target_name": "HBM_NO2_M1bclean_surface_final.csv",
        "prefix": "no2",
        "mode": "relative_reduction",
        "relative_reduction": 0.10,
        "target_cap": 10.0  # editable si luego decides usar meta fija
    }
}

# ------------------------------------------------------------
# 2) Definir rutas base
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

search_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    project_root,
    project_root / "CODIGO"
]

required_cols = [
    "cell_id", "fecha", "year", "month",
    "p05_hbm", "p50_hbm", "p95_hbm"
]

summary_rows = []

# ------------------------------------------------------------
# 3) Función para buscar archivo automáticamente
# ------------------------------------------------------------
def find_file(target_name, alternative_patterns):
    matches = []

    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(target_name))

    if not matches:
        for root in search_roots:
            if root.exists():
                for pattern in alternative_patterns:
                    matches.extend(root.rglob(pattern))

    matches = list(dict.fromkeys(matches))

    if not matches:
        raise FileNotFoundError(
            f"No se encontró el archivo {target_name} ni patrones alternativos."
        )

    return matches[0], matches

# ------------------------------------------------------------
# 4) Procesar O3 y NO2
# ------------------------------------------------------------
for pollutant, cfg in pollutant_configs.items():
    prefix = cfg["prefix"]
    target_name = cfg["target_name"]

    alternative_patterns = [
        f"*{pollutant}*surface*final*.csv",
        f"*{pollutant}*final*.csv",
        f"*{pollutant}*.csv"
    ]

    input_file, candidates = find_file(target_name, alternative_patterns)

    print(f"\n===== {pollutant} =====")
    print("Archivo seleccionado:", input_file)
    print("Candidatos encontrados:")
    for c in candidates:
        print(" -", c)

    df = pd.read_csv(input_file)

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas obligatorias en {pollutant}: {missing}")

    # Tipos
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["cell_id"] = pd.to_numeric(df["cell_id"], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["month"] = pd.to_numeric(df["month"], errors="coerce")

    if df["fecha"].isna().any():
        raise ValueError(f"Hay fechas inválidas en {pollutant}.")
    if df["cell_id"].isna().any():
        raise ValueError(f"Hay cell_id inválidos en {pollutant}.")
    if df["year"].isna().any() or df["month"].isna().any():
        raise ValueError(f"Hay year/month inválidos en {pollutant}.")

    df["cell_id"] = df["cell_id"].astype(int)
    df["year"] = df["year"].astype(int)
    df["month"] = df["month"].astype(int)

    # Baseline
    baseline = df[["cell_id", "fecha", "year", "month", "p05_hbm", "p50_hbm", "p95_hbm"]].copy()
    baseline = baseline.rename(columns={
        "p50_hbm": f"{prefix}_baseline",
        "p05_hbm": f"{prefix}_p05",
        "p95_hbm": f"{prefix}_p95"
    })

    for col in [f"{prefix}_baseline", f"{prefix}_p05", f"{prefix}_p95"]:
        baseline[col] = pd.to_numeric(baseline[col], errors="coerce").clip(lower=0)

    # Control
    control = baseline.copy()

    if cfg["mode"] == "relative_reduction":
        control[f"{prefix}_control"] = control[f"{prefix}_baseline"] * (1 - cfg["relative_reduction"])
        scenario_desc = f"Reducción uniforme del {cfg['relative_reduction']:.0%}"

    elif cfg["mode"] == "target_cap":
        control[f"{prefix}_control"] = np.minimum(control[f"{prefix}_baseline"], cfg["target_cap"])
        scenario_desc = f"Concentración máxima objetivo de {cfg['target_cap']}"

    else:
        raise ValueError(f"Modo no válido para {pollutant}: {cfg['mode']}")

    control[f"{prefix}_control"] = control[f"{prefix}_control"].clip(lower=0)

    # Exportaciones
    baseline_export = baseline[[
        "cell_id", "fecha", "year", "month",
        f"{prefix}_baseline", f"{prefix}_p05", f"{prefix}_p95"
    ]].copy()

    control_export = control[[
        "cell_id", "fecha", "year", "month",
        f"{prefix}_control"
    ]].copy()

    baseline_export["fecha"] = baseline_export["fecha"].dt.strftime("%Y-%m-%d")
    control_export["fecha"] = control_export["fecha"].dt.strftime("%Y-%m-%d")

    baseline_file = output_dir / f"benmap_{pollutant}_baseline.csv"
    control_file = output_dir / f"benmap_{pollutant}_control.csv"

    baseline_export.to_csv(baseline_file, index=False, encoding="utf-8-sig")
    control_export.to_csv(control_file, index=False, encoding="utf-8-sig")

    summary_rows.append({
        "pollutant": pollutant,
        "scenario": scenario_desc,
        "n_rows_baseline": len(baseline_export),
        "n_rows_control": len(control_export),
        "n_cells": baseline_export["cell_id"].nunique(),
        "period_start": baseline_export["fecha"].min(),
        "period_end": baseline_export["fecha"].max(),
        "baseline_mean": baseline_export[f"{prefix}_baseline"].mean(),
        "control_mean": control_export[f"{prefix}_control"].mean(),
        "baseline_min": baseline_export[f"{prefix}_baseline"].min(),
        "baseline_max": baseline_export[f"{prefix}_baseline"].max(),
        "control_min": control_export[f"{prefix}_control"].min(),
        "control_max": control_export[f"{prefix}_control"].max(),
        "baseline_file": str(baseline_file),
        "control_file": str(control_file),
    })

    print("\nResumen rápido:")
    print(" - Escenario:", scenario_desc)
    print(" - Filas baseline:", len(baseline_export))
    print(" - Filas control:", len(control_export))
    print(" - Celdas:", baseline_export["cell_id"].nunique())
    print(" - Archivo baseline:", baseline_file)
    print(" - Archivo control:", control_file)

# ------------------------------------------------------------
# 5) Exportar resumen general
# ------------------------------------------------------------
summary = pd.DataFrame(summary_rows)
summary_file = output_dir / "benmap_o3_no2_exposure_summary.csv"
summary.to_csv(summary_file, index=False, encoding="utf-8-sig")

print("\n===== RESUMEN GENERAL =====")
print(summary)

print("\nArchivo de resumen generado:")
print(" -", summary_file)


===== O3 =====
Archivo seleccionado: d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_surface_final.csv
Candidatos encontrados:
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_surface_final.csv

Resumen rápido:
 - Escenario: Reducción uniforme del 10%
 - Filas baseline: 15240
 - Filas control: 15240
 - Celdas: 254
 - Archivo baseline: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_O3_baseline.csv
 - Archivo control: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_O3_control.csv

===== NO2 =====
Archivo seleccionado: d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_surface_final.csv
Candidatos encontrados:
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_NO2_V2_OUT\HBM_NO2_M1bclean_surface_final.csv

Resumen rápido:
 - Escenario: Reducción uniforme del 10%
 - Filas baseline: 15240
 - Filas control: 15240
 - Celdas: 254
 - Archivo baseline: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_NO2_baseline.csv
 - Archivo control: d:\TRABAJ


# insumos epidemiológicos y de valoración para una primera corrida para benmap

Esta celda construye una versión limpia de los insumos epidemiológicos y de valoración para una primera corrida estable en BenMAP-CE. El objetivo es exportar únicamente las funciones que ya están completamente cerradas en términos numéricos, es decir, aquellas con estado \texttt{READY_NUMERIC}. De esta forma, se genera un paquete “ready only” que excluye temporalmente las filas que siguen en revisión, como la función personalizada de NO$2$ y la fila duplicada de PM${2.5}$, pero sin borrarlas de los archivos de trabajo. Este paquete permitirá realizar una primera corrida base en BenMAP-CE y, posteriormente, usar las funciones en revisión dentro del análisis de sensibilidad.

In [2]:
# ============================================================
# CELDA 17: Generar paquete READY_ONLY para primera corrida
# en BenMAP-CE
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

health_candidate_working_file = output_dir / "benmap_health_impact_functions_candidate_final_working.csv"
valuation_candidate_working_file = output_dir / "benmap_valuation_functions_candidate_final_working.csv"

if not health_candidate_working_file.exists():
    raise FileNotFoundError(f"No se encontró:\n{health_candidate_working_file}")
if not valuation_candidate_working_file.exists():
    raise FileNotFoundError(f"No se encontró:\n{valuation_candidate_working_file}")

health_ready_working_file = output_dir / "benmap_health_impact_functions_ready_only_working.csv"
health_ready_benmap_file = output_dir / "benmap_health_impact_functions_ready_only.csv"

valuation_ready_working_file = output_dir / "benmap_valuation_functions_ready_only_working.csv"
valuation_ready_benmap_file = output_dir / "benmap_valuation_functions_ready_only.csv"

ready_summary_file = output_dir / "benmap_ready_only_summary.csv"

# ------------------------------------------------------------
# 2) Cargar candidate final working
# ------------------------------------------------------------
df_health = pd.read_csv(health_candidate_working_file)
df_val = pd.read_csv(valuation_candidate_working_file)

# ------------------------------------------------------------
# 3) Filtrar solo READY_NUMERIC
# ------------------------------------------------------------
if "status" not in df_health.columns:
    raise ValueError("El archivo health candidate final working no tiene la columna 'status'.")
if "status" not in df_val.columns:
    raise ValueError("El archivo valuation candidate final working no tiene la columna 'status'.")

df_health_ready = df_health[df_health["status"] == "READY_NUMERIC"].copy().reset_index(drop=True)
df_val_ready = df_val[df_val["status"] == "READY_NUMERIC"].copy().reset_index(drop=True)

# ------------------------------------------------------------
# 4) Validar que no queden vacíos
# ------------------------------------------------------------
if df_health_ready.empty:
    raise ValueError("No hay filas READY_NUMERIC en health impact functions.")
if df_val_ready.empty:
    raise ValueError("No hay filas READY_NUMERIC en valuation functions.")

# ------------------------------------------------------------
# 5) Guardar working ready_only
# ------------------------------------------------------------
df_health_ready.to_csv(health_ready_working_file, index=False, encoding="utf-8-sig")
df_val_ready.to_csv(valuation_ready_working_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 6) Exportar versión tipo BenMAP
# ------------------------------------------------------------
health_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Pollutant",
    "Metric",
    "Annual Statistic",
    "Seasonal Metric",
    "Race",
    "Ethnicity",
    "Gender",
    "Start Age",
    "End Age",
    "Author",
    "Apply Function To",
    "Year of Publication",
    "Qualifier"
]

valuation_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Qualifier",
    "Reference",
    "Start Age",
    "End Age",
    "Point Estimate",
    "Function",
    "A Description",
    "A",
    "A Distribution",
    "A Parameter 1",
    "A Parameter 2",
    "Constant Description",
    "Constant Value"
]

missing_health = [c for c in health_base_cols if c not in df_health_ready.columns]
missing_val = [c for c in valuation_base_cols if c not in df_val_ready.columns]

if missing_health:
    raise ValueError(f"Faltan columnas para exportar health ready_only: {missing_health}")
if missing_val:
    raise ValueError(f"Faltan columnas para exportar valuation ready_only: {missing_val}")

df_health_ready[health_base_cols].to_csv(
    health_ready_benmap_file, index=False, encoding="utf-8-sig"
)

df_val_ready[valuation_base_cols].to_csv(
    valuation_ready_benmap_file, index=False, encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7) Resumen
# ------------------------------------------------------------
summary_rows = []

for _, row in df_health_ready.iterrows():
    summary_rows.append({
        "artifact": "health_ready_only",
        "endpoint_code_local": row.get("endpoint_code_local", ""),
        "pollutant": row.get("Pollutant", ""),
        "endpoint": row.get("Endpoint", ""),
        "status": row.get("status", "")
    })

for _, row in df_val_ready.iterrows():
    summary_rows.append({
        "artifact": "valuation_ready_only",
        "endpoint_code_local": row.get("endpoint_code_local", ""),
        "pollutant": "",
        "endpoint": row.get("Endpoint", ""),
        "status": row.get("status", "")
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(ready_summary_file, index=False, encoding="utf-8-sig")

print("Resumen READY_ONLY - Health:")
print(df_health_ready[[
    c for c in ["endpoint_code_local", "Pollutant", "Endpoint", "Metric", "status"]
    if c in df_health_ready.columns
]])

print("\nResumen READY_ONLY - Valuation:")
print(df_val_ready[[
    c for c in ["endpoint_code_local", "Endpoint", "Point Estimate", "status"]
    if c in df_val_ready.columns
]])

print("\nArchivos generados:")
print(" -", health_ready_working_file)
print(" -", health_ready_benmap_file)
print(" -", valuation_ready_working_file)
print(" -", valuation_ready_benmap_file)
print(" -", ready_summary_file)

Resumen READY_ONLY - Health:
  endpoint_code_local Pollutant               Endpoint              Metric  \
0               def_I      PM25    All-cause mortality  Annual (D8HourMax)   
1               def_J        O3  Respiratory mortality           D8HourMax   

          status  
0  READY_NUMERIC  
1  READY_NUMERIC  

Resumen READY_ONLY - Valuation:
  endpoint_code_local               Endpoint  Point Estimate         status
0               def_I  Circulatory mortality       8705114.0  READY_NUMERIC
1               def_J  Respiratory mortality       8705114.0  READY_NUMERIC

Archivos generados:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_ready_only_working.csv
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_ready_only.csv
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_ready_only_working.csv
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_ready_only.csv
 - d:\TRABAJO DE GRAD

# correcion de consistencias

Esta celda corrige la consistencia entre las funciones de impacto en salud y las funciones de valoración económica dentro del paquete READY_ONLY. En el estado actual, el endpoint \texttt{def_I} quedó definido como \texttt{All-cause mortality} en las funciones epidemiológicas, pero todavía aparece como \texttt{Circulatory mortality} en las funciones de valoración. Para evitar inconsistencias al momento de valorar resultados en BenMAP-CE, esta celda actualiza el nombre del endpoint en los archivos de valoración para que coincida exactamente con el endpoint sanitario utilizado en la corrida base.

In [3]:
# ============================================================
# CELDA 18: Corregir consistencia de endpoint entre health y valuation
# en el paquete READY_ONLY
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

valuation_ready_working_file = output_dir / "benmap_valuation_functions_ready_only_working.csv"
valuation_ready_benmap_file = output_dir / "benmap_valuation_functions_ready_only.csv"

if not valuation_ready_working_file.exists():
    raise FileNotFoundError(f"No se encontró:\n{valuation_ready_working_file}")

# ------------------------------------------------------------
# 2) Cargar archivo
# ------------------------------------------------------------
df_val = pd.read_csv(valuation_ready_working_file)

# ------------------------------------------------------------
# 3) Corregir endpoint de def_I
# ------------------------------------------------------------
mask_defI = df_val["endpoint_code_local"] == "def_I"

if mask_defI.sum() == 0:
    raise ValueError("No se encontró la fila def_I en valuation ready_only working.")

df_val.loc[mask_defI, "Endpoint"] = "All-cause mortality"

# ------------------------------------------------------------
# 4) Guardar working actualizado
# ------------------------------------------------------------
df_val.to_csv(valuation_ready_working_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 5) Exportar versión tipo BenMAP
# ------------------------------------------------------------
valuation_base_cols = [
    "Endpoint Group",
    "Endpoint",
    "Qualifier",
    "Reference",
    "Start Age",
    "End Age",
    "Point Estimate",
    "Function",
    "A Description",
    "A",
    "A Distribution",
    "A Parameter 1",
    "A Parameter 2",
    "Constant Description",
    "Constant Value"
]

missing_cols = [c for c in valuation_base_cols if c not in df_val.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas para exportar valuation ready_only: {missing_cols}")

df_val[valuation_base_cols].to_csv(
    valuation_ready_benmap_file, index=False, encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6) Mostrar resultado
# ------------------------------------------------------------
print("Valoración READY_ONLY corregida:")
print(df_val[[
    c for c in ["endpoint_code_local", "Endpoint", "Point Estimate", "status"]
    if c in df_val.columns
]])

print("\nArchivos actualizados:")
print(" -", valuation_ready_working_file)
print(" -", valuation_ready_benmap_file)

Valoración READY_ONLY corregida:
  endpoint_code_local               Endpoint  Point Estimate         status
0               def_I    All-cause mortality       8705114.0  READY_NUMERIC
1               def_J  Respiratory mortality       8705114.0  READY_NUMERIC

Archivos actualizados:
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_ready_only_working.csv
 - d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_ready_only.csv


# reconstruye la grilla espacial de BenMAP

Esta celda reconstruye la grilla espacial de BenMAP a partir de la capa de 3 km del proyecto y crea explícitamente los campos \texttt{ROW} y \texttt{COL}, que BenMAP-CE necesita para interpretar correctamente una \textit{Shapefile Grid}. En lugar de dejar que el programa los genere automáticamente, el código asigna columnas según la posición horizontal de cada celda y filas según la posición vertical, preservando la estructura real de la malla. El resultado es un nuevo shapefile listo para BenMAP, con identificador de celda, índices \texttt{ROW}/\texttt{COL} y geometría, evitando que la grilla quede interpretada erróneamente como una sola columna con 254 filas.

In [4]:
# ============================================================
# CELDA 19: Reconstruir shapefile BenMAP con ROW y COL correctos
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Definir rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

# Cambia esta ruta si quieres usar otra capa fuente
source_candidates = [
    project_root / "Qgis" / "grid_3km_time_idw_2020_2024.gpkg",
    output_dir / "benmap_grid_definition.shp",
]

source_file = None
for p in source_candidates:
    if p.exists():
        source_file = p
        break

if source_file is None:
    raise FileNotFoundError(
        "No se encontró una capa fuente de la grilla. "
        "Revisa grid_3km_time_idw_2020_2024.gpkg o benmap_grid_definition.shp."
    )

# Archivo final corregido
final_shp = output_dir / "benmap_grid_definition_ROWCOL.shp"

# ------------------------------------------------------------
# 2) Cargar capa fuente
# ------------------------------------------------------------
if source_file.suffix.lower() == ".gpkg":
    # intenta leer la primera capa
    gdf = gpd.read_file(source_file)
else:
    gdf = gpd.read_file(source_file)

if gdf.empty:
    raise ValueError("La capa fuente de la grilla está vacía.")

print("Archivo fuente cargado:", source_file)
print("Dimensión:", gdf.shape)
print("Columnas disponibles:", gdf.columns.tolist())

# ------------------------------------------------------------
# 3) Identificar cell_id
# ------------------------------------------------------------
possible_id_cols = ["cell_id", "cellid", "cell_idx", "id", "grid_id", "CellID"]

id_col = None
for c in possible_id_cols:
    if c in gdf.columns:
        id_col = c
        break

if id_col is None:
    raise ValueError(
        "No se encontró una columna identificadora tipo cell_id en la grilla."
    )

gdf = gdf.copy()
gdf["cell_id"] = pd.to_numeric(gdf[id_col], errors="coerce")
gdf = gdf.dropna(subset=["cell_id"]).copy()
gdf["cell_id"] = gdf["cell_id"].astype(int)

# ------------------------------------------------------------
# 4) Reproyectar a un CRS métrico para construir ROW/COL
# ------------------------------------------------------------
gdf_metric = gdf.copy()

if gdf_metric.crs is None:
    raise ValueError("La grilla no tiene CRS definido. Debes revisar la capa fuente.")

# Si está geográfica, pasar a EPSG:3116 para cálculos
if gdf_metric.crs.is_geographic:
    gdf_metric = gdf_metric.to_crs(epsg=3116)

# ------------------------------------------------------------
# 5) Calcular coordenadas base por celda
#    Usamos centroides para ordenar la malla
# ------------------------------------------------------------
gdf_metric["cx"] = gdf_metric.geometry.centroid.x
gdf_metric["cy"] = gdf_metric.geometry.centroid.y

# Redondeo para evitar ruido numérico
gdf_metric["cx_r"] = gdf_metric["cx"].round(3)
gdf_metric["cy_r"] = gdf_metric["cy"].round(3)

# Columnas: x ascendente
unique_x = sorted(gdf_metric["cx_r"].unique())
col_map = {x: i + 1 for i, x in enumerate(unique_x)}

# Filas: y descendente (fila 1 arriba)
unique_y = sorted(gdf_metric["cy_r"].unique(), reverse=True)
row_map = {y: i + 1 for i, y in enumerate(unique_y)}

gdf_metric["COL"] = gdf_metric["cx_r"].map(col_map).astype(int)
gdf_metric["ROW"] = gdf_metric["cy_r"].map(row_map).astype(int)

# ------------------------------------------------------------
# 6) Validaciones
# ------------------------------------------------------------
dup_pairs = gdf_metric.duplicated(subset=["ROW", "COL"]).sum()

print("\nValidaciones:")
print("Número de celdas:", len(gdf_metric))
print("Número de columnas únicas:", gdf_metric["COL"].nunique())
print("Número de filas únicas:", gdf_metric["ROW"].nunique())
print("Duplicados ROW-COL:", dup_pairs)

if dup_pairs > 0:
    raise ValueError(
        "Se encontraron combinaciones duplicadas de ROW y COL. "
        "Revisa la malla fuente antes de exportar."
    )

# ------------------------------------------------------------
# 7) Preparar shapefile final
# ------------------------------------------------------------
# Volver al CRS original para exportar si quieres conservarlo
gdf_export = gdf_metric.to_crs(gdf.crs).copy()

# Conservar campos mínimos para BenMAP
gdf_export = gdf_export[["cell_id", "ROW", "COL", "geometry"]].copy()

# Shapefile limita nombres largos, así que están bien así
gdf_export.to_file(final_shp)

# ------------------------------------------------------------
# 8) Resumen
# ------------------------------------------------------------
print("\nShapefile final exportado:")
print(final_shp)

print("\nVista previa:")
print(gdf_export.head())

print("\nResumen final:")
print("cell_id únicos:", gdf_export["cell_id"].nunique())
print("ROW únicos:", gdf_export["ROW"].nunique())
print("COL únicos:", gdf_export["COL"].nunique())

Archivo fuente cargado: d:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time_idw_2020_2024.gpkg
Dimensión: (254, 3)
Columnas disponibles: ['cell_id', 'LocNombre', 'geometry']

Validaciones:
Número de celdas: 254
Número de columnas únicas: 154
Número de filas únicas: 176
Duplicados ROW-COL: 0

Shapefile final exportado:
d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_ROWCOL.shp

Vista previa:
   cell_id  ROW  COL                                           geometry
0        0  176    1  MULTIPOLYGON (((961646.696 907337.529, 961646....
1        1  173    2  MULTIPOLYGON (((961646.696 910337.529, 961646....
2        2  170    3  MULTIPOLYGON (((961646.696 910337.529, 960815....
3       41  175    4  MULTIPOLYGON (((961646.696 907337.529, 964646....
4       42  172    5  MULTIPOLYGON (((961646.696 907337.529, 961646....

Resumen final:
cell_id únicos: 254
ROW únicos: 176
COL únicos: 154


d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'grid_3km_time_idw_2020_2024.gpkg': 'grid_3km_static' (default), 'localidades_bogota', 'grid_3km_time_idw'. Specify layer parameter to avoid this warning.
  result = read_func(


Esta celda reconstruye nuevamente los campos \texttt{ROW} y \texttt{COL} del shapefile de BenMAP, pero ahora usando la posición espacial de cada celda en la malla de 3 km a partir de sus límites geométricos, en lugar de sus centroides. Esto es importante porque, cuando las celdas del borde están recortadas por el contorno de Bogotá, los centroides dejan de representar correctamente la columna y la fila reales de la grilla. El código calcula un origen común, estima el tamaño de celda predominante y asigna índices \texttt{ROW}/\texttt{COL} de forma consistente con la estructura regular de la malla. Luego exporta un nuevo shapefile corregido para BenMAP-CE y reporta cuántas filas y columnas únicas resultan.

In [5]:
# ============================================================
# CELDA 19B: Reconstruir ROW/COL usando límites de celda
# y paso espacial de la grilla
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

source_file = project_root / "Qgis" / "grid_3km_time_idw_2020_2024.gpkg"
final_shp = output_dir / "benmap_grid_definition_ROWCOL_v2.shp"

if not source_file.exists():
    raise FileNotFoundError(f"No se encontró el archivo fuente:\n{source_file}")

# ------------------------------------------------------------
# 2) Cargar capa
# ------------------------------------------------------------
gdf = gpd.read_file(source_file)

if gdf.empty:
    raise ValueError("La capa fuente está vacía.")

print("Archivo fuente cargado:", source_file)
print("Dimensión:", gdf.shape)
print("Columnas disponibles:", gdf.columns.tolist())

# ------------------------------------------------------------
# 3) Identificar cell_id
# ------------------------------------------------------------
possible_id_cols = ["cell_id", "cellid", "cell_idx", "id", "grid_id", "CellID"]
id_col = None
for c in possible_id_cols:
    if c in gdf.columns:
        id_col = c
        break

if id_col is None:
    raise ValueError("No se encontró una columna tipo cell_id.")

gdf = gdf.copy()
gdf["cell_id"] = pd.to_numeric(gdf[id_col], errors="coerce")
gdf = gdf.dropna(subset=["cell_id"]).copy()
gdf["cell_id"] = gdf["cell_id"].astype(int)

# ------------------------------------------------------------
# 4) CRS métrico
# ------------------------------------------------------------
if gdf.crs is None:
    raise ValueError("La capa no tiene CRS definido.")

gdf_metric = gdf.copy()
if gdf_metric.crs.is_geographic:
    gdf_metric = gdf_metric.to_crs(epsg=3116)

# ------------------------------------------------------------
# 5) Obtener bounds de cada celda
# ------------------------------------------------------------
bounds = gdf_metric.geometry.bounds
gdf_metric["minx"] = bounds["minx"]
gdf_metric["miny"] = bounds["miny"]
gdf_metric["maxx"] = bounds["maxx"]
gdf_metric["maxy"] = bounds["maxy"]
gdf_metric["width"] = gdf_metric["maxx"] - gdf_metric["minx"]
gdf_metric["height"] = gdf_metric["maxy"] - gdf_metric["miny"]

# ------------------------------------------------------------
# 6) Estimar paso espacial de la malla
#    usamos la mediana de anchos/altos positivos
# ------------------------------------------------------------
width_pos = gdf_metric.loc[gdf_metric["width"] > 0, "width"]
height_pos = gdf_metric.loc[gdf_metric["height"] > 0, "height"]

cell_dx = float(width_pos.median())
cell_dy = float(height_pos.median())

print("\nPaso estimado de celda:")
print("dx =", cell_dx)
print("dy =", cell_dy)

# Si quieres forzarlo a 3000 m:
target_step = 3000.0

# ------------------------------------------------------------
# 7) Definir origen común de la grilla
# ------------------------------------------------------------
origin_x = gdf_metric["minx"].min()
origin_y = gdf_metric["miny"].min()
top_y = gdf_metric["maxy"].max()

# ------------------------------------------------------------
# 8) Asignar COL y ROW usando el origen y el paso fijo
# ------------------------------------------------------------
# COL: izquierda -> derecha
gdf_metric["COL"] = np.floor((gdf_metric["minx"] - origin_x) / target_step + 0.5).astype(int) + 1

# ROW: arriba -> abajo
gdf_metric["ROW"] = np.floor((top_y - gdf_metric["maxy"]) / target_step + 0.5).astype(int) + 1

# ------------------------------------------------------------
# 9) Validaciones
# ------------------------------------------------------------
dup_pairs = gdf_metric.duplicated(subset=["ROW", "COL"]).sum()

print("\nValidaciones:")
print("Número de celdas:", len(gdf_metric))
print("ROW únicos:", gdf_metric["ROW"].nunique())
print("COL únicos:", gdf_metric["COL"].nunique())
print("Duplicados ROW-COL:", dup_pairs)

if dup_pairs > 0:
    dup_df = gdf_metric[gdf_metric.duplicated(subset=["ROW", "COL"], keep=False)][
        ["cell_id", "ROW", "COL"]
    ].sort_values(["ROW", "COL"])
    print("\nDuplicados detectados:")
    print(dup_df.head(20))
    raise ValueError(
        "Se encontraron combinaciones duplicadas de ROW y COL. "
        "Hay que revisar la alineación de la grilla."
    )

# ------------------------------------------------------------
# 10) Exportar shapefile final
# ------------------------------------------------------------
gdf_export = gdf_metric.to_crs(gdf.crs).copy()
gdf_export = gdf_export[["cell_id", "ROW", "COL", "geometry"]].copy()
gdf_export.to_file(final_shp)

print("\nShapefile final exportado:")
print(final_shp)

print("\nVista previa:")
print(gdf_export.head())

print("\nResumen final:")
print("cell_id únicos:", gdf_export["cell_id"].nunique())
print("ROW únicos:", gdf_export["ROW"].nunique())
print("COL únicos:", gdf_export["COL"].nunique())

d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'grid_3km_time_idw_2020_2024.gpkg': 'grid_3km_static' (default), 'localidades_bogota', 'grid_3km_time_idw'. Specify layer parameter to avoid this warning.
  result = read_func(


Archivo fuente cargado: d:\TRABAJO DE GRADO BEN-MAP\Qgis\grid_3km_time_idw_2020_2024.gpkg
Dimensión: (254, 3)
Columnas disponibles: ['cell_id', 'LocNombre', 'geometry']

Paso estimado de celda:
dx = 3000.0
dy = 3000.0

Validaciones:
Número de celdas: 254
ROW únicos: 41
COL únicos: 18
Duplicados ROW-COL: 22

Duplicados detectados:
     cell_id  ROW  COL
250      695    2   17
251      696    2   17
168      488    4   13
192      529    4   13
140      446    5   12
167      487    5   12
112      402    8   10
113      403    8   10
252      729    9   18
253      730    9   18
109      398   12   11
133      439   12   11
108      397   13   11
132      438   13   11
107      396   14   11
131      437   14   11
86       352   17   10
87       353   17   10
104      393   17   10
83       349   20   10


ValueError: Se encontraron combinaciones duplicadas de ROW y COL. Hay que revisar la alineación de la grilla.

Esta celda reconstruye los campos \texttt{ROW} y \texttt{COL} para la grilla de BenMAP-CE usando como fuente el shapefile limpio previamente exportado, en lugar del archivo geopackage que incluye información adicional por localidad. Esto evita duplicidades artificiales en las combinaciones fila-columna y permite generar una versión de la grilla con una sola geometría por celda. El código calcula los centroides en un sistema métrico, asigna índices de fila y columna según la posición espacial de cada celda y exporta un nuevo shapefile listo para volver a cargarse en BenMAP-CE como \textit{Shapefile Grid}.

In [6]:
# ============================================================
# CELDA 19C: Crear shapefile BenMAP ROW/COL desde la grilla limpia
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

source_file = output_dir / "benmap_grid_definition.shp"
final_shp = output_dir / "benmap_grid_definition_ready.shp"

if not source_file.exists():
    raise FileNotFoundError(f"No se encontró el archivo fuente:\n{source_file}")

# ------------------------------------------------------------
# 2) Cargar shapefile limpio
# ------------------------------------------------------------
gdf = gpd.read_file(source_file)

if gdf.empty:
    raise ValueError("La capa fuente está vacía.")

print("Archivo fuente cargado:", source_file)
print("Dimensión:", gdf.shape)
print("Columnas disponibles:", gdf.columns.tolist())

# ------------------------------------------------------------
# 3) Identificar cell_id
# ------------------------------------------------------------
possible_id_cols = ["cell_id", "cellid", "cell_idx", "id", "grid_id", "CellID"]
id_col = None
for c in possible_id_cols:
    if c in gdf.columns:
        id_col = c
        break

if id_col is None:
    raise ValueError("No se encontró una columna tipo cell_id en la grilla limpia.")

gdf = gdf.copy()
gdf["cell_id"] = pd.to_numeric(gdf[id_col], errors="coerce")
gdf = gdf.dropna(subset=["cell_id"]).copy()
gdf["cell_id"] = gdf["cell_id"].astype(int)

# ------------------------------------------------------------
# 4) CRS métrico
# ------------------------------------------------------------
if gdf.crs is None:
    raise ValueError("La capa no tiene CRS definido.")

gdf_metric = gdf.copy()
if gdf_metric.crs.is_geographic:
    gdf_metric = gdf_metric.to_crs(epsg=3116)

# ------------------------------------------------------------
# 5) Calcular centroides
# ------------------------------------------------------------
gdf_metric["cx"] = gdf_metric.geometry.centroid.x
gdf_metric["cy"] = gdf_metric.geometry.centroid.y

# ------------------------------------------------------------
# 6) Asignar COL y ROW con paso fijo de 3000 m
# ------------------------------------------------------------
target_step = 3000.0

origin_x = gdf_metric["cx"].min()
top_y = gdf_metric["cy"].max()

gdf_metric["COL"] = np.round((gdf_metric["cx"] - origin_x) / target_step).astype(int) + 1
gdf_metric["ROW"] = np.round((top_y - gdf_metric["cy"]) / target_step).astype(int) + 1

# ------------------------------------------------------------
# 7) Validaciones
# ------------------------------------------------------------
dup_pairs = gdf_metric.duplicated(subset=["ROW", "COL"]).sum()

print("\nValidaciones:")
print("Número de celdas:", len(gdf_metric))
print("ROW únicos:", gdf_metric["ROW"].nunique())
print("COL únicos:", gdf_metric["COL"].nunique())
print("Duplicados ROW-COL:", dup_pairs)

if dup_pairs > 0:
    dup_df = gdf_metric[gdf_metric.duplicated(subset=["ROW", "COL"], keep=False)][
        ["cell_id", "ROW", "COL"]
    ].sort_values(["ROW", "COL"])
    print("\nDuplicados detectados:")
    print(dup_df.head(20))
    raise ValueError(
        "Todavía hay duplicados en ROW y COL usando la grilla limpia. "
        "En ese caso revisamos el shapefile fuente antes de volver a BenMAP."
    )

# ------------------------------------------------------------
# 8) Exportar shapefile final
# ------------------------------------------------------------
gdf_export = gdf_metric.to_crs(gdf.crs).copy()
gdf_export = gdf_export[["cell_id", "ROW", "COL", "geometry"]].copy()
gdf_export.to_file(final_shp)

# ------------------------------------------------------------
# 9) Resumen
# ------------------------------------------------------------
print("\nShapefile final exportado:")
print(final_shp)

print("\nVista previa:")
print(gdf_export.head())

print("\nResumen final:")
print("cell_id únicos:", gdf_export["cell_id"].nunique())
print("ROW únicos:", gdf_export["ROW"].nunique())
print("COL únicos:", gdf_export["COL"].nunique())

Archivo fuente cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition.shp
Dimensión: (254, 2)
Columnas disponibles: ['cell_id', 'geometry']

Validaciones:
Número de celdas: 254
ROW únicos: 41
COL únicos: 18
Duplicados ROW-COL: 11

Duplicados detectados:
     cell_id  ROW  COL
252      729    8   18
253      730    8   18
228      643   12   16
229      644   12   16
214      601   13   15
215      602   13   15
199      559   14   14
200      560   14   14
197      555   18   14
198      556   18   14
172      506   27   13
195      546   27   13
143      463   29   12
169      503   29   12
170      504   29   12
51       251   35    7
52       252   35    7
26       166   38    5
27       167   38    5
7         82   40    3


ValueError: Todavía hay duplicados en ROW y COL usando la grilla limpia. En ese caso revisamos el shapefile fuente antes de volver a BenMAP.

Esta celda identifica las capas disponibles dentro del archivo geopackage de la malla 3 km y carga explícitamente la capa espacial estática de la grilla, en lugar de dejar que GeoPandas abra automáticamente una capa que puede estar cruzada con localidades o recortada. Después, el código reconstruye los campos \texttt{ROW} y \texttt{COL} usando las coordenadas centroidales agrupadas por posición espacial, valida que no existan duplicados en la combinación fila-columna y exporta un nuevo shapefile listo para BenMAP-CE. Esta corrección es necesaria porque la lectura automática del geopackage estaba tomando una capa que no representaba la grilla pura y por eso generaba duplicidades artificiales.

In [7]:
# ============================================================
# CELDA 19D: Usar la capa correcta del GPKG (grid_3km_static)
# y generar shapefile listo para BenMAP
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import fiona

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

gpkg_file = project_root / "Qgis" / "grid_3km_time_idw_2020_2024.gpkg"
final_shp = output_dir / "benmap_grid_definition_final.shp"

if not gpkg_file.exists():
    raise FileNotFoundError(f"No se encontró el archivo:\n{gpkg_file}")

# ------------------------------------------------------------
# 2) Listar capas disponibles
# ------------------------------------------------------------
layers = fiona.listlayers(gpkg_file)
print("Capas disponibles en el GPKG:")
for i, lyr in enumerate(layers, 1):
    print(f"{i}. {lyr}")

# ------------------------------------------------------------
# 3) Elegir la capa correcta
#    Preferimos una capa estática de la grilla
# ------------------------------------------------------------
preferred_candidates = [
    "grid_3km_static",
    "grid_3km",
    "malla_3km",
    "grid_static"
]

selected_layer = None
for lyr in preferred_candidates:
    if lyr in layers:
        selected_layer = lyr
        break

if selected_layer is None:
    # Si no aparece un nombre exacto, buscamos algo que contenga "static"
    for lyr in layers:
        if "static" in lyr.lower():
            selected_layer = lyr
            break

if selected_layer is None:
    raise ValueError(
        "No se encontró una capa estática de la grilla dentro del GPKG. "
        "Revisa los nombres listados arriba."
    )

print("\nCapa seleccionada:", selected_layer)

# ------------------------------------------------------------
# 4) Cargar la capa correcta
# ------------------------------------------------------------
gdf = gpd.read_file(gpkg_file, layer=selected_layer)

if gdf.empty:
    raise ValueError("La capa seleccionada está vacía.")

print("\nDimensión:", gdf.shape)
print("Columnas disponibles:", gdf.columns.tolist())

# ------------------------------------------------------------
# 5) Identificar cell_id
# ------------------------------------------------------------
possible_id_cols = ["cell_id", "cellid", "cell_idx", "id", "grid_id", "CellID"]

id_col = None
for c in possible_id_cols:
    if c in gdf.columns:
        id_col = c
        break

if id_col is None:
    raise ValueError("No se encontró una columna identificadora tipo cell_id.")

gdf = gdf.copy()
gdf["cell_id"] = pd.to_numeric(gdf[id_col], errors="coerce")
gdf = gdf.dropna(subset=["cell_id"]).copy()
gdf["cell_id"] = gdf["cell_id"].astype(int)

# Eliminar duplicados por cell_id si existieran
gdf = gdf.drop_duplicates(subset=["cell_id"]).copy()

# ------------------------------------------------------------
# 6) Pasar a CRS métrico si hace falta
# ------------------------------------------------------------
if gdf.crs is None:
    raise ValueError("La capa no tiene CRS definido.")

gdf_metric = gdf.copy()
if gdf_metric.crs.is_geographic:
    gdf_metric = gdf_metric.to_crs(epsg=3116)

# ------------------------------------------------------------
# 7) Calcular centroides y agrupar posiciones
#    usando el tamaño de celda de 3000 m
# ------------------------------------------------------------
gdf_metric["cx"] = gdf_metric.geometry.centroid.x
gdf_metric["cy"] = gdf_metric.geometry.centroid.y

target_step = 3000.0
tol = target_step / 2  # tolerancia de 1500 m

# Agrupar coordenadas X en columnas
x_vals = np.sort(gdf_metric["cx"].values)
x_groups = []

for x in x_vals:
    if not x_groups:
        x_groups.append([x])
    else:
        if abs(x - np.mean(x_groups[-1])) <= tol:
            x_groups[-1].append(x)
        else:
            x_groups.append([x])

x_centers = [np.mean(g) for g in x_groups]

# Agrupar coordenadas Y en filas
y_vals = np.sort(gdf_metric["cy"].values)[::-1]  # de arriba hacia abajo
y_groups = []

for y in y_vals:
    if not y_groups:
        y_groups.append([y])
    else:
        if abs(y - np.mean(y_groups[-1])) <= tol:
            y_groups[-1].append(y)
        else:
            y_groups.append([y])

y_centers = [np.mean(g) for g in y_groups]

print("\nAgrupación espacial:")
print("Número de columnas detectadas:", len(x_centers))
print("Número de filas detectadas:", len(y_centers))

# Función auxiliar para asignar grupo más cercano
def assign_group(value, centers):
    distances = [abs(value - c) for c in centers]
    return int(np.argmin(distances)) + 1

gdf_metric["COL"] = gdf_metric["cx"].apply(lambda v: assign_group(v, x_centers))
gdf_metric["ROW"] = gdf_metric["cy"].apply(lambda v: assign_group(v, y_centers))

# ------------------------------------------------------------
# 8) Validaciones
# ------------------------------------------------------------
dup_pairs = gdf_metric.duplicated(subset=["ROW", "COL"]).sum()

print("\nValidaciones:")
print("Número de celdas:", len(gdf_metric))
print("cell_id únicos:", gdf_metric["cell_id"].nunique())
print("ROW únicos:", gdf_metric["ROW"].nunique())
print("COL únicos:", gdf_metric["COL"].nunique())
print("Duplicados ROW-COL:", dup_pairs)

if dup_pairs > 0:
    dup_df = gdf_metric[gdf_metric.duplicated(subset=["ROW", "COL"], keep=False)][
        ["cell_id", "ROW", "COL"]
    ].sort_values(["ROW", "COL"])
    print("\nDuplicados detectados:")
    print(dup_df.head(30))
    raise ValueError(
        "Todavía hay duplicados en ROW y COL incluso usando la capa estática. "
        "En ese caso tocaría inspeccionar directamente la geometría fuente."
    )

# ------------------------------------------------------------
# 9) Exportar shapefile final
# ------------------------------------------------------------
gdf_export = gdf_metric.to_crs(gdf.crs).copy()
gdf_export = gdf_export[["cell_id", "ROW", "COL", "geometry"]].copy()
gdf_export.to_file(final_shp)

print("\nShapefile final exportado:")
print(final_shp)

print("\nVista previa:")
print(gdf_export.head())

print("\nResumen final:")
print("cell_id únicos:", gdf_export["cell_id"].nunique())
print("ROW únicos:", gdf_export["ROW"].nunique())
print("COL únicos:", gdf_export["COL"].nunique())

Capas disponibles en el GPKG:
1. grid_3km_static
2. localidades_bogota
3. grid_3km_time_idw

Capa seleccionada: grid_3km_static

Dimensión: (254, 3)
Columnas disponibles: ['cell_id', 'LocNombre', 'geometry']

Agrupación espacial:
Número de columnas detectadas: 18
Número de filas detectadas: 42

Validaciones:
Número de celdas: 254
cell_id únicos: 254
ROW únicos: 42
COL únicos: 18
Duplicados ROW-COL: 5

Duplicados detectados:
     cell_id  ROW  COL
252      729    9   18
253      730    9   18
195      546   28   14
196      547   28   14
169      503   30   13
170      504   30   13
15       126   39    4
26       166   39    4
1          1   41    1
2          2   41    1


ValueError: Todavía hay duplicados en ROW y COL incluso usando la capa estática. En ese caso tocaría inspeccionar directamente la geometría fuente.

Esta celda construye una fishnet regular de 3 km sobre el dominio espacial de la grilla y utiliza un punto representativo de cada celda real para asignar de forma inequívoca los índices \texttt{ROW} y \texttt{COL}. A diferencia de los intentos anteriores, esta estrategia no depende de agrupar centroides por proximidad, sino de una malla regular explícita, lo que evita duplicados en la combinación fila-columna cuando el borde de Bogotá recorta parcialmente algunas celdas. El resultado es un nuevo shapefile listo para BenMAP-CE, conservando la geometría original de cada celda pero con los campos \texttt{ROW} y \texttt{COL} asignados de manera consistente.


In [8]:
# ============================================================
# CELDA 19E: Construir ROW/COL usando una fishnet regular 3 km
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.geometry import box

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

source_file = output_dir / "benmap_grid_definition.shp"
final_shp = output_dir / "benmap_grid_definition_benmap.shp"
fishnet_debug_file = output_dir / "benmap_grid_fishnet_debug.shp"

if not source_file.exists():
    raise FileNotFoundError(f"No se encontró el archivo fuente:\n{source_file}")

# ------------------------------------------------------------
# 2) Cargar grilla limpia
# ------------------------------------------------------------
gdf = gpd.read_file(source_file)

if gdf.empty:
    raise ValueError("La capa fuente está vacía.")

print("Archivo fuente cargado:", source_file)
print("Dimensión:", gdf.shape)
print("Columnas disponibles:", gdf.columns.tolist())

# ------------------------------------------------------------
# 3) Identificar cell_id
# ------------------------------------------------------------
possible_id_cols = ["cell_id", "cellid", "cell_idx", "id", "grid_id", "CellID"]
id_col = None
for c in possible_id_cols:
    if c in gdf.columns:
        id_col = c
        break

if id_col is None:
    raise ValueError("No se encontró una columna tipo cell_id en la grilla limpia.")

gdf = gdf.copy()
gdf["cell_id"] = pd.to_numeric(gdf[id_col], errors="coerce")
gdf = gdf.dropna(subset=["cell_id"]).copy()
gdf["cell_id"] = gdf["cell_id"].astype(int)
gdf = gdf.drop_duplicates(subset=["cell_id"]).copy()

# ------------------------------------------------------------
# 4) Pasar a CRS métrico
# ------------------------------------------------------------
if gdf.crs is None:
    raise ValueError("La grilla no tiene CRS definido.")

gdf_metric = gdf.copy()
if gdf_metric.crs.is_geographic:
    gdf_metric = gdf_metric.to_crs(epsg=3116)

# ------------------------------------------------------------
# 5) Crear puntos representativos (siempre quedan dentro)
# ------------------------------------------------------------
rep_points = gdf_metric.copy()
rep_points["geometry"] = gdf_metric.representative_point()

# ------------------------------------------------------------
# 6) Definir fishnet regular de 3 km
# ------------------------------------------------------------
step = 3000.0

minx, miny, maxx, maxy = rep_points.total_bounds

# Ajustar la extensión al múltiplo de 3000 más cercano hacia afuera
origin_x = np.floor(minx / step) * step
origin_y = np.floor(miny / step) * step
maxx_snap = np.ceil(maxx / step) * step
maxy_snap = np.ceil(maxy / step) * step

n_cols = int(round((maxx_snap - origin_x) / step))
n_rows = int(round((maxy_snap - origin_y) / step))

print("\nFishnet propuesta:")
print("origin_x =", origin_x)
print("origin_y =", origin_y)
print("maxx_snap =", maxx_snap)
print("maxy_snap =", maxy_snap)
print("n_cols =", n_cols)
print("n_rows =", n_rows)

# ------------------------------------------------------------
# 7) Construir fishnet
#    ROW = 1 arriba, aumenta hacia abajo
#    COL = 1 izquierda, aumenta hacia derecha
# ------------------------------------------------------------
fishnet_records = []

for row in range(1, n_rows + 1):
    for col in range(1, n_cols + 1):
        cell_minx = origin_x + (col - 1) * step
        cell_maxx = cell_minx + step

        cell_maxy = maxy_snap - (row - 1) * step
        cell_miny = cell_maxy - step

        fishnet_records.append({
            "ROW": row,
            "COL": col,
            "geometry": box(cell_minx, cell_miny, cell_maxx, cell_maxy)
        })

fishnet = gpd.GeoDataFrame(fishnet_records, geometry="geometry", crs=gdf_metric.crs)

# ------------------------------------------------------------
# 8) Asignar ROW/COL a cada punto representativo
# ------------------------------------------------------------
# Spatial join: cada punto debe caer en una celda de la fishnet
joined = gpd.sjoin(
    rep_points[["cell_id", "geometry"]],
    fishnet[["ROW", "COL", "geometry"]],
    how="left",
    predicate="within"
)

if joined["ROW"].isna().any() or joined["COL"].isna().any():
    missing = joined[joined["ROW"].isna() | joined["COL"].isna()][["cell_id"]]
    print("\nCeldas sin asignación ROW/COL:")
    print(missing.head(20))
    raise ValueError(
        "Algunas celdas no pudieron asignarse a la fishnet regular."
    )

# ------------------------------------------------------------
# 9) Validar duplicados
# ------------------------------------------------------------
dup_pairs = joined.duplicated(subset=["ROW", "COL"]).sum()

print("\nValidaciones:")
print("Número de celdas:", len(joined))
print("cell_id únicos:", joined["cell_id"].nunique())
print("ROW únicos:", joined["ROW"].nunique())
print("COL únicos:", joined["COL"].nunique())
print("Duplicados ROW-COL:", dup_pairs)

if dup_pairs > 0:
    dup_df = joined[joined.duplicated(subset=["ROW", "COL"], keep=False)][
        ["cell_id", "ROW", "COL"]
    ].sort_values(["ROW", "COL"])
    print("\nDuplicados detectados:")
    print(dup_df.head(30))
    raise ValueError(
        "Todavía hay duplicados en ROW y COL usando la fishnet regular."
    )

# ------------------------------------------------------------
# 10) Unir ROW/COL a la geometría original
# ------------------------------------------------------------
rowcol = joined[["cell_id", "ROW", "COL"]].copy()

gdf_export = gdf_metric.merge(rowcol, on="cell_id", how="left")

if gdf_export["ROW"].isna().any() or gdf_export["COL"].isna().any():
    raise ValueError("No todas las celdas recibieron ROW/COL después del merge.")

gdf_export["ROW"] = gdf_export["ROW"].astype(int)
gdf_export["COL"] = gdf_export["COL"].astype(int)

# Volver al CRS original para exportar
gdf_export = gdf_export.to_crs(gdf.crs)

# Conservar solo campos clave
gdf_export = gdf_export[["cell_id", "ROW", "COL", "geometry"]].copy()

# ------------------------------------------------------------
# 11) Exportar
# ------------------------------------------------------------
gdf_export.to_file(final_shp)
fishnet.to_file(fishnet_debug_file)

# ------------------------------------------------------------
# 12) Resumen final
# ------------------------------------------------------------
print("\nShapefile final exportado:")
print(final_shp)

print("\nFishnet debug exportada:")
print(fishnet_debug_file)

print("\nVista previa:")
print(gdf_export.head())

print("\nResumen final:")
print("cell_id únicos:", gdf_export["cell_id"].nunique())
print("ROW únicos:", gdf_export["ROW"].nunique())
print("COL únicos:", gdf_export["COL"].nunique())

Archivo fuente cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition.shp
Dimensión: (254, 2)
Columnas disponibles: ['cell_id', 'geometry']

Fishnet propuesta:
origin_x = 960000.0
origin_y = 903000.0
maxx_snap = 1011000.0
maxy_snap = 1026000.0
n_cols = 17
n_rows = 41

Validaciones:
Número de celdas: 254
cell_id únicos: 254
ROW únicos: 41
COL únicos: 17
Duplicados ROW-COL: 66

Duplicados detectados:
     cell_id  ROW  COL
227      614    1   15
240      655    1   15
238      653    2   16
239      654    2   16
249      694    2   16
250      695    2   16
237      652    4   16
248      693    4   16
236      651    5   16
247      692    5   16
234      649    6   16
235      650    6   16
245      690    6   16
246      691    6   16
88       360    8    9
89       361    8    9
111      401    8   10
112      402    8   10
233      648    8   16
244      689    8   16
252      729    8   17
253      730    8   17
135      441    9   11
136      442    9   11
232

ValueError: Todavía hay duplicados en ROW y COL usando la fishnet regular.

Esta celda reconstruye los campos \texttt{ROW} y \texttt{COL} de la grilla de BenMAP-CE directamente a partir del identificador \texttt{cell_id}, aprovechando la lógica original con la que se generó la malla 3 km × 3 km antes del recorte a Bogotá. Como el \texttt{cell_id} fue asignado secuencialmente sobre la grilla completa recorriendo primero las columnas y luego las filas, no es necesario estimar la posición espacial mediante centroides ni fishnets auxiliares. El código vuelve a calcular el número total de filas de la grilla original, recupera los índices \texttt{ROW}/\texttt{COL} de cada celda y exporta un shapefile final listo para cargar en BenMAP-CE como \textit{Shapefile Grid}.

In [9]:
# ============================================================
# CELDA 19F: Reconstruir ROW/COL directamente desde cell_id
# según la lógica original de creación de la malla 3 km
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

grid_file = output_dir / "benmap_grid_definition.shp"
final_shp = output_dir / "benmap_grid_definition_final_ok.shp"
debug_csv = output_dir / "benmap_grid_definition_final_ok_debug.csv"

if not grid_file.exists():
    raise FileNotFoundError(f"No se encontró la grilla limpia:\n{grid_file}")

# Buscar shapefile de localidades usado para generar Bogotá
search_roots = [
    project_root / "GRILLA-GEOVISOR MAPAS BEN-MAP",
    project_root / "Qgis",
    project_root
]

loc_candidates = []
for root in search_roots:
    if root.exists():
        loc_candidates.extend(root.rglob("Loca_WGS1984.shp"))
        loc_candidates.extend(root.rglob("Loca.shp"))

loc_candidates = list(dict.fromkeys(loc_candidates))

if not loc_candidates:
    raise FileNotFoundError(
        "No encontré el shapefile de localidades (Loca_WGS1984.shp o Loca.shp)."
    )

loc_file = loc_candidates[0]

# ------------------------------------------------------------
# 2) Cargar grilla limpia
# ------------------------------------------------------------
gdf_grid = gpd.read_file(grid_file)

if gdf_grid.empty:
    raise ValueError("La grilla limpia está vacía.")

print("Grilla limpia cargada:", grid_file)
print("Dimensión:", gdf_grid.shape)
print("Columnas:", gdf_grid.columns.tolist())

# Identificar cell_id
possible_id_cols = ["cell_id", "cellid", "cell_idx", "id", "grid_id", "CellID"]
id_col = None
for c in possible_id_cols:
    if c in gdf_grid.columns:
        id_col = c
        break

if id_col is None:
    raise ValueError("No se encontró una columna tipo cell_id en la grilla limpia.")

gdf_grid = gdf_grid.copy()
gdf_grid["cell_id"] = pd.to_numeric(gdf_grid[id_col], errors="coerce")
gdf_grid = gdf_grid.dropna(subset=["cell_id"]).copy()
gdf_grid["cell_id"] = gdf_grid["cell_id"].astype(int)
gdf_grid = gdf_grid.drop_duplicates(subset=["cell_id"]).copy()

# ------------------------------------------------------------
# 3) Cargar localidades y reconstruir la grilla original
#    siguiendo la lógica con la que se creó cell_id
# ------------------------------------------------------------
gdf_loc = gpd.read_file(loc_file)

if gdf_loc.empty:
    raise ValueError("El shapefile de localidades está vacío.")

print("\nLocalidades cargadas:", loc_file)
print("CRS localidades original:", gdf_loc.crs)

gdf_loc_m = gdf_loc.to_crs(epsg=3116)
bogota = gdf_loc_m.dissolve().reset_index(drop=True)

CELL = 3000.0
minx, miny, maxx, maxy = bogota.total_bounds

xs = np.arange(minx, maxx, CELL)
ys = np.arange(miny, maxy, CELL)

n_cols = len(xs)
n_rows = len(ys)

print("\nReconstrucción de grilla original:")
print("n_cols =", n_cols)
print("n_rows =", n_rows)
print("n_total_bbox =", n_cols * n_rows)
print("cell_id mínimo =", gdf_grid["cell_id"].min())
print("cell_id máximo =", gdf_grid["cell_id"].max())

if gdf_grid["cell_id"].max() >= n_cols * n_rows:
    raise ValueError(
        "Hay cell_id fuera del rango esperado según la grilla original."
    )

# ------------------------------------------------------------
# 4) Recuperar COL y ROW desde cell_id
#    Lógica original:
#    for x in xs:
#        for y in ys:
#            cell_id = k
#
#    => x_index = cell_id // n_rows
#    => y_index = cell_id % n_rows
#
#    Convención BenMAP compatible:
#    COL: izquierda -> derecha, inicia en 1
#    ROW: abajo -> arriba, inicia en 1
# ------------------------------------------------------------
gdf_grid["COL"] = (gdf_grid["cell_id"] // n_rows) + 1
gdf_grid["ROW"] = (gdf_grid["cell_id"] % n_rows) + 1

gdf_grid["COL"] = gdf_grid["COL"].astype(int)
gdf_grid["ROW"] = gdf_grid["ROW"].astype(int)

# ------------------------------------------------------------
# 5) Validaciones
# ------------------------------------------------------------
dup_pairs = gdf_grid.duplicated(subset=["ROW", "COL"]).sum()

print("\nValidaciones:")
print("Número de celdas:", len(gdf_grid))
print("cell_id únicos:", gdf_grid["cell_id"].nunique())
print("ROW únicos:", gdf_grid["ROW"].nunique())
print("COL únicos:", gdf_grid["COL"].nunique())
print("Duplicados ROW-COL:", dup_pairs)

if dup_pairs > 0:
    dup_df = gdf_grid[gdf_grid.duplicated(subset=["ROW", "COL"], keep=False)][
        ["cell_id", "ROW", "COL"]
    ].sort_values(["ROW", "COL"])
    print("\nDuplicados detectados:")
    print(dup_df.head(30))
    raise ValueError(
        "Todavía hay duplicados ROW/COL incluso reconstruyéndolos desde cell_id."
    )

# ------------------------------------------------------------
# 6) Exportar shapefile final
# ------------------------------------------------------------
gdf_export = gdf_grid[["cell_id", "ROW", "COL", "geometry"]].copy()
gdf_export.to_file(final_shp)

debug_df = gdf_export.drop(columns="geometry").sort_values("cell_id").reset_index(drop=True)
debug_df.to_csv(debug_csv, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 7) Resumen final
# ------------------------------------------------------------
print("\nShapefile final exportado:")
print(final_shp)

print("\nCSV de depuración exportado:")
print(debug_csv)

print("\nVista previa:")
print(debug_df.head())

print("\nResumen final:")
print("cell_id únicos:", gdf_export["cell_id"].nunique())
print("ROW únicos:", gdf_export["ROW"].nunique())
print("COL únicos:", gdf_export["COL"].nunique())

Grilla limpia cargada: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition.shp
Dimensión: (254, 2)
Columnas: ['cell_id', 'geometry']

Localidades cargadas: d:\TRABAJO DE GRADO BEN-MAP\GRILLA-GEOVISOR MAPAS BEN-MAP\bogota\Loca_WGS1984.shp
CRS localidades original: EPSG:4326

Reconstrucción de grilla original:
n_cols = 18
n_rows = 41
n_total_bbox = 738
cell_id mínimo = 0
cell_id máximo = 730

Validaciones:
Número de celdas: 254
cell_id únicos: 254
ROW únicos: 41
COL únicos: 18
Duplicados ROW-COL: 0

Shapefile final exportado:
d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_final_ok.shp

CSV de depuración exportado:
d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_final_ok_debug.csv

Vista previa:
   cell_id  ROW  COL
0        0    1    1
1        1    2    1
2        2    3    1
3       41    1    2
4       42    2    2

Resumen final:
cell_id únicos: 254
ROW únicos: 41
COL únicos: 18


# incidencia benmap

In [10]:
# ============================================================
# CELDA 20: Convertir incidencia al formato BenMAP READY_ONLY
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

incidence_file = output_dir / "benmap_incidence_2026.csv"
grid_debug_file = output_dir / "benmap_grid_definition_final_ok_debug.csv"
out_file = output_dir / "benmap_incidence_ready_only.csv"

for f in [incidence_file, grid_debug_file]:
    if not f.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido:\n{f}")

# ------------------------------------------------------------
# 2) Cargar insumos
# ------------------------------------------------------------
inc = pd.read_csv(incidence_file)
grid = pd.read_csv(grid_debug_file)

print("Archivo incidencia cargado:", incidence_file)
print("Dimensión incidencia:", inc.shape)

print("\nArchivo grilla debug cargado:", grid_debug_file)
print("Dimensión grilla:", grid.shape)

# ------------------------------------------------------------
# 3) Verificar columnas mínimas
# ------------------------------------------------------------
required_inc_cols = ["cell_id", "endpoint_code", "incidence_rate"]
required_grid_cols = ["cell_id", "ROW", "COL"]

missing_inc = [c for c in required_inc_cols if c not in inc.columns]
missing_grid = [c for c in required_grid_cols if c not in grid.columns]

if missing_inc:
    raise ValueError(f"Faltan columnas en incidencia: {missing_inc}")
if missing_grid:
    raise ValueError(f"Faltan columnas en la grilla debug: {missing_grid}")

# ------------------------------------------------------------
# 4) Quedarnos solo con los endpoints READY_ONLY
#    def_I -> Mortality / All-cause mortality / 65-99
#    def_J -> Mortality / Respiratory mortality / 0-99
# ------------------------------------------------------------
endpoint_map = {
    "def_I": {
        "Endpoint Group": "Mortality",
        "Endpoint": "All-cause mortality",
        "Start Age": 65,
        "End Age": 99
    },
    "def_J": {
        "Endpoint Group": "Mortality",
        "Endpoint": "Respiratory mortality",
        "Start Age": 0,
        "End Age": 99
    }
}

inc = inc[inc["endpoint_code"].isin(endpoint_map.keys())].copy()

# ------------------------------------------------------------
# 5) Unir con ROW/COL
# ------------------------------------------------------------
grid = grid.rename(columns={"ROW": "Row", "COL": "Column"})
grid = grid[["cell_id", "Row", "Column"]].copy()

df = inc.merge(grid, on="cell_id", how="left")

if df["Row"].isna().any() or df["Column"].isna().any():
    missing_cells = df[df["Row"].isna() | df["Column"].isna()]["cell_id"].unique().tolist()
    raise ValueError(f"Hay celdas sin Row/Column asignados: {missing_cells[:10]}")

# ------------------------------------------------------------
# 6) Traducir endpoints al formato BenMAP
# ------------------------------------------------------------
df["Endpoint Group"] = df["endpoint_code"].map(lambda x: endpoint_map[x]["Endpoint Group"])
df["Endpoint"] = df["endpoint_code"].map(lambda x: endpoint_map[x]["Endpoint"])
df["Start Age"] = df["endpoint_code"].map(lambda x: endpoint_map[x]["Start Age"])
df["End Age"] = df["endpoint_code"].map(lambda x: endpoint_map[x]["End Age"])

# Campos opcionales: vacíos = "todos"
df["Race"] = ""
df["Ethnicity"] = ""
df["Gender"] = ""
df["Type"] = ""   # vacío => BenMAP lo interpreta como incidencia

# Valor final
df["Value"] = pd.to_numeric(df["incidence_rate"], errors="coerce")

if df["Value"].isna().any():
    raise ValueError("Hay incidence_rate que no pudo convertirse a numérico.")

# ------------------------------------------------------------
# 7) Seleccionar columnas finales en el orden BenMAP
# ------------------------------------------------------------
df_out = df[[
    "Endpoint Group",
    "Endpoint",
    "Race",
    "Ethnicity",
    "Gender",
    "Start Age",
    "End Age",
    "Column",
    "Row",
    "Value",
    "Type"
]].copy()

# Ordenar para que quede limpio
df_out = df_out.sort_values(["Endpoint Group", "Endpoint", "Row", "Column"]).reset_index(drop=True)

# ------------------------------------------------------------
# 8) Exportar
# ------------------------------------------------------------
df_out.to_csv(out_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9) Resumen
# ------------------------------------------------------------
print("\nResumen del archivo listo para BenMAP:")
print(df_out.groupby(["Endpoint Group", "Endpoint"], as_index=False).agg(
    n_rows=("Value", "count"),
    min_value=("Value", "min"),
    max_value=("Value", "max")
))

print("\nVista previa:")
print(df_out.head())

print("\nArchivo generado:")
print(" -", out_file)

Archivo incidencia cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_incidence_2026.csv
Dimensión incidencia: (508, 9)

Archivo grilla debug cargado: d:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_final_ok_debug.csv
Dimensión grilla: (254, 3)

Resumen del archivo listo para BenMAP:
  Endpoint Group               Endpoint  n_rows  min_value  max_value
0      Mortality    All-cause mortality     254   0.000064   0.000064
1      Mortality  Respiratory mortality     254   0.000132   0.000132

Vista previa:
  Endpoint Group             Endpoint Race Ethnicity Gender  Start Age  \
0      Mortality  All-cause mortality                               65   
1      Mortality  All-cause mortality                               65   
2      Mortality  All-cause mortality                               65   
3      Mortality  All-cause mortality                               65   
4      Mortality  All-cause mortality                               65   

   End Age  Column 

### Conversión completa de la población 2026 al formato requerido por BenMAP-CE

Esta celda localiza automáticamente, dentro del proyecto, el archivo de población proyectada para 2026 y el archivo de definición de grilla asociado a BenMAP. Luego transforma la población desde el formato ancho generado previamente (`cell_id`, `year`, `age_group`, `male`, `female`, `total`) al formato largo requerido por BenMAP-CE (`Row`, `Column`, `Year`, `Population`, `Race`, `Ethnicity`, `Gender`, `AgeRange`). Además, estandariza los grupos etarios para que coincidan exactamente con la configuración creada en BenMAP (`0-4`, `5-14`, `15-44`, `45-64`, `65+`), valida consistencia básica de filas y exporta un archivo final listo para cargar y validar desde la interfaz del programa.

In [14]:
# ============================================================
# CELDA ÚNICA CORREGIDA:
# convertir población 2026 al formato largo BenMAP-CE
# usando la grilla debug con ROW/COL
# ============================================================

from pathlib import Path
import pandas as pd
import re

# ------------------------------------------------------------
# 0) Detectar raíz del proyecto
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent

print("Directorio actual :", current_dir)
print("Raíz del proyecto :", project_root)

output_dir = project_root / "SALIDAS_BENMAP"
output_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1) Buscar archivo de población
# ------------------------------------------------------------
population_candidates = [
    output_dir / "benmap_population_2026.csv",
    output_dir / "population_bogota_rangos_2026.csv",
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

pop_file = first_existing(population_candidates)

if pop_file is None:
    recursive_pop = list(project_root.rglob("benmap_population_2026.csv"))
    if recursive_pop:
        pop_file = recursive_pop[0]

if pop_file is None:
    recursive_pop_alt = list(project_root.rglob("population_bogota_rangos_2026.csv"))
    if recursive_pop_alt:
        pop_file = recursive_pop_alt[0]

if pop_file is None:
    raise FileNotFoundError(
        "No encontré ni 'benmap_population_2026.csv' ni "
        "'population_bogota_rangos_2026.csv' dentro del proyecto."
    )

# ------------------------------------------------------------
# 2) Buscar archivo de grilla CORRECTO para BenMAP
#    IMPORTANTE: aquí priorizamos el debug porque sí tiene ROW/COL
# ------------------------------------------------------------
grid_candidates = [
    output_dir / "benmap_grid_definition_final_ok_debug.csv",
    output_dir / "benmap_grid_definition_summary.csv",
]

grid_file = first_existing(grid_candidates)

if grid_file is None:
    recursive_grid_1 = list(project_root.rglob("benmap_grid_definition_final_ok_debug.csv"))
    if recursive_grid_1:
        grid_file = recursive_grid_1[0]

if grid_file is None:
    recursive_grid_2 = list(project_root.rglob("benmap_grid_definition_summary.csv"))
    if recursive_grid_2:
        grid_file = recursive_grid_2[0]

if grid_file is None:
    raise FileNotFoundError(
        "No encontré ni 'benmap_grid_definition_final_ok_debug.csv' ni "
        "'benmap_grid_definition_summary.csv' dentro del proyecto."
    )

print("\nArchivo de población encontrado:")
print(pop_file)

print("\nArchivo de grilla encontrado:")
print(grid_file)

# ------------------------------------------------------------
# 3) Cargar archivos
# ------------------------------------------------------------
pop = pd.read_csv(pop_file)
grid = pd.read_csv(grid_file)

print("\nColumnas población:", pop.columns.tolist())
print("Dimensión población:", pop.shape)

print("\nColumnas grilla:", grid.columns.tolist())
print("Dimensión grilla:", grid.shape)

# ------------------------------------------------------------
# 4) Limpiar nombres de columnas
# ------------------------------------------------------------
pop.columns = [c.strip() for c in pop.columns]
grid.columns = [c.strip() for c in grid.columns]

# ------------------------------------------------------------
# 5) Normalizar columnas de grilla
#    Acepta ROW/COL o Row/Column
# ------------------------------------------------------------
grid_rename = {}
for c in grid.columns:
    cl = c.strip().lower()
    if cl == "row":
        grid_rename[c] = "Row"
    elif cl == "column":
        grid_rename[c] = "Column"
    elif cl == "col":
        grid_rename[c] = "Column"
    elif cl == "cell_id":
        grid_rename[c] = "cell_id"

grid = grid.rename(columns=grid_rename)

# ------------------------------------------------------------
# 6) Validar que la grilla sí tenga Row/Column
# ------------------------------------------------------------
required_grid = {"cell_id", "Row", "Column"}
missing_grid = required_grid - set(grid.columns)

if missing_grid:
    raise ValueError(
        f"La grilla seleccionada no sirve para BenMAP porque faltan columnas: {sorted(missing_grid)}\n"
        f"Archivo usado: {grid_file}"
    )

# ------------------------------------------------------------
# 7) Detectar formato de población
# ------------------------------------------------------------
pop_cols = set(pop.columns)
wide_required = {"cell_id", "year", "age_group", "male", "female"}
long_required = {"Row", "Column", "Year", "Population", "Race", "Ethnicity", "Gender", "AgeRange"}

# ------------------------------------------------------------
# 8) Si ya está en formato largo, solo estandarizar
# ------------------------------------------------------------
if long_required.issubset(pop_cols):
    print("\nEl archivo de población ya está en formato largo BenMAP. Se estandarizará y reexportará.")

    benmap_pop = pop.copy()

    benmap_pop["AgeRange"] = benmap_pop["AgeRange"].astype(str).str.strip()
    benmap_pop["Gender"] = benmap_pop["Gender"].astype(str).str.strip().str.upper()
    benmap_pop["Race"] = benmap_pop["Race"].astype(str).str.strip().str.upper()
    benmap_pop["Ethnicity"] = benmap_pop["Ethnicity"].astype(str).str.strip().str.upper()

    benmap_pop["Row"] = pd.to_numeric(benmap_pop["Row"], errors="coerce").astype("Int64")
    benmap_pop["Column"] = pd.to_numeric(benmap_pop["Column"], errors="coerce").astype("Int64")
    benmap_pop["Year"] = pd.to_numeric(benmap_pop["Year"], errors="coerce").astype("Int64")
    benmap_pop["Population"] = pd.to_numeric(benmap_pop["Population"], errors="coerce").fillna(0).round(0).astype(int)

# ------------------------------------------------------------
# 9) Si está en formato ancho, convertirlo
# ------------------------------------------------------------
elif wide_required.issubset(pop_cols):
    print("\nEl archivo de población está en formato ancho. Se convertirá al formato largo BenMAP.")

    def normalize_age_group(value: str) -> str:
        s = str(value).strip().lower()
        s = s.replace("años", "").replace("año", "")
        s = s.replace("years", "")
        s = s.replace(" ", "")
        s = s.replace("–", "-").replace("—", "-").replace("_", "-")

        mapping = {
            "0-4": "0-4",
            "0a4": "0-4",
            "5-14": "5-14",
            "5a14": "5-14",
            "15-44": "15-44",
            "15a44": "15-44",
            "45-64": "45-64",
            "45a64": "45-64",
            "65+": "65+",
            "65ymas": "65+",
            "65ymás": "65+",
            "65-99": "65+",
        }

        if s in mapping:
            return mapping[s]

        m = re.match(r"^(\d+)-(\d+)$", s)
        if m:
            a, b = int(m.group(1)), int(m.group(2))
            if (a, b) == (0, 4):
                return "0-4"
            if (a, b) == (5, 14):
                return "5-14"
            if (a, b) == (15, 44):
                return "15-44"
            if (a, b) == (45, 64):
                return "45-64"
            if a >= 65:
                return "65+"

        raise ValueError(f"No pude reconocer el grupo etario: {value!r}")

    pop = pop.copy()
    pop["AgeRange"] = pop["age_group"].apply(normalize_age_group)

    long_pop = pop.melt(
        id_vars=["cell_id", "year", "AgeRange"],
        value_vars=["male", "female"],
        var_name="Gender",
        value_name="Population"
    )

    long_pop["Gender"] = long_pop["Gender"].map({
        "male": "MALE",
        "female": "FEMALE"
    })

    long_pop["Race"] = "ALL"
    long_pop["Ethnicity"] = "ALL"

    # usar la grilla debug/final con Row/Column
    grid_map = grid[["cell_id", "Row", "Column"]].copy()
    grid_map["cell_id"] = pd.to_numeric(grid_map["cell_id"], errors="coerce").astype("Int64")
    grid_map = grid_map.dropna(subset=["cell_id", "Row", "Column"]).copy()
    grid_map["cell_id"] = grid_map["cell_id"].astype(int)

    # quitar duplicados por cell_id
    grid_map = grid_map.drop_duplicates(subset=["cell_id"]).copy()

    long_pop["cell_id"] = pd.to_numeric(long_pop["cell_id"], errors="coerce").astype("Int64")
    long_pop = long_pop.dropna(subset=["cell_id"]).copy()
    long_pop["cell_id"] = long_pop["cell_id"].astype(int)

    long_pop = long_pop.merge(grid_map, on="cell_id", how="left")

    if long_pop[["Row", "Column"]].isna().any().any():
        bad = long_pop.loc[
            long_pop["Row"].isna() | long_pop["Column"].isna(),
            "cell_id"
        ].unique().tolist()
        raise ValueError(
            f"Hay cell_id sin correspondencia en la grilla debug. Primeros ejemplos: {bad[:10]}"
        )

    long_pop["Population"] = pd.to_numeric(long_pop["Population"], errors="coerce").fillna(0)
    long_pop["Population"] = long_pop["Population"].round(0).astype(int)

    if (long_pop["Population"] < 0).any():
        raise ValueError("Se encontraron valores negativos en Population.")

    benmap_pop = long_pop[[
        "Row", "Column", "year", "Population",
        "Race", "Ethnicity", "Gender", "AgeRange"
    ]].rename(columns={"year": "Year"}).copy()

else:
    raise ValueError(
        "El archivo de población no tiene ni el formato ancho esperado "
        "(cell_id, year, age_group, male, female) ni el formato largo BenMAP."
    )

# ------------------------------------------------------------
# 10) Estandarización final
# ------------------------------------------------------------
benmap_pop["AgeRange"] = benmap_pop["AgeRange"].astype(str).str.strip()
benmap_pop["Gender"] = benmap_pop["Gender"].astype(str).str.strip().str.upper()
benmap_pop["Race"] = benmap_pop["Race"].astype(str).str.strip().str.upper()
benmap_pop["Ethnicity"] = benmap_pop["Ethnicity"].astype(str).str.strip().str.upper()

benmap_pop["Row"] = pd.to_numeric(benmap_pop["Row"], errors="coerce").astype("Int64")
benmap_pop["Column"] = pd.to_numeric(benmap_pop["Column"], errors="coerce").astype("Int64")
benmap_pop["Year"] = pd.to_numeric(benmap_pop["Year"], errors="coerce").astype("Int64")
benmap_pop["Population"] = pd.to_numeric(benmap_pop["Population"], errors="coerce").fillna(0).round(0).astype(int)

# ------------------------------------------------------------
# 11) Orden final
# ------------------------------------------------------------
benmap_pop = benmap_pop[[
    "Row", "Column", "Year", "Population",
    "Race", "Ethnicity", "Gender", "AgeRange"
]].sort_values(
    by=["Row", "Column", "Year", "AgeRange", "Gender"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 12) Validaciones básicas
# ------------------------------------------------------------
print("\nResumen del archivo final:")
print(benmap_pop.head())

print("\nFilas finales:", len(benmap_pop))
print("Celdas únicas:", benmap_pop[['Row', 'Column']].drop_duplicates().shape[0])
print("Años:", benmap_pop["Year"].drop_duplicates().tolist())
print("AgeRange:", benmap_pop["AgeRange"].drop_duplicates().tolist())
print("Gender:", benmap_pop["Gender"].drop_duplicates().tolist())
print("Race:", benmap_pop["Race"].drop_duplicates().tolist())
print("Ethnicity:", benmap_pop["Ethnicity"].drop_duplicates().tolist())
print("Población total:", int(benmap_pop["Population"].sum()))

if benmap_pop["Year"].nunique() == 1 and int(benmap_pop["Year"].iloc[0]) == 2026:
    expected_rows = 254 * 5 * 2
    print("Filas esperadas:", expected_rows)
    if len(benmap_pop) == expected_rows:
        print("Validación básica OK.")
    else:
        print("ADVERTENCIA: el número de filas no coincide con 254*5*2.")

# ------------------------------------------------------------
# 13) Exportar
# ------------------------------------------------------------
output_file = output_dir / "benmap_population_2026_long_ready.csv"
benmap_pop.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\nArchivo exportado:")
print(output_file)

Directorio actual : D:\TRABAJO DE GRADO BEN-MAP\CODIGO
Raíz del proyecto : D:\TRABAJO DE GRADO BEN-MAP

Archivo de población encontrado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_population_2026.csv

Archivo de grilla encontrado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_final_ok_debug.csv

Columnas población: ['cell_id', 'year', 'age_group', 'male', 'female', 'total', 'area_weight', 'allocation_method']
Dimensión población: (1270, 8)

Columnas grilla: ['cell_id', 'ROW', 'COL']
Dimensión grilla: (254, 3)

El archivo de población está en formato ancho. Se convertirá al formato largo BenMAP.

Resumen del archivo final:
   Row  Column  Year  Population Race Ethnicity  Gender AgeRange
0    1       1  2026        1706  ALL       ALL  FEMALE      0-4
1    1       1  2026        1798  ALL       ALL    MALE      0-4
2    1       1  2026       16303  ALL       ALL  FEMALE    15-44
3    1       1  2026       16426  ALL       ALL    MALE    15-44
4    1       1  20

### Reconstrucción del archivo de Health Impact Functions al formato exacto de importación de BenMAP-CE

Esta celda toma el archivo de trabajo de funciones concentración–respuesta y lo transforma al esquema de columnas que el importador de BenMAP-CE está validando en la sección de *Health Impact Functions*. El problema previo no provenía del contenido epidemiológico, sino de un desajuste entre los nombres de columnas del archivo exportado y los encabezados requeridos por BenMAP. Por ello, esta versión renombra campos como autor, año, referencia, beta y forma funcional, agrega las columnas obligatorias faltantes, conserva los registros listos para la corrida base y exporta un nuevo archivo de importación que podrá cargarse y validarse desde la interfaz.

In [15]:
# ============================================================
# CELDA: reconstruir Health Impact Functions para importación
# exacta en BenMAP-CE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

candidate_files = [
    output_dir / "benmap_health_impact_functions_ready_only_working.csv",
    output_dir / "benmap_health_impact_functions_working.csv",
    output_dir / "benmap_health_impact_functions_candidate_final_working.csv",
]

health_file = next((p for p in candidate_files if p.exists()), None)

if health_file is None:
    raise FileNotFoundError(
        "No encontré ningún archivo working de Health Impact Functions en SALIDAS_BENMAP."
    )

print("Archivo fuente encontrado:")
print(health_file)

df = pd.read_csv(health_file)
df.columns = [c.strip() for c in df.columns]

print("\nColumnas originales:")
print(df.columns.tolist())
print("Filas:", len(df))

# ------------------------------------------------------------
# 1) Si existe status, dejar solo READY_NUMERIC
# ------------------------------------------------------------
if "status" in df.columns:
    df = df[df["status"].astype(str).str.strip().eq("READY_NUMERIC")].copy().reset_index(drop=True)
    print("\nFilas READY_NUMERIC:", len(df))

if df.empty:
    raise ValueError("Después del filtro READY_NUMERIC, el archivo quedó vacío.")

# ------------------------------------------------------------
# 2) Función auxiliar para tomar la primera columna disponible
# ------------------------------------------------------------
def pick_col(frame, candidates, default=""):
    for c in candidates:
        if c in frame.columns:
            return frame[c]
    return pd.Series([default] * len(frame), index=frame.index)

# ------------------------------------------------------------
# 3) Construir dataframe con encabezados esperados por BenMAP
#    según la validación que te mostró la interfaz
# ------------------------------------------------------------
out = pd.DataFrame(index=df.index)

out["Endpoint Group"] = pick_col(df, ["Endpoint Group"], "")
out["Endpoint"] = pick_col(df, ["Endpoint"], "")
out["Pollutant"] = pick_col(df, ["Pollutant"], "")
out["Metric"] = pick_col(df, ["Metric"], "")
out["Seasonal Metric"] = pick_col(df, ["Seasonal Metric"], "")

# BenMAP te pidió Metric Statistic, no Annual Statistic
out["Metric Statistic"] = pick_col(
    df,
    ["Metric Statistic", "Annual Statistic", "annual_statistic"],
    ""
)

# BenMAP te pidió Study Year / Study Author
out["Study Year"] = pick_col(
    df,
    ["Study Year", "Year of Publication", "year_of_publication"],
    ""
)

out["Study Author"] = pick_col(
    df,
    ["Study Author", "Author", "author"],
    ""
)

# Ubicación / referencia / calificador
out["Study Location"] = pick_col(
    df,
    ["Study Location", "reference_short", "reference_full"],
    ""
)

out["Qualifier"] = pick_col(df, ["Qualifier", "qualifier"], "")
out["Reference"] = pick_col(df, ["Reference", "reference_short", "reference_full"], "")

# Demografía
out["Race"] = pick_col(df, ["Race"], "ALL")
out["Ethnicity"] = pick_col(df, ["Ethnicity"], "ALL")
out["Gender"] = pick_col(df, ["Gender"], "ALL")
out["Start Age"] = pick_col(df, ["Start Age"], 0)
out["End Age"] = pick_col(df, ["End Age"], 99)

# Ámbito geográfico: BenMAP te rechazó "Apply Function To"
# lo movemos a Geographic Area
out["Geographic Area"] = pick_col(
    df,
    ["Geographic Area", "Apply Function To", "apply_function_to"],
    "Entire Area"
)

# Datasets auxiliares
out["Incidence DataSet"] = pick_col(
    df,
    ["Incidence DataSet", "Incidence Dataset", "incidence_dataset"],
    "Bogota_incidence_ready_only"
)

out["Prevalence DataSet"] = pick_col(
    df,
    ["Prevalence DataSet", "Prevalence Dataset", "prevalence_dataset"],
    ""
)

out["Variable DataSet"] = pick_col(
    df,
    ["Variable DataSet", "variable_dataset"],
    ""
)

# Parámetros beta
out["Beta"] = pick_col(
    df,
    ["Beta", "beta", "beta_value", "beta_per_unit"],
    ""
)

out["Distribution Beta"] = pick_col(
    df,
    ["Distribution Beta", "beta_distribution"],
    ""
)

out["Parameter 1 Beta"] = pick_col(
    df,
    ["Parameter 1 Beta", "beta_param1"],
    ""
)

out["Parameter 2 Beta"] = pick_col(
    df,
    ["Parameter 2 Beta", "beta_param2"],
    ""
)

# Otros contaminantes
out["Other Pollutants"] = pick_col(
    df,
    ["Other Pollutants", "other_pollutants"],
    ""
)

# Función y baseline function
out["Function"] = pick_col(
    df,
    ["Function", "function_form"],
    ""
)

out["Baseline Function"] = pick_col(
    df,
    ["Baseline Function", "baseline_function"],
    ""
)

# Parámetros A / B / C
out["A"] = pick_col(df, ["A"], "")
out["Name A"] = pick_col(df, ["Name A", "NameA"], "")
out["B"] = pick_col(df, ["B"], "")
out["Name B"] = pick_col(df, ["Name B", "NameB"], "")
out["C"] = pick_col(df, ["C"], "")
out["Name C"] = pick_col(df, ["Name C", "NameC"], "")

# ------------------------------------------------------------
# 4) Limpieza de tipos y texto
# ------------------------------------------------------------
text_cols = [
    "Endpoint Group", "Endpoint", "Pollutant", "Metric", "Seasonal Metric",
    "Metric Statistic", "Study Author", "Study Location", "Qualifier",
    "Reference", "Race", "Ethnicity", "Gender", "Geographic Area",
    "Incidence DataSet", "Prevalence DataSet", "Variable DataSet",
    "Distribution Beta", "Other Pollutants", "Function", "Baseline Function",
    "Name A", "Name B", "Name C"
]

for c in text_cols:
    out[c] = out[c].fillna("").astype(str).str.strip()

numeric_cols = ["Study Year", "Start Age", "End Age", "Beta", "Parameter 1 Beta", "Parameter 2 Beta", "A", "B", "C"]

for c in numeric_cols:
    out[c] = pd.to_numeric(out[c], errors="coerce")

# Si A/B/C quedaron vacíos, los dejamos vacíos al exportar
# Study Year puede venir vacío si BenMAP no lo exige numéricamente en todos los casos
for c in ["Start Age", "End Age"]:
    out[c] = out[c].fillna(0 if c == "Start Age" else 99).astype(int)

# ------------------------------------------------------------
# 5) Reordenar columnas
# ------------------------------------------------------------
final_cols = [
    "Endpoint Group",
    "Endpoint",
    "Pollutant",
    "Metric",
    "Seasonal Metric",
    "Metric Statistic",
    "Study Year",
    "Study Author",
    "Study Location",
    "Reference",
    "Qualifier",
    "Race",
    "Ethnicity",
    "Gender",
    "Start Age",
    "End Age",
    "Geographic Area",
    "Incidence DataSet",
    "Prevalence DataSet",
    "Variable DataSet",
    "Beta",
    "Distribution Beta",
    "Parameter 1 Beta",
    "Parameter 2 Beta",
    "Other Pollutants",
    "Function",
    "Baseline Function",
    "A",
    "Name A",
    "B",
    "Name B",
    "C",
    "Name C"
]

out = out[final_cols].copy()

# ------------------------------------------------------------
# 6) Vista previa
# ------------------------------------------------------------
print("\nVista previa del archivo reconstruido:")
print(out.head())

print("\nColumnas finales:")
print(out.columns.tolist())

print("\nResumen:")
print("Filas:", len(out))
print("Endpoint Group:", out["Endpoint Group"].drop_duplicates().tolist())
print("Endpoint:", out["Endpoint"].drop_duplicates().tolist())
print("Pollutant:", out["Pollutant"].drop_duplicates().tolist())

# ------------------------------------------------------------
# 7) Exportar
# ------------------------------------------------------------
export_file = output_dir / "benmap_health_impact_functions_import_full.csv"
out.to_csv(export_file, index=False, encoding="utf-8-sig")

print("\nArchivo exportado:")
print(export_file)

Archivo fuente encontrado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_ready_only_working.csv

Columnas originales:
['Endpoint Group', 'Endpoint', 'Pollutant', 'Metric', 'Annual Statistic', 'Seasonal Metric', 'Race', 'Ethnicity', 'Gender', 'Start Age', 'End Age', 'Author', 'Apply Function To', 'Year of Publication', 'Qualifier', 'endpoint_code_local', 'reference_short', 'reference_full', 'effect_type', 'beta_per_unit', 'beta_se', 'unit_change', 'concentration_unit', 'function_form', 'status', 'notes']
Filas: 2

Filas READY_NUMERIC: 2

Vista previa del archivo reconstruido:
  Endpoint Group               Endpoint Pollutant              Metric  \
0      Mortality    All-cause mortality      PM25  Annual (D8HourMax)   
1      Mortality  Respiratory mortality        O3           D8HourMax   

  Seasonal Metric Metric Statistic  Study Year        Study Author  \
0                                         2017           Di et al.   
1                             

### Corrección final del archivo de Health Impact Functions según la plantilla nativa de BenMAP

Esta celda ajusta los últimos valores que la validación de BenMAP-CE reportó como inconsistentes. En particular, completa `Metric Statistic` con el valor `None`, corrige la métrica de PM2.5 a `AnnualMean` y fuerza para ozono la combinación `O3` + `D8HourMax`, que fue la definida previamente en el setup de contaminantes. Después de estas correcciones, se exporta una nueva versión del archivo de importación para volver a validarla en BenMAP.

In [16]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

src = output_dir / "benmap_health_impact_functions_import_full.csv"
if not src.exists():
    raise FileNotFoundError(f"No se encontró:\n{src}")

df = pd.read_csv(src)
df.columns = [c.strip() for c in df.columns]

print("Columnas:", df.columns.tolist())
print("Filas:", len(df))

# ------------------------------------------------------------
# 2) Limpieza general de texto
# ------------------------------------------------------------
text_cols = df.select_dtypes(include="object").columns.tolist()
for c in text_cols:
    df[c] = df[c].fillna("").astype(str).str.strip()

# ------------------------------------------------------------
# 3) Corregir Metric Statistic
#    La plantilla sample de BenMAP usa 'None'
# ------------------------------------------------------------
if "Metric Statistic" not in df.columns:
    raise ValueError("No existe la columna 'Metric Statistic'.")

df["Metric Statistic"] = "None"

# ------------------------------------------------------------
# 4) Corregir filas según pollutant
# ------------------------------------------------------------
if "Pollutant" not in df.columns or "Metric" not in df.columns:
    raise ValueError("Faltan columnas Pollutant o Metric.")

# Normalizar pollutant
df["Pollutant"] = df["Pollutant"].str.upper().str.replace(" ", "", regex=False)

# PM25 -> AnnualMean
mask_pm25 = df["Pollutant"].eq("PM25")
df.loc[mask_pm25, "Metric"] = "AnnualMean"
df.loc[mask_pm25, "Seasonal Metric"] = ""

# O3 -> D8HourMax
mask_o3 = df["Pollutant"].eq("O3")
df.loc[mask_o3, "Metric"] = "D8HourMax"
df.loc[mask_o3, "Seasonal Metric"] = ""

# ------------------------------------------------------------
# 5) Valores por defecto seguros
# ------------------------------------------------------------
defaults = {
    "Geographic Area": "Entire Area",
    "Race": "ALL",
    "Ethnicity": "ALL",
    "Gender": "ALL",
}

for col, val in defaults.items():
    if col in df.columns:
        df[col] = df[col].replace("", val)

# ------------------------------------------------------------
# 6) Revisar resumen final
# ------------------------------------------------------------
print("\nResumen corregido:")
print(df[[
    "Endpoint Group", "Endpoint", "Pollutant", "Metric",
    "Seasonal Metric", "Metric Statistic", "Study Author", "Study Year"
]].copy())

# ------------------------------------------------------------
# 7) Exportar versión corregida
# ------------------------------------------------------------
out = output_dir / "benmap_health_impact_functions_import_full_v2.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("\nArchivo exportado:")
print(out)

Columnas: ['Endpoint Group', 'Endpoint', 'Pollutant', 'Metric', 'Seasonal Metric', 'Metric Statistic', 'Study Year', 'Study Author', 'Study Location', 'Reference', 'Qualifier', 'Race', 'Ethnicity', 'Gender', 'Start Age', 'End Age', 'Geographic Area', 'Incidence DataSet', 'Prevalence DataSet', 'Variable DataSet', 'Beta', 'Distribution Beta', 'Parameter 1 Beta', 'Parameter 2 Beta', 'Other Pollutants', 'Function', 'Baseline Function', 'A', 'Name A', 'B', 'Name B', 'C', 'Name C']
Filas: 2

Resumen corregido:
  Endpoint Group               Endpoint Pollutant      Metric Seasonal Metric  \
0      Mortality    All-cause mortality      PM25  AnnualMean                   
1      Mortality  Respiratory mortality        O3   D8HourMax                   

  Metric Statistic        Study Author  Study Year  
0             None           Di et al.        2017  
1             None  Katsouyanni et al.        2009  

Archivo exportado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_fun

C:\Users\Rog Strix\AppData\Local\Temp\ipykernel_27796\1132940108.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask_pm25, "Seasonal Metric"] = ""


In [17]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

src = output_dir / "benmap_health_impact_functions_import_full_v2.csv"
if not src.exists():
    raise FileNotFoundError(src)

df = pd.read_csv(src)
df.columns = [c.strip() for c in df.columns]

# Dejar vacío el campo geográfico para que la función sea no restringida
if "Geographic Area" in df.columns:
    df["Geographic Area"] = ""

if "Geographic Area Feature" in df.columns:
    df["Geographic Area Feature"] = ""

out = output_dir / "benmap_health_impact_functions_import_full_v3.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("Archivo exportado:")
print(out)
print(df[["Endpoint Group", "Endpoint", "Pollutant", "Metric", "Geographic Area"]])

Archivo exportado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_import_full_v3.csv
  Endpoint Group               Endpoint Pollutant      Metric Geographic Area
0      Mortality    All-cause mortality      PM25  AnnualMean                
1      Mortality  Respiratory mortality        O3   D8HourMax                


In [18]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

src = output_dir / "benmap_health_impact_functions_import_full_v3.csv"
if not src.exists():
    raise FileNotFoundError(src)

# Leer todo como texto para evitar que "None" se convierta raro
df = pd.read_csv(src, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]

# Forzar exactamente lo que BenMAP espera
df["Metric Statistic"] = "None"

# Dejar geografía vacía
if "Geographic Area" in df.columns:
    df["Geographic Area"] = ""

if "Geographic Area Feature" in df.columns:
    df["Geographic Area Feature"] = ""

# Limpiar espacios
for c in df.columns:
    df[c] = df[c].astype(str).str.strip()

out = output_dir / "benmap_health_impact_functions_import_full_v4.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("Archivo exportado:")
print(out)
print("\nVista previa:")
print(df[["Endpoint Group", "Endpoint", "Pollutant", "Metric", "Metric Statistic"]])

Archivo exportado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_import_full_v4.csv

Vista previa:
  Endpoint Group               Endpoint Pollutant      Metric Metric Statistic
0      Mortality    All-cause mortality      PM25  AnnualMean             None
1      Mortality  Respiratory mortality        O3   D8HourMax             None


### Reconstrucción del archivo de Valuation Functions al formato esperado por BenMAP-CE

Esta celda transforma el archivo READY_ONLY de funciones de valoración económica al esquema de columnas que el importador de BenMAP-CE está validando en la sección de *Valuation Functions*. El problema no proviene del contenido económico definido, sino de que la versión exportada del archivo usa nombres de columnas simplificados como `Point Estimate`, `A Description` y `Constant Value`, mientras que BenMAP espera nombres como `Estimate`, `Name A`, `Distribution A`, `Parameter 1 A`, `D` y `Name D`. Por ello, esta celda renombra los campos existentes, agrega los parámetros opcionales faltantes en blanco, conserva los endpoints de la corrida base y exporta una nueva versión lista para validación en BenMAP-CE.

In [1]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

candidate_files = [
    output_dir / "benmap_valuation_functions_ready_only_working.csv",
    output_dir / "benmap_valuation_functions_ready_only.csv",
    output_dir / "benmap_valuation_functions_candidate_final_working.csv",
    output_dir / "benmap_valuation_functions_working.csv",
]

src = next((p for p in candidate_files if p.exists()), None)

if src is None:
    raise FileNotFoundError(
        "No encontré un archivo de valuation en SALIDAS_BENMAP."
    )

print("Archivo fuente encontrado:")
print(src)

df = pd.read_csv(src, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]

print("\nColumnas originales:")
print(df.columns.tolist())
print("Filas:", len(df))

# ------------------------------------------------------------
# 2) Si existe status, dejar solo READY_NUMERIC
# ------------------------------------------------------------
if "status" in df.columns:
    df = df[df["status"].astype(str).str.strip().eq("READY_NUMERIC")].copy().reset_index(drop=True)
    print("\nFilas READY_NUMERIC:", len(df))

if df.empty:
    raise ValueError("Después del filtro READY_NUMERIC, el archivo quedó vacío.")

# ------------------------------------------------------------
# 3) Función auxiliar
# ------------------------------------------------------------
def pick_col(frame, candidates, default=""):
    for c in candidates:
        if c in frame.columns:
            return frame[c]
    return pd.Series([default] * len(frame), index=frame.index)

# ------------------------------------------------------------
# 4) Construir formato de importación BenMAP para Valuation
# ------------------------------------------------------------
out = pd.DataFrame(index=df.index)

out["Endpoint Group"] = pick_col(df, ["Endpoint Group"], "")
out["Endpoint"] = pick_col(df, ["Endpoint"], "")
out["Qualifier"] = pick_col(df, ["Qualifier"], "")
out["Reference"] = pick_col(df, ["Reference"], "")
out["Start Age"] = pick_col(df, ["Start Age"], "0")
out["End Age"] = pick_col(df, ["End Age"], "99")

# Point Estimate -> Estimate
out["Estimate"] = pick_col(df, ["Estimate", "Point Estimate"], "")

# A block
out["Name A"] = pick_col(df, ["Name A", "A Description"], "")
out["A"] = pick_col(df, ["A"], "")
out["Distribution A"] = pick_col(df, ["Distribution A", "A Distribution"], "")
out["Parameter 1 A"] = pick_col(df, ["Parameter 1 A", "A Parameter 1"], "")
out["Parameter 2 A"] = pick_col(df, ["Parameter 2 A", "A Parameter 2"], "")

# B block (vacío si no existe)
out["Name B"] = pick_col(df, ["Name B"], "")
out["B"] = pick_col(df, ["B"], "")

# C block
out["Name C"] = pick_col(df, ["Name C"], "")
out["C"] = pick_col(df, ["C"], "")

# D block: usar Constant Description / Constant Value
out["Name D"] = pick_col(df, ["Name D", "Constant Description"], "")
out["D"] = pick_col(df, ["D", "Constant Value"], "")

# Function
out["Function"] = pick_col(df, ["Function"], "")

# ------------------------------------------------------------
# 5) Limpiar texto
# ------------------------------------------------------------
for c in out.columns:
    out[c] = out[c].astype(str).str.strip()

# Tipos numéricos donde aplica
for c in ["Start Age", "End Age"]:
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0 if c == "Start Age" else 99).astype(int)

# ------------------------------------------------------------
# 6) Vista previa
# ------------------------------------------------------------
print("\nVista previa del archivo reconstruido:")
print(out)

print("\nColumnas finales:")
print(out.columns.tolist())

# ------------------------------------------------------------
# 7) Exportar
# ------------------------------------------------------------
export_file = output_dir / "benmap_valuation_functions_import_full.csv"
out.to_csv(export_file, index=False, encoding="utf-8-sig")

print("\nArchivo exportado:")
print(export_file)

Archivo fuente encontrado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_ready_only_working.csv

Columnas originales:
['Endpoint Group', 'Endpoint', 'Qualifier', 'Reference', 'Start Age', 'End Age', 'Point Estimate', 'Function', 'A Description', 'A', 'A Distribution', 'A Parameter 1', 'A Parameter 2', 'Constant Description', 'Constant Value', 'endpoint_code_local', 'valuation_type', 'currency', 'price_year', 'status', 'notes']
Filas: 2

Filas READY_NUMERIC: 2

Vista previa del archivo reconstruido:
  Endpoint Group               Endpoint                         Qualifier  \
0      Mortality    All-cause mortality  EPA VSL central value (2015 USD)   
1      Mortality  Respiratory mortality  EPA VSL central value (2015 USD)   

                                         Reference  Start Age  End Age  \
0  EPA TSD 2024, Table 22; VSL based on 26 studies          0       99   
1  EPA TSD 2024, Table 22; VSL based on 26 studies          0       99   

    Estimate Name

In [2]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

src = output_dir / "benmap_valuation_functions_import_full.csv"
if not src.exists():
    raise FileNotFoundError(src)

df = pd.read_csv(src, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]

# Pasar el valor de Estimate a D
if "Estimate" in df.columns:
    if "D" not in df.columns:
        df["D"] = ""
    df["D"] = df["Estimate"].astype(str).str.strip()
    df = df.drop(columns=["Estimate"])

# Si Name D está vacío, poner una descripción útil
if "Name D" in df.columns:
    df["Name D"] = df["Name D"].astype(str).str.strip()
    df.loc[df["Name D"] == "", "Name D"] = "EPA VSL central value (2015 USD)"

# Ajustar función para que use D en vez de PointEstimate
if "Function" in df.columns:
    df["Function"] = (
        df["Function"]
        .astype(str)
        .str.replace("PointEstimate", "D", regex=False)
        .str.strip()
    )

out = output_dir / "benmap_valuation_functions_import_full_v2.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("Archivo exportado:")
print(out)
print("\nColumnas finales:")
print(df.columns.tolist())
print("\nVista previa:")
print(df.head())

Archivo exportado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_import_full_v2.csv

Columnas finales:
['Endpoint Group', 'Endpoint', 'Qualifier', 'Reference', 'Start Age', 'End Age', 'Name A', 'A', 'Distribution A', 'Parameter 1 A', 'Parameter 2 A', 'Name B', 'B', 'Name C', 'C', 'Name D', 'D', 'Function']

Vista previa:
  Endpoint Group               Endpoint                         Qualifier  \
0      Mortality    All-cause mortality  EPA VSL central value (2015 USD)   
1      Mortality  Respiratory mortality  EPA VSL central value (2015 USD)   

                                         Reference Start Age End Age Name A A  \
0  EPA TSD 2024, Table 22; VSL based on 26 studies         0      99            
1  EPA TSD 2024, Table 22; VSL based on 26 studies         0      99            

  Distribution A Parameter 1 A Parameter 2 A Name B B Name C C  \
0                                                                
1                                            

In [3]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

src = output_dir / "benmap_valuation_functions_import_full_v2.csv"
if not src.exists():
    raise FileNotFoundError(src)

df = pd.read_csv(src, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]

# Valor monetario central ya usado en tu flujo
vsl_value = "8705114.0"

# Llenar bloque A, que es el que BenMAP está exigiendo
df["Name A"] = "EPA VSL central value (2015 USD)"
df["A"] = vsl_value
df["Distribution A"] = "None"

# Si existen, los parámetros de A pueden quedar vacíos con Distribution A = None
if "Parameter 1 A" in df.columns:
    df["Parameter 1 A"] = ""
if "Parameter 2 A" in df.columns:
    df["Parameter 2 A"] = ""

# La función debe usar A, no D
if "Function" in df.columns:
    df["Function"] = "A*Incidence"

# D y Name D pueden quedar vacíos
if "Name D" in df.columns:
    df["Name D"] = ""
if "D" in df.columns:
    df["D"] = ""

out = output_dir / "benmap_valuation_functions_import_full_v3.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("Archivo exportado:")
print(out)
print("\nVista previa:")
print(df[[
    "Endpoint Group", "Endpoint", "Qualifier",
    "Name A", "A", "Distribution A", "Function"
]])

Archivo exportado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_valuation_functions_import_full_v3.csv

Vista previa:
  Endpoint Group               Endpoint                         Qualifier  \
0      Mortality    All-cause mortality  EPA VSL central value (2015 USD)   
1      Mortality  Respiratory mortality  EPA VSL central value (2015 USD)   

                             Name A          A Distribution A     Function  
0  EPA VSL central value (2015 USD)  8705114.0           None  A*Incidence  
1  EPA VSL central value (2015 USD)  8705114.0           None  A*Incidence  


### Conversión de PM2.5 baseline/control al formato de Model Data requerido por BenMAP-CE

Esta celda transforma los archivos intermedios de exposición de PM$_{2.5}$ (`benmap_PM25_baseline.csv` y `benmap_PM25_control.csv`) al formato exacto que BenMAP-CE requiere para importar superficies de calidad del aire como *Model Data*. Para ello, toma la grilla espacial validada con `ROW` y `COL`, agrega las concentraciones mensuales a un valor anual por celda, y construye archivos con las columnas `Column`, `Row`, `Metric`, `Seasonal Metric`, `Annual Metric` y `Values`. En esta primera corrida, la métrica utilizada para PM$_{2.5}$ es `AnnualMean`, consistente con la función sanitaria que ya fue validada en el setup de BenMAP. Los archivos exportados quedarán listos para cargarse como escenario *Baseline* y *Control*.

In [4]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1) Rutas del proyecto
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

baseline_src = output_dir / "benmap_PM25_baseline.csv"
control_src  = output_dir / "benmap_PM25_control.csv"
grid_src     = output_dir / "benmap_grid_definition_final_ok_debug.csv"

for p in [baseline_src, control_src, grid_src]:
    if not p.exists():
        raise FileNotFoundError(f"No se encontró:\n{p}")

print("Archivo baseline:", baseline_src)
print("Archivo control :", control_src)
print("Archivo grilla  :", grid_src)

# ------------------------------------------------------------
# 2) Cargar archivos
# ------------------------------------------------------------
df_b = pd.read_csv(baseline_src)
df_c = pd.read_csv(control_src)
df_g = pd.read_csv(grid_src)

df_b.columns = [c.strip() for c in df_b.columns]
df_c.columns = [c.strip() for c in df_c.columns]
df_g.columns = [c.strip() for c in df_g.columns]

print("\nColumnas baseline:", df_b.columns.tolist())
print("Columnas control :", df_c.columns.tolist())
print("Columnas grilla  :", df_g.columns.tolist())

# ------------------------------------------------------------
# 3) Detectar columna de valor
# ------------------------------------------------------------
baseline_value_candidates = ["pm25_baseline", "PM25_baseline", "baseline", "value", "Values"]
control_value_candidates  = ["pm25_control", "PM25_control", "control", "value", "Values"]

baseline_value_col = next((c for c in baseline_value_candidates if c in df_b.columns), None)
control_value_col  = next((c for c in control_value_candidates if c in df_c.columns), None)

if baseline_value_col is None:
    raise ValueError("No encontré la columna de valores baseline en benmap_PM25_baseline.csv")
if control_value_col is None:
    raise ValueError("No encontré la columna de valores control en benmap_PM25_control.csv")

# ------------------------------------------------------------
# 4) Validar columnas mínimas
# ------------------------------------------------------------
required_common = {"cell_id"}
if not required_common.issubset(df_b.columns):
    raise ValueError("Falta 'cell_id' en baseline.")
if not required_common.issubset(df_c.columns):
    raise ValueError("Falta 'cell_id' en control.")

# Grilla con ROW / COL
rename_grid = {}
for c in df_g.columns:
    cl = c.lower().strip()
    if cl == "row":
        rename_grid[c] = "Row"
    elif cl == "col":
        rename_grid[c] = "Column"
    elif cl == "column":
        rename_grid[c] = "Column"
    elif cl == "cell_id":
        rename_grid[c] = "cell_id"

df_g = df_g.rename(columns=rename_grid)

for col in ["cell_id", "Row", "Column"]:
    if col not in df_g.columns:
        raise ValueError(f"Falta '{col}' en la grilla debug.")

# ------------------------------------------------------------
# 5) Agregar a valor anual por celda
#    Si hay una fila por mes, tomamos el promedio anual.
# ------------------------------------------------------------
df_b["cell_id"] = pd.to_numeric(df_b["cell_id"], errors="coerce")
df_c["cell_id"] = pd.to_numeric(df_c["cell_id"], errors="coerce")
df_g["cell_id"] = pd.to_numeric(df_g["cell_id"], errors="coerce")

df_b[baseline_value_col] = pd.to_numeric(df_b[baseline_value_col], errors="coerce")
df_c[control_value_col]  = pd.to_numeric(df_c[control_value_col], errors="coerce")

b_annual = (
    df_b.groupby("cell_id", as_index=False)[baseline_value_col]
    .mean()
    .rename(columns={baseline_value_col: "Values"})
)

c_annual = (
    df_c.groupby("cell_id", as_index=False)[control_value_col]
    .mean()
    .rename(columns={control_value_col: "Values"})
)

# ------------------------------------------------------------
# 6) Unir con grilla
# ------------------------------------------------------------
grid_map = df_g[["cell_id", "Row", "Column"]].drop_duplicates(subset=["cell_id"]).copy()

b_annual = b_annual.merge(grid_map, on="cell_id", how="left")
c_annual = c_annual.merge(grid_map, on="cell_id", how="left")

if b_annual[["Row", "Column"]].isna().any().any():
    bad = b_annual.loc[b_annual["Row"].isna() | b_annual["Column"].isna(), "cell_id"].tolist()[:10]
    raise ValueError(f"Baseline tiene cell_id sin Row/Column. Ejemplos: {bad}")

if c_annual[["Row", "Column"]].isna().any().any():
    bad = c_annual.loc[c_annual["Row"].isna() | c_annual["Column"].isna(), "cell_id"].tolist()[:10]
    raise ValueError(f"Control tiene cell_id sin Row/Column. Ejemplos: {bad}")

# ------------------------------------------------------------
# 7) Construir formato BenMAP Model Data
#    Para PM25 usamos AnnualMean y un valor anual por celda.
# ------------------------------------------------------------
def build_benmap_model_data(df):
    out = pd.DataFrame()
    out["Column"] = df["Column"].astype(int)
    out["Row"] = df["Row"].astype(int)
    out["Metric"] = "AnnualMean"
    out["Seasonal Metric"] = ""
    out["Annual Metric"] = "None"
    out["Values"] = df["Values"].round(6).astype(str)
    return out.sort_values(["Row", "Column"]).reset_index(drop=True)

pm25_baseline_import = build_benmap_model_data(b_annual)
pm25_control_import  = build_benmap_model_data(c_annual)

# ------------------------------------------------------------
# 8) Resúmenes
# ------------------------------------------------------------
print("\nResumen baseline import-ready:")
print(pm25_baseline_import.head())
print("Filas baseline:", len(pm25_baseline_import))

print("\nResumen control import-ready:")
print(pm25_control_import.head())
print("Filas control:", len(pm25_control_import))

# ------------------------------------------------------------
# 9) Exportar
# ------------------------------------------------------------
baseline_out = output_dir / "benmap_PM25_baseline_import_ready.csv"
control_out  = output_dir / "benmap_PM25_control_import_ready.csv"

pm25_baseline_import.to_csv(baseline_out, index=False, encoding="utf-8-sig")
pm25_control_import.to_csv(control_out, index=False, encoding="utf-8-sig")

print("\nArchivos exportados:")
print(baseline_out)
print(control_out)

Archivo baseline: D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_PM25_baseline.csv
Archivo control : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_PM25_control.csv
Archivo grilla  : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_grid_definition_final_ok_debug.csv

Columnas baseline: ['cell_id', 'fecha', 'year', 'month', 'pm25_baseline', 'pm25_p05', 'pm25_p95']
Columnas control : ['cell_id', 'fecha', 'year', 'month', 'pm25_control']
Columnas grilla  : ['cell_id', 'ROW', 'COL']

Resumen baseline import-ready:
   Column  Row      Metric Seasonal Metric Annual Metric     Values
0       1    1  AnnualMean                          None  15.051786
1       2    1  AnnualMean                          None  14.730532
2       3    1  AnnualMean                          None  14.851986
3       1    2  AnnualMean                          None  14.821127
4       2    2  AnnualMean                          None  14.056199
Filas baseline: 254

Resumen control import-ready:
   Column  Row    

### Corrección del formato de PM2.5 baseline/control para que BenMAP reconozca un valor anual por celda

Esta celda corrige el error de importación detectado al cargar la superficie de PM$_{2.5}$ en BenMAP-CE. Aunque el archivo había pasado la validación de columnas, BenMAP rechazó la creación de la superficie porque interpretó que no se había especificado un estadístico anual válido. Según el formato requerido por BenMAP, cuando se suministra un único valor por celda y por año, la columna `Annual Metric` debe indicar explícitamente un estadístico anual como `Mean`, en lugar de `None`. Por ello, esta celda reexporta los archivos baseline y control de PM$_{2.5}$ con `Annual Metric = Mean`, manteniendo la misma grilla y la métrica `AnnualMean`.

In [5]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

baseline_src = output_dir / "benmap_PM25_baseline_import_ready.csv"
control_src  = output_dir / "benmap_PM25_control_import_ready.csv"

for p in [baseline_src, control_src]:
    if not p.exists():
        raise FileNotFoundError(f"No se encontró:\n{p}")

# ------------------------------------------------------------
# 2) Cargar y corregir
# ------------------------------------------------------------
df_b = pd.read_csv(baseline_src, dtype=str, keep_default_na=False)
df_c = pd.read_csv(control_src, dtype=str, keep_default_na=False)

for df in [df_b, df_c]:
    df.columns = [c.strip() for c in df.columns]

    required = ["Column", "Row", "Metric", "Seasonal Metric", "Annual Metric", "Values"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    # Mantener métrica anual de PM2.5
    df["Metric"] = "AnnualMean"
    df["Seasonal Metric"] = ""
    df["Annual Metric"] = "Mean"

    # Limpiar Values
    df["Values"] = pd.to_numeric(df["Values"], errors="coerce")
    if df["Values"].isna().any():
        raise ValueError("Hay valores no numéricos en 'Values'.")
    df["Values"] = df["Values"].round(6).astype(str)

# ------------------------------------------------------------
# 3) Exportar versiones corregidas
# ------------------------------------------------------------
baseline_out = output_dir / "benmap_PM25_baseline_import_ready_v2.csv"
control_out  = output_dir / "benmap_PM25_control_import_ready_v2.csv"

df_b.to_csv(baseline_out, index=False, encoding="utf-8-sig")
df_c.to_csv(control_out, index=False, encoding="utf-8-sig")

print("Archivos exportados:")
print(baseline_out)
print(control_out)

print("\nVista previa baseline:")
print(df_b.head())

print("\nVista previa control:")
print(df_c.head())

Archivos exportados:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_PM25_baseline_import_ready_v2.csv
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_PM25_control_import_ready_v2.csv

Vista previa baseline:
  Column Row      Metric Seasonal Metric Annual Metric     Values
0      1   1  AnnualMean                          Mean  15.051786
1      2   1  AnnualMean                          Mean  14.730532
2      3   1  AnnualMean                          Mean  14.851986
3      1   2  AnnualMean                          Mean  14.821127
4      2   2  AnnualMean                          Mean  14.056199

Vista previa control:
  Column Row      Metric Seasonal Metric Annual Metric     Values
0      1   1  AnnualMean                          Mean  13.546608
1      2   1  AnnualMean                          Mean  13.257479
2      3   1  AnnualMean                          Mean  13.366787
3      1   2  AnnualMean                          Mean  13.339014
4      2   2  AnnualMean               

### Construcción de un archivo mínimo de Health Impact Functions para BenMAP-CE

Esta celda localiza el archivo de funciones de salud que ya había sido reconstruido al esquema de importación de BenMAP-CE y genera una versión mínima con una sola función para la corrida base: **PM25 → All-cause mortality → Di et al. → 65–99**. La idea no es rehacer el esquema completo, sino reutilizar el archivo de importación válido y reducirlo a una sola fila para evitar que la ventana de *Health Impact Functions* se quede cargando innecesariamente.

In [ ]:
# ============================================================
# CELDA: reconstruir archivo mínimo de Health Impact Functions
# SIN inventar columnas inválidas
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1) Rutas
# ------------------------------------------------------------
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

print("Carpeta actual :", current_dir)
print("Project root   :", project_root)
print("SALIDAS_BENMAP :", output_dir)

if not output_dir.exists():
    raise FileNotFoundError(f"No existe SALIDAS_BENMAP en: {output_dir}")

# ------------------------------------------------------------
# 2) Buscar archivo fuente ORIGINAL con esquema válido
#    (preferimos los import_full que ya venían del flujo BenMAP)
# ------------------------------------------------------------
candidate_files = [
    output_dir / "benmap_health_impact_functions_import_full_v3.csv",
    output_dir / "benmap_health_impact_functions_import_full_v2.csv",
    output_dir / "benmap_health_impact_functions_import_full.csv",
    output_dir / "benmap_health_impact_functions_ready_only_working.csv",
    output_dir / "benmap_health_impact_functions_ready_only.csv",
    output_dir / "benmap_health_impact_functions_working.csv",
]

src = next((p for p in candidate_files if p.exists()), None)

if src is None:
    raise FileNotFoundError(
        "No encontré un archivo fuente ORIGINAL de health impact functions en SALIDAS_BENMAP."
    )

print("\nArchivo fuente seleccionado:")
print(src)

# ------------------------------------------------------------
# 3) Leer archivo fuente preservando texto
# ------------------------------------------------------------
df = pd.read_csv(src, dtype=str, keep_default_na=False, encoding="utf-8-sig")
df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

print("\nColumnas originales detectadas:")
print(df.columns.tolist())
print("\nDimensión original:", df.shape)

# ------------------------------------------------------------
# 4) Buscar columnas por nombre
# ------------------------------------------------------------
def find_col(candidates, columns):
    cols_map = {c.lower().strip(): c for c in columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in cols_map:
            return cols_map[key]
    for c in columns:
        cl = c.lower().strip()
        for cand in candidates:
            if cand.lower().strip() in cl:
                return c
    return None

col_endpoint_group = find_col(["Endpoint Group"], df.columns)
col_endpoint       = find_col(["Endpoint"], df.columns)
col_pollutant      = find_col(["Pollutant"], df.columns)
col_author         = find_col(["Study Author", "Author"], df.columns)
col_start_age      = find_col(["Start Age"], df.columns)
col_end_age        = find_col(["End Age"], df.columns)
col_metric         = find_col(["Metric"], df.columns)
col_metric_stat    = find_col(["Metric Statistic"], df.columns)
col_geo            = find_col(["Geographic Area"], df.columns)
col_race           = find_col(["Race"], df.columns)
col_eth            = find_col(["Ethnicity"], df.columns)
col_gender         = find_col(["Gender"], df.columns)
col_inc            = find_col(["Incidence Dataset", "Incidence DataSet"], df.columns)
col_prev           = find_col(["Prevalence Dataset", "Prevalence DataSet"], df.columns)
col_var            = find_col(["Variable Dataset", "Variable DataSet"], df.columns)

required = {
    "Endpoint Group": col_endpoint_group,
    "Endpoint": col_endpoint,
    "Pollutant": col_pollutant,
}

missing_required = [k for k, v in required.items() if v is None]
if missing_required:
    raise ValueError(
        f"No encontré estas columnas mínimas en el archivo fuente: {missing_required}"
    )

# ------------------------------------------------------------
# 5) Filtrar SOLO la fila mínima de la corrida base
# ------------------------------------------------------------
mask = (
    df[col_pollutant].astype(str).str.strip().str.upper().eq("PM25") &
    df[col_endpoint].astype(str).str.strip().str.lower().eq("all-cause mortality")
)

if col_endpoint_group is not None:
    mask = mask & df[col_endpoint_group].astype(str).str.strip().str.lower().eq("mortality")

# Autor y edades, solo si existen
if col_author is not None:
    mask = mask & df[col_author].astype(str).str.contains("Di", case=False, na=False)

if col_start_age is not None:
    mask = mask & pd.to_numeric(df[col_start_age], errors="coerce").fillna(-1).eq(65)

if col_end_age is not None:
    mask = mask & pd.to_numeric(df[col_end_age], errors="coerce").fillna(-1).eq(99)

df_min = df.loc[mask].copy()

# Filtro relajado si no encuentra con autor/edades
if df_min.empty:
    print("\nNo encontré coincidencia exacta. Intento filtro relajado...")
    mask_relaxed = (
        df[col_pollutant].astype(str).str.strip().str.upper().eq("PM25") &
        df[col_endpoint].astype(str).str.strip().str.lower().eq("all-cause mortality")
    )
    if col_endpoint_group is not None:
        mask_relaxed = mask_relaxed & df[col_endpoint_group].astype(str).str.strip().str.lower().eq("mortality")
    df_min = df.loc[mask_relaxed].copy()

if df_min.empty:
    raise ValueError("No encontré una fila PM25 + All-cause mortality en el archivo fuente.")

# Dejar una sola fila
df_min = df_min.head(1).copy()

# ------------------------------------------------------------
# 6) Modificar SOLO columnas que YA EXISTEN en el esquema fuente
# ------------------------------------------------------------
# NO crear Annual Statistic ni otras columnas nuevas

if col_metric is not None:
    df_min[col_metric] = "AnnualMean"

if col_metric_stat is not None:
    # El error anterior mostró que este campo es obligatorio
    df_min[col_metric_stat] = "None"

if col_start_age is not None:
    df_min[col_start_age] = "65"

if col_end_age is not None:
    df_min[col_end_age] = "99"

if col_geo is not None:
    df_min[col_geo] = "Everywhere"

if col_race is not None:
    df_min[col_race] = "ALL"

if col_eth is not None:
    df_min[col_eth] = "ALL"

if col_gender is not None:
    df_min[col_gender] = "ALL"

# Dejar datasets auxiliares vacíos si existen
if col_inc is not None:
    df_min[col_inc] = ""

if col_prev is not None:
    df_min[col_prev] = ""

if col_var is not None:
    df_min[col_var] = ""

# Limpiar espacios
for c in df_min.columns:
    df_min[c] = df_min[c].astype(str).str.strip()

# ------------------------------------------------------------
# 7) Eliminar explícitamente columnas inválidas si existieran
#    por arrastre de pruebas anteriores
# ------------------------------------------------------------
invalid_cols = [c for c in ["Annual Statistic"] if c in df_min.columns]
if invalid_cols:
    print("\nEliminando columnas inválidas:", invalid_cols)
    df_min = df_min.drop(columns=invalid_cols)

# ------------------------------------------------------------
# 8) Exportar versión final
# ------------------------------------------------------------
out = output_dir / "benmap_health_impact_functions_min_pm25_v3.csv"
df_min.to_csv(out, index=False, encoding="utf-8-sig")

print("\nArchivo exportado:")
print(out)

print("\nDimensión final:", df_min.shape)
print("\nVista previa:")
print(df_min.T)

Carpeta actual : D:\TRABAJO DE GRADO BEN-MAP\CODIGO
Project root   : D:\TRABAJO DE GRADO BEN-MAP
SALIDAS_BENMAP : D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP

Archivo fuente seleccionado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_import_full_v3.csv

Columnas originales detectadas:
['Endpoint Group', 'Endpoint', 'Pollutant', 'Metric', 'Seasonal Metric', 'Metric Statistic', 'Study Year', 'Study Author', 'Study Location', 'Reference', 'Qualifier', 'Race', 'Ethnicity', 'Gender', 'Start Age', 'End Age', 'Geographic Area', 'Incidence DataSet', 'Prevalence DataSet', 'Variable DataSet', 'Beta', 'Distribution Beta', 'Parameter 1 Beta', 'Parameter 2 Beta', 'Other Pollutants', 'Function', 'Baseline Function', 'A', 'Name A', 'B', 'Name B', 'C', 'Name C']

Dimensión original: (2, 33)

Archivo exportado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_min_pm25_v3.csv

Dimensión final: (1, 33)

Vista previa:
                                    

In [9]:
# ============================================================
# CELDA: quitar Geographic Area del archivo mínimo
# ============================================================

from pathlib import Path
import pandas as pd

# Rutas
current_dir = Path.cwd().resolve()
project_root = current_dir if current_dir.name != "CODIGO" else current_dir.parent
output_dir = project_root / "SALIDAS_BENMAP"

src = output_dir / "benmap_health_impact_functions_min_pm25_v3.csv"
if not src.exists():
    raise FileNotFoundError(f"No se encontró:\n{src}")

print("Archivo fuente:")
print(src)

# Leer como texto
df = pd.read_csv(src, dtype=str, keep_default_na=False, encoding="utf-8-sig")
df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

print("\nColumnas:")
print(df.columns.tolist())

# Buscar columna Geographic Area
geo_candidates = ["Geographic Area"]
geo_col = next((c for c in df.columns if c.strip().lower() in [g.lower() for g in geo_candidates]), None)

if geo_col is not None:
    df[geo_col] = ""
    print(f"\nSe vació la columna: {geo_col}")
else:
    print("\nNo existe columna Geographic Area; no se modifica nada.")

# Exportar nueva versión
out = output_dir / "benmap_health_impact_functions_min_pm25_v4.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("\nArchivo exportado:")
print(out)

print("\nVista previa:")
print(df.T)

Archivo fuente:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_min_pm25_v3.csv

Columnas:
['Endpoint Group', 'Endpoint', 'Pollutant', 'Metric', 'Seasonal Metric', 'Metric Statistic', 'Study Year', 'Study Author', 'Study Location', 'Reference', 'Qualifier', 'Race', 'Ethnicity', 'Gender', 'Start Age', 'End Age', 'Geographic Area', 'Incidence DataSet', 'Prevalence DataSet', 'Variable DataSet', 'Beta', 'Distribution Beta', 'Parameter 1 Beta', 'Parameter 2 Beta', 'Other Pollutants', 'Function', 'Baseline Function', 'A', 'Name A', 'B', 'Name B', 'C', 'Name C']

Se vació la columna: Geographic Area

Archivo exportado:
D:\TRABAJO DE GRADO BEN-MAP\SALIDAS_BENMAP\benmap_health_impact_functions_min_pm25_v4.csv

Vista previa:
                                                       0
Endpoint Group                                 Mortality
Endpoint                             All-cause mortality
Pollutant                                           PM25
Metric               